In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [4]:
%pip install -q -U \
  transformers \
  datasets \
  accelerate \
  peft \
  trl \
  bitsandbytes \
  "huggingface_hub>=1.32.0,<2.0" \
  "pandas==2.2.3" \
  "pyarrow>=19.0.0,<24.0" \
  scikit-learn

In [5]:
import importlib.metadata as metadata

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "trl",
    "bitsandbytes",
    "huggingface_hub",
    "pandas",
    "pyarrow"
]

for package in packages:
    print(f"{package}: {metadata.version(package)}")

from transformers import AutoTokenizer
from datasets import load_dataset
from peft import LoraConfig
from trl import SFTTrainer

print("\n✅ All core libraries imported successfully!")

transformers: 5.20.2
datasets: 5.0.1
accelerate: 1.15.0
peft: 0.21.2
trl: 1.16.0
bitsandbytes: 0.50.2
huggingface_hub: 1.33.0
pandas: 2.2.3
pyarrow: 23.0.1

✅ All core libraries imported successfully!


In [6]:
from datasets import load_dataset

dataset = load_dataset(
    "Qdrant/hm_ecommerce_products",
    split="train",
    streaming=True
)

for i, example in enumerate(dataset):
    print(f"\n--- Product {i+1} ---")
    print("Name:", example.get("prod_name"))
    print("Description:", example.get("detail_desc"))
    print("Category:", example.get("product_type_name"))
    print("Color:", example.get("colour_group_name"))
    print("Pattern:", example.get("graphical_appearance_name"))

    if i == 4:
        break

README.md:   0%|          | 0.00/9.66k [00:00<?, ?B/s]


--- Product 1 ---
Name: Strap top
Description: Jersey top with narrow shoulder straps.
Category: Vest top
Color: Black
Pattern: Solid

--- Product 2 ---
Name: Strap top
Description: Jersey top with narrow shoulder straps.
Category: Vest top
Color: White
Pattern: Solid

--- Product 3 ---
Name: Strap top (1)
Description: Jersey top with narrow shoulder straps.
Category: Vest top
Color: Off White
Pattern: Stripe

--- Product 4 ---
Name: OP T-shirt (Idro)
Description: Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.
Category: Bra
Color: Black
Pattern: Solid

--- Product 5 ---
Name: OP T-shirt (Idro)
Description: Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the

In [7]:
import json

first_product = next(iter(dataset))

print("Available fields:\n")
for key, value in first_product.items():
    print(f"{key}: {str(value)[:150]}")

Available fields:

article_id: 0108775015
product_code: 0108775
prod_name: Strap top
product_type_no: 253
product_type_name: Vest top
product_group_name: Garment Upper body
graphical_appearance_no: 1010016
graphical_appearance_name: Solid
colour_group_code: 09
colour_group_name: Black
perceived_colour_value_id: 4
perceived_colour_value_name: Dark
perceived_colour_master_id: 5
perceived_colour_master_name: Black
department_no: 1676
department_name: Jersey Basic
index_code: A
index_name: Ladieswear
index_group_no: 1
index_group_name: Ladieswear
section_no: 16
section_name: Womens Everyday Basics
garment_group_no: 1002
garment_group_name: Jersey Basic
detail_desc: Jersey top with narrow shoulder straps.
text_to_embed: Strap top Vest top Garment Upper body Solid Black Dark Black Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic Jersey top with na
image_url: https://qdrant-nextjs-demo-product-images.s3.us-east-1.amazonaws.com/images/010/0108775015.jpg
dense_embedding: [

In [8]:
from itertools import islice

total = 0
color_mentioned = 0
examples_missing_color = []

for product in islice(dataset, 1000):
    total += 1

    text = (
        str(product.get("prod_name") or "") + " " +
        str(product.get("detail_desc") or "")
    ).lower()

    color = str(
        product.get("colour_group_name") or ""
    ).lower()

    if color and color in text:
        color_mentioned += 1
    elif len(examples_missing_color) < 5:
        examples_missing_color.append({
            "name": product.get("prod_name"),
            "description": product.get("detail_desc"),
            "catalog_color": product.get("colour_group_name")
        })

print("Products checked:", total)
print("Color explicitly mentioned:", color_mentioned)
print("Color not directly mentioned:", total - color_mentioned)
print("Direct mention rate:", round(color_mentioned / total * 100, 2), "%")

print("\nExamples where catalog color is not mentioned:")
for example in examples_missing_color:
    print(example)

Products checked: 1000
Color explicitly mentioned: 6
Color not directly mentioned: 994
Direct mention rate: 0.6 %

Examples where catalog color is not mentioned:
{'name': 'Strap top', 'description': 'Jersey top with narrow shoulder straps.', 'catalog_color': 'Black'}
{'name': 'Strap top', 'description': 'Jersey top with narrow shoulder straps.', 'catalog_color': 'White'}
{'name': 'Strap top (1)', 'description': 'Jersey top with narrow shoulder straps.', 'catalog_color': 'Off White'}
{'name': 'OP T-shirt (Idro)', 'description': 'Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.', 'catalog_color': 'Black'}
{'name': 'OP T-shirt (Idro)', 'description': 'Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps an

## Check product category distribution

In [9]:
from collections import Counter
from itertools import islice

categories = Counter()

for product in islice(dataset, 5000):
    category = product.get("product_type_name")

    if category:
        categories[category] += 1

print("Unique categories:", len(categories))

print("\nTop 20 product categories:")
for category, count in categories.most_common(20):
    print(f"{category:<25} {count}")

Unique categories: 80

Top 20 product categories:
Trousers                  574
Socks                     398
T-shirt                   381
Underwear bottom          277
Shorts                    257
Vest top                  193
Underwear Tights          188
Dress                     174
Sweater                   170
Leggings/Tights           164
Hoodie                    158
Pyjama set                144
Top                       133
Bra                       119
Sunglasses                105
Pyjama jumpsuit/playsuit  104
Bodysuit                  99
Jacket                    95
Swimwear bottom           83
Shirt                     83


## We found 80 unique fashion categories in the first 5,000 records.

## Check duplicate product descriptions

In [10]:
from collections import Counter
from itertools import islice

descriptions = []
categories_by_description = {}

for product in islice(dataset, 5000):
    description = (
        str(product.get("prod_name") or "") + " " +
        str(product.get("detail_desc") or "")
    ).strip().lower()

    category = product.get("product_type_name")

    if description:
        descriptions.append(description)

        if description not in categories_by_description:
            categories_by_description[description] = set()

        categories_by_description[description].add(category)

counts = Counter(descriptions)

print("Total records:", len(descriptions))
print("Unique descriptions:", len(counts))
print("Duplicate records:", len(descriptions) - len(counts))

conflicting = {
    desc: cats
    for desc, cats in categories_by_description.items()
    if len(cats) > 1
}

print("Descriptions with conflicting categories:", len(conflicting))

print("\nMost repeated descriptions:")
for desc, count in counts.most_common(5):
    print(f"\nRepeated {count} times")
    print(desc[:180])

Total records: 5000
Unique descriptions: 1538
Duplicate records: 3462
Descriptions with conflicting categories: 18

Most repeated descriptions:

Repeated 30 times
clay tee sports top in fast-drying functional fabric with taped details and short raglan sleeves. rounded and longer at the back.

Repeated 27 times
sb trunks printed swimming trunks with an elasticated, drawstring waist and lined front.

Repeated 23 times
brax shorts short shorts in soft organic cotton jersey with an elasticated drawstring waist, and side pockets.

Repeated 19 times
glassig espadrille espadrilles in cotton with a braided jute trim around the soles and concealed elastic at the back of the heels. fabric linings and insoles and patterned soles.

Repeated 19 times
gullan 5-p ankle sock jacquard-knit socks in various colours and designs.


## 69.2% of the sampled records are duplicates
Create our first clean dataset
We'll deduplicate the 5,000 sampled records and exclude ambiguous category labels.

In [11]:
from itertools import islice
from collections import defaultdict
from datasets import Dataset

groups = defaultdict(list)

for product in islice(dataset, 5000):
    name = str(product.get("prod_name") or "").strip()
    description = str(product.get("detail_desc") or "").strip()
    category = product.get("product_type_name")

    if not description or not category:
        continue

    input_text = f"{name}. {description}"
    key = " ".join(input_text.lower().split())

    groups[key].append({
        "input": input_text,
        "category": category
    })

clean_records = []
conflict_count = 0

for records in groups.values():
    categories = {r["category"] for r in records}

    if len(categories) != 1:
        conflict_count += 1
        continue

    clean_records.append(records[0])

clean_dataset = Dataset.from_list(clean_records)

print("Clean records:", len(clean_dataset))
print("Conflicting descriptions excluded:", conflict_count)

print("\nSample clean record:")
print(clean_dataset[0])

Clean records: 1519
Conflicting descriptions excluded: 18

Sample clean record:
{'input': 'Strap top. Jersey top with narrow shoulder straps.', 'category': 'Vest top'}


## Check the full dataset size

In [12]:
from huggingface_hub import HfApi

api = HfApi()

info = api.dataset_info(
    "Qdrant/hm_ecommerce_products"
)

print("Dataset:", info.id)

if info.card_data:
    print("Dataset metadata:", info.card_data)

print("\nAvailable repository files:")
for file in info.siblings[:15]:
    print(file.rfilename)

Dataset: Qdrant/hm_ecommerce_products
Dataset metadata: license: cc-by-4.0
size_categories:
- 100K<n<1M
task_categories:
- image-classification
- text-classification
- image-to-text
- image-feature-extraction
pretty_name: H&M Fashion - Enhanced with Embeddings
tags:
- fashion
- recommendation
- embeddings
- retail
- e-commerce
- multimodal

Available repository files:
.gitattributes
README.md
hm_ecommerce_products_enriched.parquet


## Count unique descriptions across the full dataset

In [13]:
from collections import defaultdict

unique_products = defaultdict(set)
total = 0

for product in dataset:
    total += 1

    name = str(product.get("prod_name") or "").strip()
    desc = str(product.get("detail_desc") or "").strip()
    category = product.get("product_type_name")

    if not desc or not category:
        continue

    key = " ".join(f"{name} {desc}".lower().split())
    unique_products[key].add(category)

valid = sum(
    len(categories) == 1
    for categories in unique_products.values()
)

conflicting = sum(
    len(categories) > 1
    for categories in unique_products.values()
)

print("Total records:", total)
print("Unique descriptions:", len(unique_products))
print("Valid unique descriptions:", valid)
print("Conflicting descriptions:", conflicting)

Total records: 105126
Unique descriptions: 49085
Valid unique descriptions: 48816
Conflicting descriptions: 269


## Build a balanced 5,000-record dataset

In [14]:
from collections import defaultdict
from datasets import Dataset
import random

random.seed(42)

groups = defaultdict(list)

for product in dataset:
    name = str(product.get("prod_name") or "").strip()
    desc = str(product.get("detail_desc") or "").strip()
    category = product.get("product_type_name")

    if not desc or not category:
        continue

    text = f"{name}. {desc}"
    key = " ".join(text.lower().split())

    groups[key].append({
        "input": text,
        "category": category
    })

by_category = defaultdict(list)

for records in groups.values():
    labels = {r["category"] for r in records}

    if len(labels) == 1:
        record = records[0]
        by_category[record["category"]].append(record)

selected = []

for category, records in by_category.items():
    random.shuffle(records)
    selected.extend(records[:100])

random.shuffle(selected)
selected = selected[:5000]

fashion_dataset = Dataset.from_list(selected)

print("Selected records:", len(fashion_dataset))
print("Categories:", len(set(fashion_dataset["category"])))

print("\nExample:")
print(fashion_dataset[0])

Selected records: 5000
Categories: 120

Example:
{'input': 'Love Shack Low Brazilian. Fully lined bikini bottoms with a low waist and half-string back.', 'category': 'Swimwear bottom'}


## Save the dataset to Google Drive

In [15]:
from google.colab import drive
import os

drive.mount("/content/drive")

project_dir = "/content/drive/MyDrive/fashion-distillation"
os.makedirs(project_dir, exist_ok=True)

dataset_path = f"{project_dir}/fashion_dataset_5000.jsonl"

fashion_dataset.to_json(
    dataset_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Dataset saved:", dataset_path)
print("Total records:", len(fashion_dataset))

Mounted at /content/drive


Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Dataset saved: /content/drive/MyDrive/fashion-distillation/fashion_dataset_5000.jsonl
Total records: 5000


## Load the teacher model

In [16]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

teacher_id = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

teacher_tokenizer = AutoTokenizer.from_pretrained(
    teacher_id
)

teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

teacher_model.eval()

print("Teacher model loaded successfully!")
print("Model:", teacher_id)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Teacher model loaded successfully!
Model: Qwen/Qwen2.5-7B-Instruct
GPU memory allocated: 5.18 GB


## Test the teacher on one fashion product

In [23]:
import json
import torch

product = fashion_dataset[0]["input"]

system_prompt = """
You are a fashion product attribute extractor.

Extract exactly these seven attributes:
category, color, material, fit, pattern, sleeve_length, neckline.

Rules:
- Use only information explicitly stated in the product text.
- Do not guess missing attributes.
- Use null for missing attributes.
- Return only valid JSON.
- Do not include explanations.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": product}
]

inputs = teacher_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(teacher_model.device)

with torch.inference_mode():
    outputs = teacher_model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
        pad_token_id=teacher_tokenizer.eos_token_id
    )

response = teacher_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("PRODUCT:")
print(product)

print("\nTEACHER OUTPUT:")
print(response)

try:
    parsed = json.loads(response)
    print("\n✅ Valid JSON")
except json.JSONDecodeError as e:
    print("\n❌ Invalid JSON:", e)

PRODUCT:
Love Shack Low Brazilian. Fully lined bikini bottoms with a low waist and half-string back.

TEACHER OUTPUT:
{
  "category": "bikini bottoms",
  "color": null,
  "material": null,
  "fit": "low waist",
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}

✅ Valid JSON


## Test the teacher on five fashion descriptions

In [24]:
test_products = [
    "Men's oversized cotton T-shirt in navy blue with a crew neck.",
    "Women's floral midi dress with short puff sleeves.",
    "Slim-fit black denim jeans with a high-rise waist.",
    "Beige linen shirt with long sleeves and a mandarin collar.",
    "Red and white striped polyester sweater with a V-neck."
]

for product_text in test_products:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": product_text}
    ]

    inputs = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print("\nPRODUCT:", product_text)

    try:
        parsed = json.loads(response)
        print(json.dumps(parsed, indent=2))
        print("✅ Valid JSON")
    except json.JSONDecodeError:
        print("❌ Invalid JSON:", response)


PRODUCT: Men's oversized cotton T-shirt in navy blue with a crew neck.
{
  "category": "T-shirt",
  "color": "navy blue",
  "material": "cotton",
  "fit": "oversized",
  "pattern": null,
  "sleeve_length": null,
  "neckline": "crew neck"
}
✅ Valid JSON

PRODUCT: Women's floral midi dress with short puff sleeves.
{
  "category": "dress",
  "color": null,
  "material": null,
  "fit": null,
  "pattern": "floral",
  "sleeve_length": "short puff",
  "neckline": null
}
✅ Valid JSON

PRODUCT: Slim-fit black denim jeans with a high-rise waist.
{
  "category": "jeans",
  "color": "black",
  "material": null,
  "fit": "slim-fit",
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
✅ Valid JSON

PRODUCT: Beige linen shirt with long sleeves and a mandarin collar.
{
  "category": "shirt",
  "color": "beige",
  "material": "linen",
  "fit": null,
  "pattern": null,
  "sleeve_length": "long",
  "neckline": "mandarin collar"
}
✅ Valid JSON

PRODUCT: Red and white striped polyester sweate

In [25]:
system_prompt = """
You are a fashion product attribute extraction engine.

Extract exactly these seven attributes from the product title
and description:

category, color, material, fit, pattern, sleeve_length, neckline.

RULES:
1. Return only valid JSON with exactly seven keys.
2. Use JSON null when information is missing.
3. Never invent attributes.
4. Extract information only from the supplied text.

ATTRIBUTE GUIDELINES:
- category: Garment type, e.g. shirt, jeans, dress, kurta.
- color: Explicitly mentioned colors, including combinations.
- material: Explicit fabric names, including cotton, linen,
  denim, polyester, silk, wool, jersey and leather.
- fit: Only garment fits such as slim, regular, relaxed,
  oversized, loose or tailored.
- pattern: Solid, striped, floral, checked, printed, etc.
  Only output solid when explicitly stated.
- sleeve_length: Only length, e.g. sleeveless, short,
  three-quarter or long. Exclude sleeve styles such as puff.
- neckline: Neck or collar style, e.g. crew neck, V-neck,
  round neck, mandarin collar or polo collar.

IMPORTANT:
- Low-rise, mid-rise and high-rise describe waist rise,
  not garment fit.
- Do not confuse sleeve style with sleeve length.
- Do not infer colors from garment names or categories.
- Do not infer gender unless explicitly requested.
- Do not add extra fields.

Return JSON only.
"""

print("✅ Updated teacher prompt.")

✅ Updated teacher prompt.


## Retest with new system prompt

In [27]:
test_products = [
    "Women's relaxed-fit trousers with a high-rise waist.",
    "Women's floral midi dress with short puff sleeves.",
    "Slim-fit black denim jeans with a high-rise waist."
]

for product_text in test_products:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": product_text}
    ]

    inputs = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print("\nPRODUCT:", product_text)

    try:
        parsed = json.loads(response)
        print(json.dumps(parsed, indent=2))
        print("✅ Valid JSON")
    except json.JSONDecodeError:
        print("❌ Invalid JSON:", response)


PRODUCT: Women's relaxed-fit trousers with a high-rise waist.
{
  "category": "trousers",
  "color": null,
  "material": null,
  "fit": "relaxed",
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
✅ Valid JSON

PRODUCT: Women's floral midi dress with short puff sleeves.
{
  "category": "dress",
  "color": null,
  "material": null,
  "fit": null,
  "pattern": "floral",
  "sleeve_length": "short",
  "neckline": null
}
✅ Valid JSON

PRODUCT: Slim-fit black denim jeans with a high-rise waist.
{
  "category": "jeans",
  "color": "black",
  "material": "denim",
  "fit": "slim",
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
✅ Valid JSON


## Filter our training dataset

In [28]:
from datasets import load_dataset
from collections import Counter

dataset_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "fashion_dataset_5000.jsonl"
)

fashion_dataset = load_dataset(
    "json",
    data_files=dataset_path,
    split="train"
)

excluded_keywords = [
    "underwear", "bra", "lingerie",
    "brief", "panties", "bikini",
    "swimwear", "swimsuit", "swimming trunks",
    "thong", "bralette", "bodysuit"
]

def is_allowed(example):
    category = example["category"].lower()

    return not any(
        keyword in category
        for keyword in excluded_keywords
    )

filtered_dataset = fashion_dataset.filter(is_allowed)

print("Original records:", len(fashion_dataset))
print("Filtered records:", len(filtered_dataset))
print("Removed:", len(fashion_dataset) - len(filtered_dataset))

print("\nTop 10 remaining categories:")
for category, count in Counter(
    filtered_dataset["category"]
).most_common(10):
    print(f"{category}: {count}")

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Original records: 5000
Filtered records: 4205
Removed: 795

Top 10 remaining categories:
Shorts: 85
Garment Set: 80
Gloves: 78
Blazer: 77
Hair/alice band: 77
Socks: 77
Boots: 77
Other accessories: 76
Hair clip: 76
Trousers: 76


## Review remaining categories

In [29]:
from collections import Counter

category_counts = Counter(filtered_dataset["category"])

print("Total remaining categories:", len(category_counts))
print("\nAll categories:\n")

for category, count in sorted(category_counts.items()):
    print(f"{category:<35} {count}")

Total remaining categories: 105

All categories:

Alice band                          4
Baby Bib                            1
Backpack                            1
Bag                                 62
Ballerinas                          61
Beanie                              12
Belt                                70
Blanket                             1
Blazer                              77
Blouse                              68
Bootie                              18
Boots                               77
Bucket hat                          2
Bumbag                              1
Cap                                 3
Cap/peaked                          64
Cardigan                            69
Chem. cosmetics                     2
Coat                                69
Costumes                            36
Cushion                             1
Dog Wear                            11
Dog wear                            3
Dress                               72
Dungarees               

## Create a garment-only dataset

In [30]:
garment_categories = {
    "T-shirt", "Shirt", "Polo shirt", "Top", "Vest top",
    "Blouse", "Sweater", "Hoodie", "Cardigan",
    "Jacket", "Blazer", "Coat",
    "Trousers", "Outdoor trousers", "Jeans",
    "Shorts", "Leggings/Tights",
    "Dress", "Skirt",
    "Garment Set", "Tailored Waistcoat",
    "Outdoor Waistcoat"
}

garment_dataset = filtered_dataset.filter(
    lambda row: row["category"] in garment_categories
)

print("Previous records:", len(filtered_dataset))
print("Garment-only records:", len(garment_dataset))
print("Categories retained:", len(set(garment_dataset["category"])))

print("\nRetained categories:")
for category, count in sorted(
    Counter(garment_dataset["category"]).items()
):
    print(f"{category:<25} {count}")

Filter:   0%|          | 0/4205 [00:00<?, ? examples/s]

Previous records: 4205
Garment-only records: 1434
Categories retained: 21

Retained categories:
Blazer                    77
Blouse                    68
Cardigan                  69
Coat                      69
Dress                     72
Garment Set               80
Hoodie                    63
Jacket                    69
Leggings/Tights           74
Outdoor Waistcoat         66
Outdoor trousers          53
Polo shirt                74
Shirt                     69
Shorts                    85
Skirt                     59
Sweater                   71
T-shirt                   75
Tailored Waistcoat        26
Top                       69
Trousers                  76
Vest top                  70


In [31]:
import os

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "garment_dataset_1434.jsonl"
)

garment_dataset.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("✅ Dataset saved successfully!")
print("Path:", output_path)
print("Records:", len(garment_dataset))
print("Categories:", len(set(garment_dataset["category"])))

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

✅ Dataset saved successfully!
Path: /content/drive/MyDrive/fashion-distillation/garment_dataset_1434.jsonl
Records: 1434
Categories: 21


## Generate the first 5 teacher labels

In [32]:
import json
import torch

teacher_results = []

for i in range(5):
    record = garment_dataset[i]
    product_text = record["input"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": product_text}
    ]

    inputs = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    try:
        attributes = json.loads(response)
        status = "✅ Valid JSON"
    except json.JSONDecodeError:
        attributes = None
        status = "❌ Invalid JSON"

    teacher_results.append({
        "input": product_text,
        "catalog_category": record["category"],
        "attributes": attributes
    })

    print(f"\n--- PRODUCT {i+1} ---")
    print("Input:", product_text)
    print("Catalog category:", record["category"])
    print("Teacher output:", response)
    print(status)


--- PRODUCT 1 ---
Input: Mary tuxedo blazer.. Straight-cut jacket in a crêpe weave with a satin shawl collar, jetted front pockets with a flap, decorative buttons at the cuffs, and a single back vent. No fasteners. Lining made from viscose and recycled polyester.
Catalog category: Blazer
Teacher output: ```json
{
  "category": "blazer",
  "color": null,
  "material": "crêpe, viscose, recycled polyester",
  "fit": "straight-cut",
  "pattern": null,
  "sleeve_length": null,
  "neckline": "shawl collar"
}
```
❌ Invalid JSON

--- PRODUCT 2 ---
Input: PQ SULLY SILK MIX SKIRT. Long skirt in soft, heavily draping Tencel™ lyocell jersey. High waist with covered elastication. Lined.
Catalog category: Skirt
Teacher output: {
  "category": "skirt",
  "color": null,
  "material": "Tencel™ lyocell jersey",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
✅ Valid JSON

--- PRODUCT 3 ---
Input: THEA set. Patterned set with a top in soft jersey and shorts and a sun hat 

## Improve teacher output consistency

In [33]:
system_prompt += """

STRICT OUTPUT REQUIREMENTS:
- Output raw JSON only.
- Never wrap JSON in ```json or Markdown fences.
- Use exactly the seven specified keys.
- All values must be strings or JSON null.
- Use lowercase category names.

CATEGORY NORMALIZATION:
- "tee" means "t-shirt".
- "jacket" and "blazer" are different categories.
- "garment set" is the category for multi-item clothing sets.
- Prefer the product's specific garment type over generic "top".
- Never invent a more specific category than the text supports.

ATTRIBUTE ACCURACY:
- Do not infer neckline from an opening at the back of the neck.
- Only extract neckline when a specific neck or collar type
  is explicitly mentioned.
- Do not classify embroidery, sequins, beads or frills as patterns.
- "patterned" alone does not identify a specific pattern;
  return null unless the pattern type is stated.
- Distinguish main garment fabric from lining material.
  For material, prefer the main outer garment fabric.
- If multiple garment components have different materials,
  include only explicitly supported materials.
- Never guess color, fit, pattern or sleeve length.

Return only the JSON object.
"""

print("✅ Teacher prompt updated.")

✅ Teacher prompt updated.


In [34]:
import json
import re
import torch

expected_keys = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

for i in range(5):
    record = garment_dataset[i]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": record["input"]}
    ]

    inputs = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # Remove Markdown fences if the teacher adds them
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        response
    ).strip()

    print(f"\n--- PRODUCT {i+1} ---")
    print("Catalog category:", record["category"])

    try:
        attributes = json.loads(cleaned)

        valid_schema = (
            isinstance(attributes, dict)
            and set(attributes.keys()) == expected_keys
            and all(
                v is None or isinstance(v, str)
                for v in attributes.values()
            )
        )

        print(json.dumps(attributes, indent=2))
        print("✅ Valid JSON")
        print("✅ Valid schema" if valid_schema else "❌ Invalid schema")

    except json.JSONDecodeError:
        print("❌ Invalid JSON:", response)


--- PRODUCT 1 ---
Catalog category: Blazer
{
  "category": "jacket",
  "color": null,
  "material": "cr\u00eape",
  "fit": "straight-cut",
  "pattern": null,
  "sleeve_length": null,
  "neckline": "shawl collar"
}
✅ Valid JSON
✅ Valid schema

--- PRODUCT 2 ---
Catalog category: Skirt
{
  "category": "skirt",
  "color": null,
  "material": "tencel\u2122 lyocell jersey",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
✅ Valid JSON
✅ Valid schema

--- PRODUCT 3 ---
Catalog category: Garment Set
{
  "category": "garment set",
  "color": null,
  "material": "jersey, cotton",
  "fit": null,
  "pattern": "patterned",
  "sleeve_length": "short",
  "neckline": "small opening with a button at the back of the neck"
}
✅ Valid JSON
✅ Valid schema

--- PRODUCT 4 ---
Catalog category: T-shirt
{
  "category": "t-shirt",
  "color": null,
  "material": "jersey",
  "fit": "fitted",
  "pattern": null,
  "sleeve_length": "short",
  "neckline": null
}
✅ Valid JSON
✅ Valid sc

In [35]:
system_prompt += """

FINAL EXTRACTION RULES:

- The user input will include a CATALOG_CATEGORY and DESCRIPTION.
- Always use CATALOG_CATEGORY as the category value,
  converted to lowercase.
- Extract all other attributes only from DESCRIPTION.
- Never use the catalog category to guess missing attributes.
- If neckline is only described as an opening, return null.
- If pattern is only described as "patterned", return null.
- Words such as narrow, short, long or fitted describe garment
  fit only when they explicitly refer to the overall fit.
- Shoulder strap width is not garment fit.
- Sleeve length must not be inferred for sleeveless garments
  unless explicitly specified.

Return exactly seven keys as raw JSON.
"""

print("✅ Category-aware teacher prompt updated.")

✅ Category-aware teacher prompt updated.


In [36]:
import json
import torch

for i in range(5):
    record = garment_dataset[i]

    user_input = (
        f"CATALOG_CATEGORY: {record['category']}\n"
        f"DESCRIPTION: {record['input']}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    inputs = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # Remove Markdown code fences if present
    cleaned = response.removeprefix("```json")
    cleaned = cleaned.removeprefix("```")
    cleaned = cleaned.removesuffix("```").strip()

    print(f"\n--- PRODUCT {i+1} ---")
    print("Catalog:", record["category"])

    try:
        attributes = json.loads(cleaned)
        print(json.dumps(attributes, indent=2, ensure_ascii=False))

        expected_category = record["category"].lower()

        print(
            "✅ Category matched"
            if attributes.get("category") == expected_category
            else "❌ Category mismatch"
        )

    except json.JSONDecodeError:
        print("❌ Invalid JSON:", response)


--- PRODUCT 1 ---
Catalog: Blazer
{
  "category": "blazer",
  "color": null,
  "material": "crêpe",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "satin shawl collar"
}
✅ Category matched

--- PRODUCT 2 ---
Catalog: Skirt
{
  "category": "skirt",
  "color": null,
  "material": "tencel™ lyocell jersey",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
✅ Category matched

--- PRODUCT 3 ---
Catalog: Garment Set
{
  "category": "garment set",
  "color": null,
  "material": "jersey, cotton",
  "fit": null,
  "pattern": "patterned",
  "sleeve_length": "short",
  "neckline": null
}
✅ Category matched

--- PRODUCT 4 ---
Catalog: T-shirt
{
  "category": "t-shirt",
  "color": null,
  "material": "jersey",
  "fit": "fitted",
  "pattern": null,
  "sleeve_length": "short",
  "neckline": null
}
✅ Category matched

--- PRODUCT 5 ---
Catalog: Blouse
{
  "category": "blouse",
  "color": null,
  "material": "polyester",
  "fit": null,
  "pattern

In [37]:
import json
import torch
import os
from tqdm.auto import tqdm

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_20.jsonl"
)

expected_keys = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

results = []

for record in tqdm(garment_dataset.select(range(20))):
    user_input = (
        f"CATALOG_CATEGORY: {record['category']}\n"
        f"DESCRIPTION: {record['input']}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    inputs = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    cleaned = response.removeprefix("```json")
    cleaned = cleaned.removeprefix("```")
    cleaned = cleaned.removesuffix("```").strip()

    try:
        attributes = json.loads(cleaned)

        valid = (
            isinstance(attributes, dict)
            and set(attributes) == expected_keys
            and all(
                v is None or isinstance(v, str)
                for v in attributes.values()
            )
            and attributes["category"] == record["category"].lower()
        )

        if not valid:
            attributes = None

    except json.JSONDecodeError:
        attributes = None

    results.append({
        "input": record["input"],
        "catalog_category": record["category"],
        "attributes": attributes,
        "valid": attributes is not None
    })

with open(output_path, "w", encoding="utf-8") as f:
    for record in results:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Total:", len(results))
print("Structurally valid:", sum(r["valid"] for r in results))
print("Invalid:", sum(not r["valid"] for r in results))
print("Saved:", output_path)

  0%|          | 0/20 [00:00<?, ?it/s]

Total: 20
Structurally valid: 20
Invalid: 0
Saved: /content/drive/MyDrive/fashion-distillation/teacher_labels_pilot_20.jsonl


## Inspect the 20 generated labels

In [38]:
import json

pilot_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_20.jsonl"
)

with open(pilot_path, "r", encoding="utf-8") as f:
    pilot_records = [json.loads(line) for line in f]

for i, record in enumerate(pilot_records, start=1):
    print(f"\n{'='*50}")
    print(f"PRODUCT {i}")
    print(f"{'='*50}")

    print("INPUT:")
    print(record["input"])

    print("\nCATALOG CATEGORY:")
    print(record["catalog_category"])

    print("\nTEACHER ATTRIBUTES:")
    print(
        json.dumps(
            record["attributes"],
            indent=2,
            ensure_ascii=False
        )
    )


PRODUCT 1
INPUT:
Mary tuxedo blazer.. Straight-cut jacket in a crêpe weave with a satin shawl collar, jetted front pockets with a flap, decorative buttons at the cuffs, and a single back vent. No fasteners. Lining made from viscose and recycled polyester.

CATALOG CATEGORY:
Blazer

TEACHER ATTRIBUTES:
{
  "category": "blazer",
  "color": null,
  "material": "crêpe",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "satin shawl collar"
}

PRODUCT 2
INPUT:
PQ SULLY SILK MIX SKIRT. Long skirt in soft, heavily draping Tencel™ lyocell jersey. High waist with covered elastication. Lined.

CATALOG CATEGORY:
Skirt

TEACHER ATTRIBUTES:
{
  "category": "skirt",
  "color": null,
  "material": "tencel™ lyocell jersey",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}

PRODUCT 3
INPUT:
THEA set. Patterned set with a top in soft jersey and shorts and a sun hat in a cotton weave. Top with short, flounced sleeves, a chest pocket and a small openi

In [39]:
system_prompt += """

ATTRIBUTE CORRECTION RULES (HIGHEST PRIORITY):

FIT:
- Extract explicit overall garment fit descriptions.
- Examples: straight-cut, slim, relaxed, fitted, oversized.
- Do not derive fit from waist height, garment length,
  strap width, or elasticated waistbands.
- Return null when overall fit is unspecified.

PATTERN:
- Extract a specific visible textile pattern only.
- Examples: striped, floral, checked, polka dot, ribbed.
- "Patterned" without a specific type means null.
- Sequins, beads, embroidery and frills are embellishments,
  not patterns by themselves.

NECKLINE:
- Accept explicit neckline or collar constructions.
- Examples: V-neck, round neck, shawl collar,
  stand-up collar, polo neck, notch lapels.
- Ties, buttons, plackets and back openings alone
  are not neckline styles.
- Never infer neckline from fastening details.

MATERIAL:
- Prefer explicitly identified main garment fabrics.
- Do not confuse lining fabrics with outer materials.
- Do not infer fabric composition from the product category.

CATEGORY:
- Copy CATALOG_CATEGORY exactly in lowercase.
- Do not silently correct catalog category mismatches.

OUTPUT:
- Return exactly seven keys as raw JSON.
- Use null for unknown attributes.
- Do not add explanations.
"""

print("✅ Attribute correction rules added.")

✅ Attribute correction rules added.


## RETESt

In [40]:
import json
import torch

problem_indices = [0, 2, 6, 7, 9, 12, 13, 14, 18]

for idx in problem_indices:
    record = garment_dataset[idx]

    user_input = (
        f"CATALOG_CATEGORY: {record['category']}\n"
        f"DESCRIPTION: {record['input']}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

    inputs = teacher_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(teacher_model.device)

    with torch.inference_mode():
        outputs = teacher_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=teacher_tokenizer.eos_token_id
        )

    response = teacher_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    cleaned = response.removeprefix("```json")
    cleaned = cleaned.removeprefix("```")
    cleaned = cleaned.removesuffix("```").strip()

    print(f"\n--- PRODUCT {idx + 1} ---")
    print("Input:", record["input"])
    print("Catalog category:", record["category"])

    try:
        attributes = json.loads(cleaned)
        print(json.dumps(attributes, indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print("❌ Invalid JSON:", response)


--- PRODUCT 1 ---
Input: Mary tuxedo blazer.. Straight-cut jacket in a crêpe weave with a satin shawl collar, jetted front pockets with a flap, decorative buttons at the cuffs, and a single back vent. No fasteners. Lining made from viscose and recycled polyester.
Catalog category: Blazer
{
  "category": "blazer",
  "color": null,
  "material": "crêpe",
  "fit": "straight-cut",
  "pattern": null,
  "sleeve_length": null,
  "neckline": "shawl collar"
}

--- PRODUCT 3 ---
Input: THEA set. Patterned set with a top in soft jersey and shorts and a sun hat in a cotton weave. Top with short, flounced sleeves, a chest pocket and a small opening with a button at the back of the neck. Shorts with an elasticated waist and elasticated hems. Lined sun hat with ties under the chin. Width of brim 4.5 cm.
Catalog category: Garment Set
{
  "category": "garment set",
  "color": null,
  "material": "jersey, cotton",
  "fit": null,
  "pattern": "patterned",
  "sleeve_length": "short",
  "neckline": null
}

In [41]:
INVALID_PATTERN_VALUES = {
    "patterned", "sequined", "sequins",
    "embroidered", "embroidery",
    "beaded", "frills"
}

INVALID_NECKLINE_VALUES = {
    "ties", "button placket",
    "buttons", "back opening",
    "opening", "zipper"
}

def validate_attributes(attributes, product_text):
    result = attributes.copy()
    text = product_text.lower()

    pattern = result.get("pattern")
    neckline = result.get("neckline")
    fit = result.get("fit")

    if isinstance(pattern, str):
        if pattern.lower().strip() in INVALID_PATTERN_VALUES:
            result["pattern"] = None

    if isinstance(neckline, str):
        if neckline.lower().strip() in INVALID_NECKLINE_VALUES:
            result["neckline"] = None

    if isinstance(fit, str):
        if (
            fit.lower().strip() == "regular"
            and "regular waist" in text
            and not any(
                phrase in text
                for phrase in [
                    "regular fit",
                    "regular-fit",
                    "regular cut"
                ]
            )
        ):
            result["fit"] = None

    return result


print("✅ Attribute validation rules initialized.")

✅ Attribute validation rules initialized.


In [42]:
test_cases = [
    {
        "input": "Patterned set with a top and shorts.",
        "attributes": {"pattern": "patterned"}
    },
    {
        "input": "Dress with ties at the neckline.",
        "attributes": {"neckline": "ties"}
    },
    {
        "input": "Top in sequined velvet.",
        "attributes": {"pattern": "sequined"}
    },
    {
        "input": "Long-sleeved top with a button placket.",
        "attributes": {"neckline": "button placket"}
    },
    {
        "input": "Corduroy skirt with a regular waist.",
        "attributes": {"fit": "regular"}
    }
]

for i, case in enumerate(test_cases, 1):
    corrected = validate_attributes(
        case["attributes"],
        case["input"]
    )

    print(f"\nTest {i}")
    print("Before:", case["attributes"])
    print("After: ", corrected)


Test 1
Before: {'pattern': 'patterned'}
After:  {'pattern': None}

Test 2
Before: {'neckline': 'ties'}
After:  {'neckline': None}

Test 3
Before: {'pattern': 'sequined'}
After:  {'pattern': None}

Test 4
Before: {'neckline': 'button placket'}
After:  {'neckline': None}

Test 5
Before: {'fit': 'regular'}
After:  {'fit': None}


In [43]:
import json
import torch
from tqdm.auto import tqdm

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_21_40.jsonl"
)

expected_keys = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

valid_count = 0
invalid_count = 0

with open(output_path, "w", encoding="utf-8") as f:
    for i in tqdm(range(20, 40)):
        record = garment_dataset[i]

        user_input = (
            f"CATALOG_CATEGORY: {record['category']}\n"
            f"DESCRIPTION: {record['input']}"
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]

        inputs = teacher_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(teacher_model.device)

        with torch.inference_mode():
            outputs = teacher_model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=teacher_tokenizer.eos_token_id
            )

        response = teacher_tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()

        cleaned = response.removeprefix("```json")
        cleaned = cleaned.removeprefix("```")
        cleaned = cleaned.removesuffix("```").strip()

        try:
            attributes = json.loads(cleaned)

            valid = (
                isinstance(attributes, dict)
                and set(attributes) == expected_keys
                and all(
                    v is None or isinstance(v, str)
                    for v in attributes.values()
                )
                and attributes["category"] == record["category"].lower()
            )

            if valid:
                attributes = validate_attributes(
                    attributes, record["input"]
                )
                valid_count += 1
            else:
                attributes = None
                invalid_count += 1

        except json.JSONDecodeError:
            attributes = None
            invalid_count += 1

        result = {
            "input": record["input"],
            "catalog_category": record["category"],
            "attributes": attributes,
            "valid": attributes is not None
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()

print("\nTotal:", valid_count + invalid_count)
print("Structurally valid:", valid_count)
print("Invalid:", invalid_count)
print("Saved:", output_path)

  0%|          | 0/20 [00:00<?, ?it/s]


Total: 20
Structurally valid: 19
Invalid: 1
Saved: /content/drive/MyDrive/fashion-distillation/teacher_labels_pilot_21_40.jsonl


In [44]:
import json

pilot_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_21_40.jsonl"
)

with open(pilot_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

invalid = [r for r in records if not r["valid"]]
valid = [r for r in records if r["valid"]]

print("=== INVALID RECORDS ===")

for record in invalid:
    print("\nInput:", record["input"])
    print("Catalog:", record["catalog_category"])
    print("Attributes:", record["attributes"])

print("\n=== FIRST 5 VALID RECORDS ===")

for i, record in enumerate(valid[:5], 1):
    print(f"\n--- VALID PRODUCT {i} ---")
    print("Input:", record["input"])
    print("Catalog:", record["catalog_category"])
    print(
        "Attributes:",
        json.dumps(
            record["attributes"],
            indent=2,
            ensure_ascii=False
        )
    )

=== INVALID RECORDS ===

Input: Chelsea wd waistcoat TP. Waistcoat in twill with buttons down the front and welt front pockets. Lined.
Catalog: Tailored Waistcoat
Attributes: None

=== FIRST 5 VALID RECORDS ===

--- VALID PRODUCT 1 ---
Input: FRANKLIN HOOD. Top in lightweight sweatshirt fabric made from a cotton blend with a kangaroo pocket and wrapover, jersey-lined hood with a decorative drawstring. Long sleeves and ribbing at the cuffs and hem.
Catalog: Hoodie
Attributes: {
  "category": "hoodie",
  "color": null,
  "material": "cotton blend",
  "fit": null,
  "pattern": null,
  "sleeve_length": "long",
  "neckline": null
}

--- VALID PRODUCT 2 ---
Input: VIOLA 2-pack (TVP). Long-sleeved, polo-neck tops in organic cotton jersey.
Catalog: Polo shirt
Attributes: {
  "category": "polo shirt",
  "color": null,
  "material": "organic cotton jersey",
  "fit": null,
  "pattern": null,
  "sleeve_length": "long",
  "neckline": "polo neck"
}

--- VALID PRODUCT 3 ---
Input: Justin tank SB 3-p.

In [45]:
previous_validate = validate_attributes

def validate_attributes(attributes, product_text):
    result = previous_validate(attributes, product_text)
    text = product_text.lower()

    # Convert string nulls into actual JSON nulls
    for key, value in result.items():
        if isinstance(value, str) and value.strip().lower() in {
            "null", "none", "unknown", "n/a"
        }:
            result[key] = None

    # Neckline values that describe decoration, not shape
    neckline = result.get("neckline")

    if isinstance(neckline, str):
        if neckline.lower().strip() in {
            "ribbed trim", "trim", "ties",
            "button placket", "collar"
        }:
            result["neckline"] = None

    # Reject unsupported neckline claims
    neckline = result.get("neckline")

    if isinstance(neckline, str):
        normalized = neckline.lower().strip()

        if normalized not in text:
            result["neckline"] = None

    # Avoid inferring sleeveless from garment category
    sleeve = result.get("sleeve_length")

    if sleeve == "sleeveless":
        if not any(
            term in text
            for term in ["sleeveless", "no sleeves", "without sleeves"]
        ):
            result["sleeve_length"] = None

    return result

print("✅ Validation function updated.")

✅ Validation function updated.


In [46]:
import json

pilot_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_21_40.jsonl"
)

with open(pilot_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

corrected_records = []
changed_count = 0

for record in records:
    if not record["valid"] or record["attributes"] is None:
        continue

    original = record["attributes"]

    corrected = validate_attributes(
        original,
        record["input"]
    )

    if original != corrected:
        changed_count += 1

        print("\n" + "=" * 50)
        print("PRODUCT:", record["input"][:150])

        for key in original:
            if original[key] != corrected[key]:
                print(
                    f"{key}: "
                    f"{original[key]!r} → {corrected[key]!r}"
                )

    corrected_records.append({
        **record,
        "attributes": corrected
    })

print("\n=== VALIDATION SUMMARY ===")
print("Original records:", len(records))
print("Valid records reviewed:", len(corrected_records))
print("Records modified:", changed_count)


PRODUCT: VIOLA 2-pack (TVP). Long-sleeved, polo-neck tops in organic cotton jersey.
neckline: 'polo neck' → None

PRODUCT: Justin tank SB 3-p. Vest tops in soft cotton jersey with a ribbed trim around the neckline and armholes.
sleeve_length: 'sleeveless' → None
neckline: 'ribbed trim' → None

PRODUCT: Sam price shirt PLUSSIZE. Shirt in a patterned cotton weave with a collar, buttons down the front, long sleeves with buttoned cuffs, and a rounded hem
fit: 'null' → None
neckline: 'round neck' → None

PRODUCT: Lufsen. Short halterneck dress in a crêpe weave with a V-neck, decorative tie detail at one side and press-studs at the back of the neck. Silicone tri
sleeve_length: 'sleeveless' → None

PRODUCT: KALDEN FAIRISLE ROLLNECK. Polo-neck jumper in a knit containing some wool with long sleeves and ribbing at the cuffs and hem.
neckline: 'polo neck' → None

PRODUCT: DIV Chung dress. Short, round-necked dress in a crêpe weave with an opening and covered button at the back of the neck, an e

In [47]:
# Preserve the existing validator
previous_validate_v2 = validate_attributes

def validate_attributes(attributes, product_text):
    result = previous_validate_v2(attributes, product_text)
    text = product_text.lower()

    # Restore valid neckline values when supported
    neckline_variants = {
        "polo neck": [
            "polo neck", "polo-neck", "polo-necked"
        ],
        "round neck": [
            "round neck", "round-neck", "round-necked"
        ],
        "v-neck": [
            "v-neck", "v neck", "v-necked"
        ]
    }

    original_neckline = attributes.get("neckline")

    if isinstance(original_neckline, str):
        normalized = original_neckline.lower().strip()

        if normalized in neckline_variants:
            if any(
                phrase in text
                for phrase in neckline_variants[normalized]
            ):
                result["neckline"] = normalized

    return result


# Test known examples
tests = [
    ("polo neck", "Long-sleeved polo-neck tops"),
    ("round neck", "Short round-necked dress"),
    ("v-neck", "A blouse featuring a V-neck"),
    ("ribbed trim", "Ribbed trim around the neckline")
]

for neckline, description in tests:
    corrected = validate_attributes(
        {"neckline": neckline},
        description
    )

    print(f"{neckline} → {corrected['neckline']}")

polo neck → polo neck
round neck → round neck
v-neck → v-neck
ribbed trim → None


In [48]:
import json

input_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_21_40.jsonl"
)

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_21_40_cleaned.jsonl"
)

with open(input_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

cleaned = []
modified = 0

for record in records:
    if not record["valid"] or record["attributes"] is None:
        continue

    corrected = validate_attributes(
        record["attributes"],
        record["input"]
    )

    if corrected != record["attributes"]:
        modified += 1

    cleaned.append({
        **record,
        "attributes": corrected
    })

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Cleaned pilot saved")
print("Valid records:", len(cleaned))
print("Modified records:", modified)
print("Output:", output_path)

✅ Cleaned pilot saved
Valid records: 19
Modified records: 4
Output: /content/drive/MyDrive/fashion-distillation/teacher_labels_pilot_21_40_cleaned.jsonl


In [49]:
import json

original_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "teacher_labels_pilot_21_40.jsonl"
)

with open(original_path, encoding="utf-8") as f:
    original_records = [json.loads(line) for line in f]

for i, record in enumerate(original_records, 1):
    if not record["valid"] or record["attributes"] is None:
        continue

    before = record["attributes"]
    after = validate_attributes(before, record["input"])

    if before != after:
        print(f"\n--- PRODUCT {i} ---")
        print("Description:", record["input"])

        for key in before:
            if before[key] != after[key]:
                print(
                    f"{key}: {before[key]!r} → {after[key]!r}"
                )


--- PRODUCT 3 ---
Description: Justin tank SB 3-p. Vest tops in soft cotton jersey with a ribbed trim around the neckline and armholes.
sleeve_length: 'sleeveless' → None
neckline: 'ribbed trim' → None

--- PRODUCT 5 ---
Description: Sam price shirt PLUSSIZE. Shirt in a patterned cotton weave with a collar, buttons down the front, long sleeves with buttoned cuffs, and a rounded hem.
fit: 'null' → None
neckline: 'round neck' → None

--- PRODUCT 7 ---
Description: Lufsen. Short halterneck dress in a crêpe weave with a V-neck, decorative tie detail at one side and press-studs at the back of the neck. Silicone trim at the back, a concealed zip in one side, seam at the waist and bell-shaped skirt. Lined front bodice.
sleeve_length: 'sleeveless' → None

--- PRODUCT 15 ---
Description: Magnus blouse. Wide, V-neck blouse in a viscose with a pompom trim at the top, gathered yoke at the back and long balloon sleeves with elasticated cuffs.
neckline: 'V-neck' → 'v-neck'


In [50]:
import re

category_patterns = {
    "dress": r"\bdress(?:es)?\b",
    "shirt": r"\bshirts?\b",
    "skirt": r"\bskirts?\b",
    "trousers": r"\btrousers?\b",
    "shorts": r"\bshorts\b",
    "jacket": r"\bjackets?\b",
    "coat": r"\bcoats?\b",
    "blazer": r"\bblazers?\b"
}

suspected_mismatches = []

for i, record in enumerate(garment_dataset):
    text = record["input"].lower()
    catalog = record["category"].lower()

    mentioned = [
        category
        for category, pattern in category_patterns.items()
        if re.search(pattern, text)
    ]

    if len(mentioned) == 1:
        detected = mentioned[0]

        # Simple heuristic, not a definitive mismatch
        if detected != catalog:
            suspected_mismatches.append({
                "index": i,
                "catalog": catalog,
                "mentioned": detected,
                "input": record["input"]
            })

print("Potential mismatches:", len(suspected_mismatches))

for record in suspected_mismatches[:10]:
    print("\nCatalog:", record["catalog"])
    print("Mentioned:", record["mentioned"])
    print("Description:", record["input"][:200])

Potential mismatches: 324

Catalog: garment set
Mentioned: shorts
Description: THEA set. Patterned set with a top in soft jersey and shorts and a sun hat in a cotton weave. Top with short, flounced sleeves, a chest pocket and a small opening with a button at the back of the neck

Catalog: blazer
Mentioned: jacket
Description: Miranda. Gently fitted, sleeveless jacket in woven fabric with a shawl collar and a concealed press-stud at the front. Tie belt at the waist, welt front pockets and a single back vent. Lined.

Catalog: top
Mentioned: dress
Description: Harper top v6. Short, straight-cut dress in sequined velvet with a stand-up collar, opening and button at the back of the neck and short sleeves. Jersey lining.

Catalog: t-shirt
Mentioned: shirt
Description: Bambi top. Jersey T-shirt with a small opening and button at the back of the neck. Large, double-layered collar in a cotton weave that has a frill trim and is divided at the back.

Catalog: polo shirt
Mentioned: shirt
Descripti

In [51]:
import re

# Only compare strongly distinct categories.
# Exclude sets and broad categories that overlap with others.

strict_categories = {
    "dress": r"\bdresses?\b",
    "skirt": r"\bskirts?\b",
    "trousers": r"\btrousers\b",
    "shorts": r"\bshorts\b",
    "coat": r"\bcoats?\b",
    "blazer": r"\bblazers?\b",
}

skip_categories = {
    "garment set",
    "top",
    "t-shirt",
    "polo shirt",
    "shirt",
    "blouse",
    "jacket",
    "hoodie",
    "sweater",
    "cardigan"
}

strong_mismatches = []

for i, record in enumerate(garment_dataset):
    catalog = record["category"].lower()
    text = record["input"].lower()

    if catalog in skip_categories:
        continue

    detected = [
        name
        for name, pattern in strict_categories.items()
        if re.search(pattern, text)
    ]

    if len(detected) == 1 and detected[0] != catalog:
        strong_mismatches.append({
            "index": i,
            "catalog": catalog,
            "detected": detected[0],
            "input": record["input"]
        })

print("Potential strong mismatches:", len(strong_mismatches))

for item in strong_mismatches[:10]:
    print("\nCatalog:", item["catalog"])
    print("Detected:", item["detected"])
    print("Description:", item["input"][:200])

Potential strong mismatches: 88

Catalog: dress
Detected: skirt
Description: Lufsen. Short halterneck dress in a crêpe weave with a V-neck, decorative tie detail at one side and press-studs at the back of the neck. Silicone trim at the back, a concealed zip in one side, seam a

Catalog: outdoor trousers
Detected: trousers
Description: SB rain trs fleece lining. Fleece-lined rain trousers in wind- and waterproof functional fabric with welded seams and reflective details. Press-studs at the sides, elasticated hems and a reinforced el

Catalog: outdoor waistcoat
Detected: coat
Description: Dion Handmade Coat. Coat in a soft, felted wool blend with a brushed finish. Notch lapels, a tie belt at the waist, concealed side pockets and a single back vent. Unlined.

Catalog: outdoor trousers
Detected: trousers
Description: ANT PANTS. Shell trousers in wind- and waterproof, breathable functional fabric with closed seams. Adjustable elasticated waist, a reflective strip on one leg and a reinforced

## Identify obvious catalog mismatches

In [52]:
import re

suspect_records = []

for i, record in enumerate(garment_dataset):
    category = record["category"].lower()
    text = record["input"].lower()

    if category == "outdoor waistcoat":
        if re.search(r"\b(coat|jacket|blazer)\b", text):
            suspect_records.append({
                "index": i,
                "category": record["category"],
                "input": record["input"]
            })

print("Suspected records:", len(suspect_records))

for record in suspect_records[:15]:
    print("\nIndex:", record["index"])
    print("Category:", record["category"])
    print("Description:", record["input"][:250])

Suspected records: 12

Index: 73
Category: Outdoor Waistcoat
Description: Dion Handmade Coat. Coat in a soft, felted wool blend with a brushed finish. Notch lapels, a tie belt at the waist, concealed side pockets and a single back vent. Unlined.

Index: 225
Category: Outdoor Waistcoat
Description: Tokyo sherling. Biker jacket in imitation leather with a faux shearling lining. Notch lapels, a diagonal zip down the front and front pockets.

Index: 403
Category: Outdoor Waistcoat
Description: Memphis pile. Jacket in soft faux shearling with a stand-up collar, zip down the front, gently dropped shoulders and long sleeves with covered elastication at the cuffs. Diagonal, welt front pockets and an elasticated, contrasting colour drawstring a

Index: 520
Category: Outdoor Waistcoat
Description: New york pile. Teddy jacket in soft faux shearling with a zip down the front, concealed pockets in the side seams and covered elastication at the cuffs and hem. Lined in woven fabric made from recycled

In [54]:
# Remove the 12 suspicious records identified in previous cell
suspect_indices = {
    record["index"]
    for record in suspect_records
}

clean_garment_dataset = garment_dataset.select([
    i for i in range(len(garment_dataset))
    if i not in suspect_indices
])

print("Original garment records:", len(garment_dataset))
print("Confirmed mismatches excluded:", len(suspect_indices))
print("Remaining records:", len(clean_garment_dataset))

# Save cleaned source data
clean_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "garment_dataset_clean_v2.jsonl"
)

clean_garment_dataset.to_json(
    clean_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("\n✅ Clean dataset saved:", clean_path)

Original garment records: 1434
Confirmed mismatches excluded: 12
Remaining records: 1422


Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]


✅ Clean dataset saved: /content/drive/MyDrive/fashion-distillation/garment_dataset_clean_v2.jsonl


## Create training and evaluation splits
Before generating teacher labels, we should reserve some descriptions for evaluating our student model. This avoids training and testing on the same products.

We'll allocate 80% for training, 10% for validation and 10% for testing

In [55]:
import os

splits = clean_garment_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

remaining = splits["test"].train_test_split(
    test_size=0.50,
    seed=42
)

train_dataset = splits["train"]
validation_dataset = remaining["train"]
test_dataset = remaining["test"]

split_dir = (
    "/content/drive/MyDrive/fashion-distillation/splits"
)
os.makedirs(split_dir, exist_ok=True)

for name, ds in [
    ("train", train_dataset),
    ("validation", validation_dataset),
    ("test", test_dataset)
]:
    path = f"{split_dir}/{name}.jsonl"
    ds.to_json(path, orient="records", lines=True)
    print(f"{name}: {len(ds)} records → {path}")

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

train: 1137 records → /content/drive/MyDrive/fashion-distillation/splits/train.jsonl


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

validation: 142 records → /content/drive/MyDrive/fashion-distillation/splits/validation.jsonl


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

test: 143 records → /content/drive/MyDrive/fashion-distillation/splits/test.jsonl


## Generate the first 50 training labels

In [56]:
import os
import json
import torch
from tqdm.auto import tqdm

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

expected_keys = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

# Resume from previously saved records
completed = set()

if os.path.exists(output_path):
    with open(output_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                completed.add(json.loads(line)["index"])

print("Already processed:", len(completed))

# Process only the first 50 training examples
with open(output_path, "a", encoding="utf-8") as f:
    for i in tqdm(range(min(50, len(train_dataset)))):

        if i in completed:
            continue

        record = train_dataset[i]

        user_input = (
            f"CATALOG_CATEGORY: {record['category']}\n"
            f"DESCRIPTION: {record['input']}"
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]

        try:
            inputs = teacher_tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt"
            ).to(teacher_model.device)

            with torch.inference_mode():
                outputs = teacher_model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False,
                    pad_token_id=teacher_tokenizer.eos_token_id
                )

            response = teacher_tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True
            ).strip()

            cleaned = response.removeprefix("```json")
            cleaned = cleaned.removeprefix("```")
            cleaned = cleaned.removesuffix("```").strip()

            attributes = json.loads(cleaned)

            valid = (
                isinstance(attributes, dict)
                and set(attributes) == expected_keys
                and all(
                    v is None or isinstance(v, str)
                    for v in attributes.values()
                )
                and attributes["category"] == record["category"].lower()
            )

            if valid:
                attributes = validate_attributes(
                    attributes, record["input"]
                )
            else:
                attributes = None

            error = None

        except Exception as e:
            response = locals().get("response", "")
            attributes = None
            valid = False
            error = str(e)

        result = {
            "index": i,
            "input": record["input"],
            "catalog_category": record["category"],
            "attributes": attributes,
            "valid": valid,
            "raw_teacher_response": response,
            "error": error
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()

print("\n✅ First batch processed.")

with open(output_path, encoding="utf-8") as f:
    saved = [json.loads(line) for line in f if line.strip()]

batch = [r for r in saved if r["index"] < 50]

print("Processed:", len(batch))
print("Valid:", sum(r["valid"] for r in batch))
print("Invalid:", sum(not r["valid"] for r in batch))
print("Saved:", output_path)

Already processed: 0


  0%|          | 0/50 [00:00<?, ?it/s]


✅ First batch processed.
Processed: 50
Valid: 45
Invalid: 5
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


## Diagnose the five invalid records

In [57]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

invalid_records = [
    r for r in records
    if r["index"] < 50 and not r["valid"]
]

print("Invalid records:", len(invalid_records))

for record in invalid_records:
    print("\n" + "=" * 60)
    print("INDEX:", record["index"])
    print("CATEGORY:", record["catalog_category"])
    print("INPUT:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record.get("raw_teacher_response"))
    print("\nERROR:", record.get("error"))

Invalid records: 5

INDEX: 12
CATEGORY: Hoodie
INPUT: DIV Lollo jumpsuit. Playsuit in sequined, glittery jersey with spaghetti shoulder straps. V-neck at the front, straight cut at the back, a gathered seam at the waist and short, wide legs.

RAW TEACHER RESPONSE:
{
  "category": "playsuit",
  "color": null,
  "material": "jersey",
  "fit": "straight cut",
  "pattern": "sequined",
  "sleeve_length": "short",
  "neckline": "v-neck"
}

ERROR: None

INDEX: 25
CATEGORY: Outdoor Waistcoat
INPUT: Lorelai vest. Lightly padded gilet with a stand-up collar, zip down the front, side pockets and a short frill trim at the hem. Detachable, lined hood with a short frill trim. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": null,
  "fit": null,
  "pattern": null,
  "sleeve_length": "none",
  "neckline": "stand-up collar"
}

ERROR: None

INDEX: 34
CATEGORY: Outdoor Waistcoat
INPUT: ANYA vest. Padded, gently tailored gilet with a lined drawstring hood, zip down

## Separate genuine errors from category aliases

In [58]:
import json

category_aliases = {
    "outdoor waistcoat": {"waistcoat", "gilet"},
    "leggings/tights": {"leggings", "tights"}
}

genuine_mismatches = []
valid_aliases = []

for record in invalid_records:
    raw = record.get("raw_teacher_response", "")

    try:
        attributes = json.loads(raw)
    except json.JSONDecodeError:
        continue

    catalog = record["catalog_category"].lower()
    predicted = str(attributes.get("category", "")).lower()

    if predicted in category_aliases.get(catalog, set()):
        valid_aliases.append(record["index"])
        status = "ALIAS"
    else:
        genuine_mismatches.append(record["index"])
        status = "REVIEW / MISMATCH"

    print(
        f"Index {record['index']}: "
        f"{catalog} → {predicted} [{status}]"
    )

print("\nValid aliases:", valid_aliases)
print("Needs review:", genuine_mismatches)

Index 12: hoodie → playsuit [REVIEW / MISMATCH]
Index 25: outdoor waistcoat → waistcoat [ALIAS]
Index 34: outdoor waistcoat → waistcoat [ALIAS]
Index 36: leggings/tights → leggings [ALIAS]
Index 42: outdoor waistcoat → waistcoat [ALIAS]

Valid aliases: [25, 34, 36, 42]
Needs review: [12]


## Repair the five rejected training labels

In [59]:
import json

recovered_indices = {25, 34, 36}
excluded_indices = {12, 42}

recovered = []
excluded = []

for record in invalid_records:
    idx = record["index"]

    if idx in excluded_indices:
        excluded.append(idx)
        continue

    if idx in recovered_indices:
        attributes = json.loads(
            record["raw_teacher_response"]
        )

        attributes["category"] = (
            record["catalog_category"].lower()
        )

        attributes = validate_attributes(
            attributes,
            record["input"]
        )

        recovered.append({
            **record,
            "attributes": attributes,
            "valid": True,
            "recovered_from_alias": True
        })

print("Recovered records:", len(recovered))
print("Excluded mismatches:", excluded)

for record in recovered:
    print(
        f"Index {record['index']}:",
        record["attributes"]["category"]
    )

Recovered records: 3
Excluded mismatches: [12, 42]
Index 25: outdoor waistcoat
Index 34: outdoor waistcoat
Index 36: leggings/tights


## Save the corrected training batch

In [60]:
import json

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_50.jsonl"
)

# Index recovered records for lookup
recovered_by_index = {
    record["index"]: record
    for record in recovered
}

cleaned_batch = []

for record in records:
    idx = record["index"]

    if idx >= 50 or idx in excluded_indices:
        continue

    if idx in recovered_by_index:
        record = recovered_by_index[idx]

    if not record["valid"] or record["attributes"] is None:
        continue

    cleaned_batch.append({
        **record,
        "attributes": validate_attributes(
            record["attributes"],
            record["input"]
        )
    })

cleaned_batch.sort(key=lambda r: r["index"])

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Corrected batch saved")
print("Usable records:", len(cleaned_batch))
print("Excluded indices:", sorted(excluded_indices))
print("Saved:", output_path)

✅ Corrected batch saved
Usable records: 48
Excluded indices: [12, 42]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_50.jsonl


## Audit a random sample of training labels

In [61]:
import json
import random

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_50.jsonl"
)

with open(path, encoding="utf-8") as f:
    cleaned_records = [
        json.loads(line)
        for line in f if line.strip()
    ]

random.seed(42)
sample = random.sample(
    cleaned_records,
    min(10, len(cleaned_records))
)

for i, record in enumerate(sample, 1):
    print(f"\n{'=' * 55}")
    print(f"SAMPLE {i} | INDEX {record['index']}")
    print(f"{'=' * 55}")

    print("DESCRIPTION:")
    print(record["input"])

    print("\nCATALOG CATEGORY:")
    print(record["catalog_category"])

    print("\nEXTRACTED ATTRIBUTES:")
    print(json.dumps(
        record["attributes"],
        indent=2,
        ensure_ascii=False
    ))


SAMPLE 1 | INDEX 41
DESCRIPTION:
MAMA LAGUNA 2LAYER TANK. Double-layered sports vest top in soft, fast-drying functional fabric with a racer back.

CATALOG CATEGORY:
Vest top

EXTRACTED ATTRIBUTES:
{
  "category": "vest top",
  "color": null,
  "material": "soft, fast-drying functional fabric",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}

SAMPLE 2 | INDEX 7
DESCRIPTION:
Madde Play Pant. Pull-on trousers in cotton twill with elasticated ribbing and decorative ties at the waist and a fake fly. Side and back pockets, and ribbed hems.

CATALOG CATEGORY:
Trousers

EXTRACTED ATTRIBUTES:
{
  "category": "trousers",
  "color": null,
  "material": "cotton twill",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}

SAMPLE 3 | INDEX 1
DESCRIPTION:
Daniels checked emb shirt. Shirt in woven fabric with a collar, buttons down the front and embroidery at the top and back. Dropped shoulders and long sleeves with buttoned cuffs. Short sl

## Training-label quality audit

In [62]:
import re

review_flags = []

for record in cleaned_records:
    text = record["input"].lower()
    category = record["catalog_category"].lower()
    attrs = record["attributes"]
    reasons = []

    # Obvious garment-category contradictions
    if category == "top" and re.search(
        r"\b(hooded jacket|biker jacket|puffer jacket)\b",
        text
    ):
        reasons.append("Possible jacket categorized as top")

    if category == "shirt" and re.search(
        r"\b(short-sleeved top|sleeveless top)\b",
        text
    ):
        reasons.append("Possible top categorized as shirt")

    # Lining included in the extracted material
    material = str(attrs.get("material") or "").lower()

    if "lining" in material:
        reasons.append("Material includes lining")

    if reasons:
        review_flags.append({
            "index": record["index"],
            "category": record["catalog_category"],
            "input": record["input"],
            "attributes": attrs,
            "reasons": reasons
        })

print("Records flagged:", len(review_flags))

for item in review_flags:
    print("\n" + "=" * 50)
    print("Index:", item["index"])
    print("Category:", item["category"])
    print("Description:", item["input"])
    print("Reasons:", item["reasons"])

Records flagged: 3

Index: 5
Category: Jacket
Description: SULIMA jacket. Outdoor jacket with lightly padded front and back sections in functional fabric, a lined hood and zip down the front. Zipped side pockets and long, ribbed raglan sleeves with thumbholes at the cuffs. Lightly brushed thermal lining in the sleeves and side sections. Partly lined.
Reasons: ['Material includes lining']

Index: 6
Category: Shirt
Description: EQ Avril velvet top. Short, short-sleeved top in crushed velvet with a small stand-up collar and visible zip at the top.
Reasons: ['Possible top categorized as shirt']

Index: 15
Category: Top
Description: BB Alf pile hoddie. Hooded jacket in soft pile with a small fabric appliqué on the front, a zip down the front with a chin guard, and ribbed cuffs. Slightly longer at the back.
Reasons: ['Possible jacket categorized as top']


In [63]:
import json

exclude_indices = {6, 15}

audited_records = []

for record in cleaned_records:
    idx = record["index"]

    if idx in exclude_indices:
        continue

    updated = {
        **record,
        "attributes": record["attributes"].copy()
    }

    if idx == 5:
        updated["attributes"]["material"] = "functional fabric"

    audited_records.append(updated)

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_audited_50.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in audited_records:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("✅ Audited training batch saved")
print("Records retained:", len(audited_records))
print("Excluded indices:", sorted(exclude_indices))
print("Material corrections:", 1)
print("Output:", output_path)

✅ Audited training batch saved
Records retained: 46
Excluded indices: [6, 15]
Material corrections: 1
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_audited_50.jsonl


## Generate the next 50 teacher labels

In [64]:
import json
import torch
from tqdm.auto import tqdm

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

# Read existing indices to avoid duplicate processing
completed = set()

with open(output_path, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            completed.add(json.loads(line)["index"])

new_count = 0

with open(output_path, "a", encoding="utf-8") as f:
    for i in tqdm(range(50, 100)):

        if i in completed:
            continue

        record = train_dataset[i]

        user_input = (
            f"CATALOG_CATEGORY: {record['category']}\n"
            f"DESCRIPTION: {record['input']}"
        )

        response = ""
        error = None
        attributes = None
        valid = False

        try:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_input}
            ]

            inputs = teacher_tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt"
            ).to(teacher_model.device)

            with torch.inference_mode():
                outputs = teacher_model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False,
                    pad_token_id=teacher_tokenizer.eos_token_id
                )

            response = teacher_tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True
            ).strip()

            cleaned = response.removeprefix("```json")
            cleaned = cleaned.removeprefix("```")
            cleaned = cleaned.removesuffix("```").strip()

            candidate = json.loads(cleaned)

            valid = (
                isinstance(candidate, dict)
                and set(candidate) == expected_keys
                and all(
                    v is None or isinstance(v, str)
                    for v in candidate.values()
                )
                and candidate["category"] == record["category"].lower()
            )

            if valid:
                attributes = validate_attributes(
                    candidate, record["input"]
                )

        except Exception as e:
            error = str(e)

        result = {
            "index": i,
            "input": record["input"],
            "catalog_category": record["category"],
            "attributes": attributes,
            "valid": valid,
            "raw_teacher_response": response,
            "error": error
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()
        new_count += 1

print("\n✅ Batch completed")
print("New records processed:", new_count)

with open(output_path, encoding="utf-8") as f:
    batch = [
        json.loads(line)
        for line in f
        if line.strip()
        and 50 <= json.loads(line)["index"] < 100
    ]

print("Batch records:", len(batch))
print("Valid:", sum(r["valid"] for r in batch))
print("Invalid:", sum(not r["valid"] for r in batch))

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch completed
New records processed: 50
Batch records: 50
Valid: 46
Invalid: 4


## Diagnose the four invalid records

In [65]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    batch = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in batch
    if 50 <= r["index"] < 100 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nTEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 4

INDEX: 65
CATALOG: Tailored Waistcoat
DESCRIPTION: Floyd puffer vest. Padded gilet in windproof, water-repellent functional fabric with a concealed stand up collar and hood with an elastic drawstring. Taped zip down the front with a chin guard, zipped, fleece-lined side pockets and a concealed elastic drawstring at the hem. Lined.

TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "windproof, water-repellent functional fabric",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "stand up collar"
}
ERROR: None

INDEX: 71
CATALOG: Outdoor Waistcoat
DESCRIPTION: Sage vest fur. Gilet in supersoft faux fur with a concealed fastening at the top. Lined.

TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": "sage",
  "material": "faux fur",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
ERROR: None

INDEX: 77
CATALOG: Tailored Waistcoat
DESCRIPTION: Clark PQ waistcoat. Suit waistcoat in 

## Recover the four records

In [66]:
import json

recovered_batch_2 = []

for record in invalid:
    idx = record["index"]

    attributes = json.loads(
        record["raw_teacher_response"]
    )

    # Restore the original catalog category
    attributes["category"] = (
        record["catalog_category"].lower()
    )

    # Correct ambiguous color from product name
    if idx == 71:
        attributes["color"] = None

    attributes = validate_attributes(
        attributes,
        record["input"]
    )

    recovered_batch_2.append({
        **record,
        "attributes": attributes,
        "valid": True,
        "recovered_from_alias": True
    })

print("Recovered:", len(recovered_batch_2))

for record in recovered_batch_2:
    print(
        f"Index {record['index']}: "
        f"{record['attributes']['category']} | "
        f"Color: {record['attributes']['color']}"
    )

Recovered: 4
Index 65: tailored waistcoat | Color: None
Index 71: outdoor waistcoat | Color: None
Index 77: tailored waistcoat | Color: None
Index 79: outdoor waistcoat | Color: None


In [67]:
import json

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_50_99.jsonl"
)

recovered_lookup = {
    r["index"]: r for r in recovered_batch_2
}

cleaned_batch_2 = []

for record in batch:
    idx = record["index"]

    if not (50 <= idx < 100):
        continue

    if idx in recovered_lookup:
        record = recovered_lookup[idx]

    if not record["valid"] or record["attributes"] is None:
        continue

    corrected = validate_attributes(
        record["attributes"],
        record["input"]
    )

    cleaned_batch_2.append({
        **record,
        "attributes": corrected
    })

cleaned_batch_2.sort(key=lambda r: r["index"])

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_2:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("✅ Second batch saved")
print("Records:", len(cleaned_batch_2))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_2)))
print("Recovered aliases:", len(recovered_batch_2))
print("Output:", output_path)

✅ Second batch saved
Records: 50
Unique indices: 50
Recovered aliases: 4
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_50_99.jsonl


## Combine the two cleaned batches

In [68]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

files = [
    "train_teacher_labels_audited_50.jsonl",
    "train_teacher_labels_cleaned_50_99.jsonl"
]

combined = {}

for filename in files:
    path = os.path.join(base_dir, filename)

    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            record = json.loads(line)
            combined[record["index"]] = record

combined_records = [
    combined[i] for i in sorted(combined)
]

output_path = os.path.join(
    base_dir,
    "train_teacher_labels_master.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in combined_records:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("✅ Master dataset created")
print("Total records:", len(combined_records))
print("Unique indices:", len(combined))
print("Missing indices:", sorted(set(range(100)) - set(combined)))
print("Saved:", output_path)

✅ Master dataset created
Total records: 96
Unique indices: 96
Missing indices: [6, 12, 15, 42]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master.jsonl


## Prepare the next batch (100–149)

In [69]:
import json
import torch
from tqdm.auto import tqdm

def generate_teacher_batch(start, end):
    output_path = (
        "/content/drive/MyDrive/fashion-distillation/"
        "train_teacher_labels.jsonl"
    )

    # Read existing records for resume support
    completed = set()

    if os.path.exists(output_path):
        with open(output_path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    completed.add(json.loads(line)["index"])

    new_count = 0
    valid_count = 0
    invalid_count = 0

    with open(output_path, "a", encoding="utf-8") as f:
        for i in tqdm(range(start, min(end, len(train_dataset)))):
            if i in completed:
                continue

            record = train_dataset[i]

            user_input = (
                f"CATALOG_CATEGORY: {record['category']}\n"
                f"DESCRIPTION: {record['input']}"
            )

            response = ""
            attributes = None
            error = None
            valid = False

            try:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_input}
                ]

                inputs = teacher_tokenizer.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors="pt"
                ).to(teacher_model.device)

                with torch.inference_mode():
                    outputs = teacher_model.generate(
                        **inputs,
                        max_new_tokens=200,
                        do_sample=False,
                        pad_token_id=teacher_tokenizer.eos_token_id
                    )

                response = teacher_tokenizer.decode(
                    outputs[0][inputs["input_ids"].shape[-1]:],
                    skip_special_tokens=True
                ).strip()

                cleaned = response.removeprefix("```json")
                cleaned = cleaned.removeprefix("```")
                cleaned = cleaned.removesuffix("```").strip()

                candidate = json.loads(cleaned)

                valid = (
                    isinstance(candidate, dict)
                    and set(candidate) == expected_keys
                    and all(
                        v is None or isinstance(v, str)
                        for v in candidate.values()
                    )
                    and candidate["category"] == record["category"].lower()
                )

                if valid:
                    attributes = validate_attributes(
                        candidate, record["input"]
                    )

            except Exception as e:
                error = str(e)

            result = {
                "index": i,
                "input": record["input"],
                "catalog_category": record["category"],
                "attributes": attributes,
                "valid": valid,
                "raw_teacher_response": response,
                "error": error
            }

            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()

            new_count += 1

            if valid:
                valid_count += 1
            else:
                invalid_count += 1

    print("\n✅ Batch complete")
    print("New records:", new_count)
    print("Valid:", valid_count)
    print("Invalid:", invalid_count)
    print("Saved:", output_path)

In [70]:
generate_teacher_batch(100, 150)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 48
Invalid: 2
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


## Inspect 2 invalid record

In [71]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 100 <= r["index"] < 150 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 2

INDEX: 110
CATALOG: Outdoor Waistcoat
DESCRIPTION: EVA vest DE unique style. Padded gilet with a detachable, lined hood, zip down the front and side pockets. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": null,
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
ERROR: None

INDEX: 135
CATALOG: Tailored Waistcoat
DESCRIPTION: Codger waistcoat. Waistcoat in woven fabric with buttons down the front, fake front pockets and an adjustable tab at the back. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "woven",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
ERROR: None


## Recover and save the third batch

In [72]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

recovered_indices = {110, 135}
cleaned_batch_3 = []

for record in records:
    idx = record["index"]

    if not (100 <= idx < 150):
        continue

    updated = record.copy()

    if idx in recovered_indices:
        attributes = json.loads(record["raw_teacher_response"])

        attributes["category"] = (
            record["catalog_category"].lower()
        )

        updated["attributes"] = attributes
        updated["valid"] = True
        updated["recovered_from_alias"] = True

    if not updated["valid"] or updated["attributes"] is None:
        continue

    updated["attributes"] = validate_attributes(
        updated["attributes"],
        updated["input"]
    )

    cleaned_batch_3.append(updated)

cleaned_batch_3.sort(key=lambda r: r["index"])

output_path = os.path.join(
    base_dir,
    "train_teacher_labels_cleaned_100_149.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_3:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Third batch saved")
print("Records:", len(cleaned_batch_3))
print("Recovered aliases:", len(recovered_indices))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_3)))
print("Output:", output_path)

✅ Third batch saved
Records: 50
Recovered aliases: 2
Unique indices: 50
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_100_149.jsonl


In [73]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_100_149.jsonl"
)

combined = {}

for path in [master_path, batch_path]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                combined[record["index"]] = record

merged_records = [
    combined[i] for i in sorted(combined)
]

# Save to a new version, preserving the old master
new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v2.jsonl"
)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Master dataset updated")
print("Total records:", len(merged_records))
print("Unique indices:", len(combined))
print("Excluded indices:", sorted(set(range(150)) - set(combined)))
print("Saved:", new_master_path)

✅ Master dataset updated
Total records: 146
Unique indices: 146
Excluded indices: [6, 12, 15, 42]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v2.jsonl


## Generate the next 50 teacher labels

In [74]:
generate_teacher_batch(150, 200)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 48
Invalid: 2
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


In [75]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 150 <= r["index"] < 200 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 2

INDEX: 161
CATALOG: Top
DESCRIPTION: Bruce shirt. Shirt in a soft cotton piqué with a collar, buttons down the front and yoke at the back. Open chest pocket and long sleeves with buttoned cuffs.

RAW TEACHER RESPONSE:
{
  "category": "shirt",
  "color": null,
  "material": "cotton piqué",
  "fit": null,
  "pattern": null,
  "sleeve_length": "long",
  "neckline": "buttoned"
}
ERROR: None

INDEX: 198
CATALOG: Outdoor Waistcoat
DESCRIPTION: BB Ville parka. Padded parka in woven fabric with a pile-lined, detachable hood, stand-up collar and a zip and wind flap with press-studs down the front. Zipped handwarmer pockets, flap front pockets with concealed press-studs, inner ribbing at the cuffs and imitation leather details. Partly lined in pile. Polyester filling.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "polyester",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "stand-up collar"
}
ERROR: None


## Save the fourth batch without mismatches

In [76]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

excluded_indices = {161, 198}

cleaned_batch_4 = []

for record in records:
    idx = record["index"]

    if not (150 <= idx < 200):
        continue

    if idx in excluded_indices:
        continue

    if not record["valid"] or record["attributes"] is None:
        continue

    updated = {
        **record,
        "attributes": validate_attributes(
            record["attributes"],
            record["input"]
        )
    }

    cleaned_batch_4.append(updated)

cleaned_batch_4.sort(key=lambda r: r["index"])

output_path = os.path.join(
    base_dir,
    "train_teacher_labels_cleaned_150_199.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_4:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Fourth batch saved")
print("Records retained:", len(cleaned_batch_4))
print("Excluded indices:", sorted(excluded_indices))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_4)))
print("Output:", output_path)

✅ Fourth batch saved
Records retained: 48
Excluded indices: [161, 198]
Unique indices: 48
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_150_199.jsonl


## Update the master dataset

In [77]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v2.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_150_199.jsonl"
)

combined = {}

for path in [master_path, batch_path]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                combined[record["index"]] = record

merged_records = [
    combined[i] for i in sorted(combined)
]

new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v3.jsonl"
)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Master dataset updated")
print("Total records:", len(merged_records))
print("Unique indices:", len(combined))
print("Excluded indices:", sorted(set(range(200)) - set(combined)))
print("Saved:", new_master_path)

✅ Master dataset updated
Total records: 194
Unique indices: 194
Excluded indices: [6, 12, 15, 42, 161, 198]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v3.jsonl


## Generate labels for records 200–249

In [78]:
generate_teacher_batch(200, 250)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 45
Invalid: 5
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


## Inspect the five invalid records

In [79]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 200 <= r["index"] < 250 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 5

INDEX: 200
CATALOG: Tailored Waistcoat
DESCRIPTION: EMMA hood FUR. Faux fur gilet with a hood with sewn-on ears at the top and a zip with a chin guard at the front. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "faux fur",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
ERROR: None

INDEX: 203
CATALOG: Outdoor Waistcoat
DESCRIPTION: Freja fur vest. Faux fur gilet with a V-neck, hook-and-eye fastening at the front and discreet pockets in the side seams. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "faux fur",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "v-neck"
}
ERROR: None

INDEX: 207
CATALOG: Tailored Waistcoat
DESCRIPTION: Chelsea welldressed waistcoat. Twill waistcoat with buttons down the front, a chest pocket and front pockets. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "materia

## Recover and save the fifth batch

In [80]:
import json
import os

recover_indices = {203, 207, 221}
exclude_indices = {200, 228}

cleaned_batch_5 = []

for record in records:
    idx = record["index"]

    if not (200 <= idx < 250):
        continue

    if idx in exclude_indices:
        continue

    updated = record.copy()

    if idx in recover_indices:
        attributes = json.loads(record["raw_teacher_response"])

        attributes["category"] = record["catalog_category"].lower()

        updated["attributes"] = attributes
        updated["valid"] = True
        updated["recovered_from_alias"] = True

    if not updated["valid"] or updated["attributes"] is None:
        continue

    updated["attributes"] = validate_attributes(
        updated["attributes"],
        updated["input"]
    )

    cleaned_batch_5.append(updated)

cleaned_batch_5.sort(key=lambda r: r["index"])

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_200_249.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_5:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Fifth batch saved")
print("Records retained:", len(cleaned_batch_5))
print("Recovered aliases:", len(recover_indices))
print("Excluded indices:", sorted(exclude_indices))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_5)))
print("Output:", output_path)

✅ Fifth batch saved
Records retained: 48
Recovered aliases: 3
Excluded indices: [200, 228]
Unique indices: 48
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_200_249.jsonl


## Update the master dataset

In [81]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v3.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_200_249.jsonl"
)

combined = {}

for path in [master_path, batch_path]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                combined[record["index"]] = record

merged_records = [
    combined[i] for i in sorted(combined)
]

new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v4.jsonl"
)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("✅ Master dataset updated")
print("Total records:", len(merged_records))
print("Unique indices:", len(combined))
print("Excluded indices:", sorted(set(range(250)) - set(combined)))
print("Saved:", new_master_path)

✅ Master dataset updated
Total records: 242
Unique indices: 242
Excluded indices: [6, 12, 15, 42, 161, 198, 200, 228]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v4.jsonl


## Generate labels for records 250–299

In [82]:
generate_teacher_batch(250, 300)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 45
Invalid: 5
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


## Inspect the five invalid records

In [83]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 250 <= r["index"] < 300 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 5

INDEX: 253
CATALOG: Dress
DESCRIPTION: Elaine shirt dress. Shirt dress in an airy viscose weave with a grandad collar, buttons down the front and long sleeves with a tab and button. Seam under the bust and a gently flared skirt. Unlined.

RAW TEACHER RESPONSE:
{
  "category": "shirt dress",
  "color": null,
  "material": "viscose",
  "fit": null,
  "pattern": null,
  "sleeve_length": "long",
  "neckline": "grandad collar"
}
ERROR: None

INDEX: 262
CATALOG: Outdoor Waistcoat
DESCRIPTION: LUSH vest padded. Padded gilet with a detachable hood, stand-up collar, zip down the front with a chin guard, and side pockets. Slightly longer and rounded at the back.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": null,
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "stand-up collar"
}
ERROR: None

INDEX: 263
CATALOG: Top
DESCRIPTION: Caramba fitted shirt. Straight-cut shirt in airy, stretch viscose with a collar, darts 

## Recover and correct the sixth batch

In [84]:
import json
import os

recover_indices = {253, 262, 290, 294}
exclude_indices = {263}

cleaned_batch_6 = []

for record in records:
    idx = record["index"]

    if not (250 <= idx < 300):
        continue

    if idx in exclude_indices:
        continue

    updated = record.copy()

    if idx in recover_indices:
        attributes = json.loads(
            record["raw_teacher_response"]
        )

        # Preserve catalog category
        attributes["category"] = (
            record["catalog_category"].lower()
        )

        # Correct unsupported attributes
        if idx == 290:
            attributes["neckline"] = None

        if idx == 294:
            attributes["pattern"] = None

        updated["attributes"] = attributes
        updated["valid"] = True
        updated["recovered_from_review"] = True

    if not updated["valid"] or updated["attributes"] is None:
        continue

    updated["attributes"] = validate_attributes(
        updated["attributes"],
        updated["input"]
    )

    cleaned_batch_6.append(updated)

cleaned_batch_6.sort(key=lambda r: r["index"])

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_250_299.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_6:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("✅ Sixth batch saved")
print("Records retained:", len(cleaned_batch_6))
print("Recovered:", len(recover_indices))
print("Excluded indices:", sorted(exclude_indices))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_6)))
print("Output:", output_path)

✅ Sixth batch saved
Records retained: 49
Recovered: 4
Excluded indices: [263]
Unique indices: 49
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_250_299.jsonl


## Update the master training dataset

In [85]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v4.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_250_299.jsonl"
)

combined = {}

for path in [master_path, batch_path]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                combined[record["index"]] = record

merged_records = [
    combined[i] for i in sorted(combined)
]

new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v5.jsonl"
)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Master dataset updated")
print("Total records:", len(merged_records))
print("Unique indices:", len(combined))
print("Excluded indices:", sorted(set(range(300)) - set(combined)))
print("Saved:", new_master_path)

✅ Master dataset updated
Total records: 291
Unique indices: 291
Excluded indices: [6, 12, 15, 42, 161, 198, 200, 228, 263]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v5.jsonl


## Generate labels for records 300–349

In [86]:
generate_teacher_batch(300, 350)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 46
Invalid: 4
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


In [87]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 300 <= r["index"] < 350 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 4

INDEX: 306
CATALOG: Outdoor Waistcoat
DESCRIPTION: LUSH VEST. Padded gilet with a detachable faux-fur trimmed hood, stand-up collar, zip with a chin guard at the front and side pockets. Slightly longer and rounded at the back.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": null,
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "stand-up collar"
}
ERROR: None

INDEX: 312
CATALOG: Leggings/Tights
DESCRIPTION: DIV Bella Biker leggings. Jersey leggings with an elasticated waist and sewn-in pin-tucks on the legs.

RAW TEACHER RESPONSE:
{
  "category": "leggings",
  "color": null,
  "material": "jersey",
  "fit": "elasticated",
  "pattern": null,
  "sleeve_length": null,
  "neckline": null
}
ERROR: None

INDEX: 333
CATALOG: Top
DESCRIPTION: Sibling 1 SG L/S T-shirt. Long-sleeved top in soft cotton jersey with a text print motif on the front.

RAW TEACHER RESPONSE:
{"category": "t-shirt", "color": null, "material"

## Recover and save the seventh batch

In [88]:
import json

recover_indices = {306, 312, 333}
exclude_indices = {341}

cleaned_batch_7 = []

for record in records:
    idx = record["index"]

    if not (300 <= idx < 350):
        continue

    if idx in exclude_indices:
        continue

    updated = record.copy()

    if idx in recover_indices:
        attributes = json.loads(
            record["raw_teacher_response"]
        )

        attributes["category"] = (
            record["catalog_category"].lower()
        )

        if idx == 312:
            attributes["fit"] = None

        updated["attributes"] = attributes
        updated["valid"] = True
        updated["recovered_from_review"] = True

    if not updated["valid"] or updated["attributes"] is None:
        continue

    updated["attributes"] = validate_attributes(
        updated["attributes"],
        updated["input"]
    )

    cleaned_batch_7.append(updated)

cleaned_batch_7.sort(key=lambda r: r["index"])

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_300_349.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_7:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Seventh batch saved")
print("Records retained:", len(cleaned_batch_7))
print("Recovered:", len(recover_indices))
print("Excluded indices:", sorted(exclude_indices))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_7)))
print("Output:", output_path)

✅ Seventh batch saved
Records retained: 49
Recovered: 3
Excluded indices: [341]
Unique indices: 49
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_300_349.jsonl


## Update the master training dataset

In [89]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v5.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_300_349.jsonl"
)

combined = {}

for path in [master_path, batch_path]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                combined[record["index"]] = record

merged_records = [
    combined[i] for i in sorted(combined)
]

new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v6.jsonl"
)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Master dataset updated")
print("Total records:", len(merged_records))
print("Unique indices:", len(combined))
print("Excluded indices:", sorted(set(range(350)) - set(combined)))
print("Saved:", new_master_path)

✅ Master dataset updated
Total records: 340
Unique indices: 340
Excluded indices: [6, 12, 15, 42, 161, 198, 200, 228, 263, 341]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v6.jsonl


## Generate labels for records 350–399

In [90]:
generate_teacher_batch(350, 400)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 47
Invalid: 3
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


## Inspect the three invalid records

In [91]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 350 <= r["index"] < 400 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 3

INDEX: 353
CATALOG: Top
DESCRIPTION: Tiramisu lace tee. Lace top with short sleeves and an opening with a concealed button at the back of the neck. Partly lined in jersey.

RAW TEACHER RESPONSE:
{"category": "tee", "color": null, "material": "lace", "fit": null, "pattern": null, "sleeve_length": "short", "neckline": "round neck"}
ERROR: None

INDEX: 359
CATALOG: Outdoor Waistcoat
DESCRIPTION: 58-STUDIO POET KNITTED VNECK V. STUDIO COLLECTION. V-neck sweater vest in a textured wool knit with a ribbed hem and contrasting colour ribbing around the neckline and armholes.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "wool",
  "fit": null,
  "pattern": null,
  "sleeve_length": "long",
  "neckline": "v-neck"
}
ERROR: None

INDEX: 389
CATALOG: Skirt
DESCRIPTION: Kaija trousers. Trousers in twill made from a wool blend with a regular waist, pleats at the top, concealed zip in one side and side pockets. Wide, gently tapered legs with crea

## Recover and save the eighth batch

In [92]:
import json

recover_indices = {353, 359}
exclude_indices = {389}

cleaned_batch_8 = []

for record in records:
    idx = record["index"]

    if not (350 <= idx < 400):
        continue

    if idx in exclude_indices:
        continue

    updated = record.copy()

    if idx in recover_indices:
        attributes = json.loads(
            record["raw_teacher_response"]
        )

        attributes["category"] = (
            record["catalog_category"].lower()
        )

        if idx == 353:
            attributes["neckline"] = None

        if idx == 359:
            attributes["sleeve_length"] = None

        updated["attributes"] = attributes
        updated["valid"] = True
        updated["recovered_from_review"] = True

    if not updated["valid"] or updated["attributes"] is None:
        continue

    updated["attributes"] = validate_attributes(
        updated["attributes"],
        updated["input"]
    )

    cleaned_batch_8.append(updated)

cleaned_batch_8.sort(key=lambda r: r["index"])

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_350_399.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_8:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Eighth batch saved")
print("Records retained:", len(cleaned_batch_8))
print("Recovered:", len(recover_indices))
print("Excluded indices:", sorted(exclude_indices))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_8)))
print("Output:", output_path)

✅ Eighth batch saved
Records retained: 49
Recovered: 2
Excluded indices: [389]
Unique indices: 49
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_350_399.jsonl


## Update the master training dataset

In [93]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v6.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_350_399.jsonl"
)

combined = {}

for path in [master_path, batch_path]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                combined[record["index"]] = record

merged_records = [
    combined[i] for i in sorted(combined)
]

new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v7.jsonl"
)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Master dataset updated")
print("Total records:", len(merged_records))
print("Unique indices:", len(combined))
print("Excluded indices:", sorted(set(range(400)) - set(combined)))
print("Saved:", new_master_path)

✅ Master dataset updated
Total records: 389
Unique indices: 389
Excluded indices: [6, 12, 15, 42, 161, 198, 200, 228, 263, 341, 389]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v7.jsonl


## Save the teacher configuration


In [94]:
import os
import json
import inspect

base_dir = "/content/drive/MyDrive/fashion-distillation"
config_dir = os.path.join(base_dir, "config")
os.makedirs(config_dir, exist_ok=True)

config = {
    "teacher_model": "Qwen/Qwen2.5-7B-Instruct",
    "student_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "system_prompt": system_prompt,
    "expected_keys": sorted(expected_keys),
    "training_target": 500,
    "last_processed_index": 399,
    "latest_master": "train_teacher_labels_master_v7.jsonl",
    "raw_labels": "train_teacher_labels.jsonl",
    "validation_source": inspect.getsource(validate_attributes)
}

config_path = os.path.join(
    config_dir, "teacher_config.json"
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✅ Configuration saved:", config_path)
print("Prompt characters:", len(system_prompt))
print("Last processed index:", config["last_processed_index"])

✅ Configuration saved: /content/drive/MyDrive/fashion-distillation/config/teacher_config.json
Prompt characters: 4453
Last processed index: 399


## Verify recovery checkpoints

In [95]:
import os
import json

base_dir = "/content/drive/MyDrive/fashion-distillation"

required_files = [
    "config/teacher_config.json",
    "garment_dataset_clean_v2.jsonl",
    "splits/train.jsonl",
    "splits/validation.jsonl",
    "splits/test.jsonl",
    "train_teacher_labels.jsonl",
    "train_teacher_labels_master_v7.jsonl"
]

print("=== RESUME CHECKPOINT VERIFICATION ===")

all_present = True

for filename in required_files:
    path = os.path.join(base_dir, filename)
    exists = os.path.isfile(path)

    print(f"{'✅' if exists else '❌'} {filename}")

    if not exists:
        all_present = False

# Check saved labeling progress
raw_path = os.path.join(base_dir, "train_teacher_labels.jsonl")

if os.path.isfile(raw_path):
    with open(raw_path, encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]

    indices = {r["index"] for r in records}

    print("\nProcessed indices:", len(indices))
    print("Completed 0–399:", all(i in indices for i in range(400)))
    print("Next batch:", "400–449")

print("\nAll checkpoint files present:", all_present)

=== RESUME CHECKPOINT VERIFICATION ===
✅ config/teacher_config.json
✅ garment_dataset_clean_v2.jsonl
✅ splits/train.jsonl
✅ splits/validation.jsonl
✅ splits/test.jsonl
✅ train_teacher_labels.jsonl
✅ train_teacher_labels_master_v7.jsonl

Processed indices: 400
Completed 0–399: True
Next batch: 400–449

All checkpoint files present: True


## Save validation functions for recovery

In [96]:
import os

config_dir = (
    "/content/drive/MyDrive/fashion-distillation/config"
)

history = In

# Collect cells that define validation functions
validation_cells = [
    cell for cell in history
    if "def validate_attributes(" in cell
]

print("Validation function definitions found:", len(validation_cells))

if len(validation_cells) >= 3:
    source = "\n\n".join(validation_cells)

    output_path = os.path.join(
        config_dir,
        "validation_functions.py"
    )

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(source)

    print("✅ Validation functions saved")
    print("Path:", output_path)
else:
    print("⚠️ Expected validation definitions not found.")
    print("Do not disconnect yet.")

Validation function definitions found: 4
✅ Validation functions saved
Path: /content/drive/MyDrive/fashion-distillation/config/validation_functions.py


## Test the saved validation functions

In [97]:
import os
import json
import ast

source_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "config/validation_functions.py"
)

with open(source_path, encoding="utf-8") as f:
    source = f.read()

try:
    # Confirm the saved file contains valid Python syntax
    ast.parse(source)
    print("✅ Python syntax valid")

    # Load into an independent namespace
    namespace = {
        "json": json,
        "re": __import__("re")
    }

    exec(compile(source, source_path, "exec"), namespace)

    restored_validator = namespace["validate_attributes"]

    # Compare against the current function
    sample_attributes = {
        "category": "top",
        "color": None,
        "material": "cotton jersey",
        "fit": None,
        "pattern": "text print",
        "sleeve_length": "long",
        "neckline": None
    }

    sample_text = (
        "Long-sleeved top in soft cotton jersey "
        "with a text print motif on the front."
    )

    original = validate_attributes(
        sample_attributes.copy(), sample_text
    )

    restored = restored_validator(
        sample_attributes.copy(), sample_text
    )

    assert original == restored

    print("✅ Validation functions restored successfully")
    print("✅ Original and restored outputs match")
    print("Result:", restored)

except Exception as e:
    print("❌ Recovery test failed")
    print("Error:", type(e).__name__, str(e))
    print("Do not disconnect yet.")

✅ Python syntax valid
✅ Attribute validation rules initialized.
✅ Validation function updated.
polo neck → polo neck
round neck → round neck
v-neck → v-neck
ribbed trim → None
❌ Recovery test failed
Error: NameError name 'In' is not defined
Do not disconnect yet.


## Clean the validation recovery script

In [100]:
import os
import cloudpickle

config_dir = "/content/drive/MyDrive/fashion-distillation/config"
os.makedirs(config_dir, exist_ok=True)

validator_path = os.path.join(
    config_dir,
    "validate_attributes.pkl"
)

# Save the live, working validator and its dependencies
with open(validator_path, "wb") as f:
    cloudpickle.dump(validate_attributes, f)

# Restore independently
with open(validator_path, "rb") as f:
    restored_validator = cloudpickle.load(f)

# Compare several examples
test_cases = [
    (
        {
            "category": "top",
            "color": None,
            "material": "cotton jersey",
            "fit": None,
            "pattern": "text print",
            "sleeve_length": "long",
            "neckline": None
        },
        "Long-sleeved top in cotton jersey with a text print."
    ),
    (
        {
            "category": "top",
            "color": None,
            "material": "lace",
            "fit": None,
            "pattern": None,
            "sleeve_length": "short",
            "neckline": "round neck"
        },
        "Lace top with short sleeves and a button at the back."
    ),
    (
        {
            "category": "outdoor waistcoat",
            "color": None,
            "material": "wool",
            "fit": None,
            "pattern": None,
            "sleeve_length": "long",
            "neckline": "v-neck"
        },
        "V-neck sweater vest in textured wool knit."
    )
]

for i, (attributes, description) in enumerate(test_cases, 1):
    original = validate_attributes(attributes.copy(), description)
    restored = restored_validator(attributes.copy(), description)

    assert original == restored, f"Mismatch in test {i}"

print("✅ Validator serialized successfully")
print("✅ All", len(test_cases), "recovery tests passed")
print("File:", validator_path)
print("File size:", os.path.getsize(validator_path), "bytes")

✅ Validator serialized successfully
✅ All 3 recovery tests passed
File: /content/drive/MyDrive/fashion-distillation/config/validate_attributes.pkl
File size: 4535 bytes


## Verify the saved validator against real training records

In [101]:
import os
import json
import cloudpickle
import copy

base_dir = "/content/drive/MyDrive/fashion-distillation"

with open(
    os.path.join(base_dir, "config/validate_attributes.pkl"),
    "rb"
) as f:
    restored_validator = cloudpickle.load(f)

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v7.jsonl"
)

with open(master_path, encoding="utf-8") as f:
    master_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

matches = 0
mismatches = []
errors = []

for record in master_records:
    index = record["index"]
    attributes = record["attributes"]
    description = record["input"]

    try:
        original = validate_attributes(
            copy.deepcopy(attributes), description
        )
        restored = restored_validator(
            copy.deepcopy(attributes), description
        )

        if original == restored:
            matches += 1
        else:
            mismatches.append(index)

    except Exception as e:
        errors.append((index, str(e)))

print("=== VALIDATOR RECOVERY VERIFICATION ===")
print("Records tested:", len(master_records))
print("Matching outputs:", matches)
print("Mismatches:", mismatches)
print("Errors:", errors[:5])

if matches == 389 and not mismatches and not errors:
    print("✅ Validator recovery verified on all 389 records")
else:
    print("❌ Verification incomplete. Do not disconnect yet.")

=== VALIDATOR RECOVERY VERIFICATION ===
Records tested: 389
Matching outputs: 389
Mismatches: []
Errors: []
✅ Validator recovery verified on all 389 records


## Save the teacher-generation function

In [102]:
import ast
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"
output_path = os.path.join(
    base_dir, "config/generate_teacher_batch.py"
)

# Find the most recent original definition in Colab history
function_sources = []

for cell in In:
    try:
        tree = ast.parse(cell)
    except SyntaxError:
        continue

    for node in tree.body:
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == "generate_teacher_batch"
        ):
            function_sources.append(
                ast.get_source_segment(cell, node)
            )

if not function_sources:
    raise RuntimeError(
        "Teacher-generation function not found. Do not disconnect."
    )

source = function_sources[-1]

# Compile without running generation
compile(source, output_path, "exec")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(source + "\n")

# Verify the saved file
with open(output_path, encoding="utf-8") as f:
    saved_source = f.read()

ast.parse(saved_source)

print("✅ Teacher-generation function saved")
print("Definitions found:", len(function_sources))
print("Source characters:", len(saved_source))
print("✅ Saved Python syntax valid")
print("Path:", output_path)

✅ Teacher-generation function saved
Definitions found: 1
Source characters: 3518
✅ Saved Python syntax valid
Path: /content/drive/MyDrive/fashion-distillation/config/generate_teacher_batch.py


## Identify generation-function dependencies

In [103]:
import ast
import os

source_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "config/generate_teacher_batch.py"
)

with open(source_path, encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)
function = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name == "generate_teacher_batch"
)

# Variables referenced inside the function
loaded_names = {
    node.id for node in ast.walk(function)
    if isinstance(node, ast.Name)
    and isinstance(node.ctx, ast.Load)
}

# Names defined locally inside the function
local_names = {
    node.id for node in ast.walk(function)
    if isinstance(node, ast.Name)
    and isinstance(node.ctx, (ast.Store, ast.Del))
}

local_names.update(arg.arg for arg in function.args.args)

import builtins

dependencies = sorted(
    loaded_names
    - local_names
    - set(dir(builtins))
)

print("=== GENERATION FUNCTION DEPENDENCIES ===")

for name in dependencies:
    status = "✅ Available" if name in globals() else "❌ Missing"
    print(f"{status}: {name}")

print("\nTotal dependencies:", len(dependencies))

=== GENERATION FUNCTION DEPENDENCIES ===
❌ Missing: e
✅ Available: expected_keys
✅ Available: json
✅ Available: os
✅ Available: system_prompt
✅ Available: teacher_model
✅ Available: teacher_tokenizer
✅ Available: torch
✅ Available: tqdm
✅ Available: train_dataset
✅ Available: validate_attributes

Total dependencies: 11


## Save the Colab environment versions

In [104]:
import sys
import json
import os
import importlib.metadata as metadata

packages = [
    "torch",
    "transformers",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "peft",
    "trl",
    "cloudpickle",
    "huggingface_hub"
]

versions = {}

for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = None

environment = {
    "python_version": sys.version,
    "packages": versions,
    "teacher_model": "Qwen/Qwen2.5-7B-Instruct",
    "student_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "next_batch_start": 400,
    "next_batch_end": 450,
    "master_dataset": "train_teacher_labels_master_v7.jsonl"
}

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "config/environment.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)

print("✅ Environment configuration saved")
print("Python:", sys.version.split()[0])

for package, version in versions.items():
    print(f"{package}: {version}")

print("\nSaved:", output_path)

✅ Environment configuration saved
Python: 3.13.16
torch: 2.11.0+cu130
transformers: 5.20.2
accelerate: 1.15.0
bitsandbytes: 0.50.2
datasets: 5.0.1
peft: 0.21.2
trl: 1.16.0
cloudpickle: 3.1.2
huggingface_hub: 1.33.0

Saved: /content/drive/MyDrive/fashion-distillation/config/environment.json


## Verify the saved configuration and resume readiness

In [105]:
import os
import json

base_dir = "/content/drive/MyDrive/fashion-distillation"

# Restore saved configuration
with open(
    os.path.join(base_dir, "config/teacher_config.json"),
    encoding="utf-8"
) as f:
    saved_config = json.load(f)

# Restore training split
train_path = os.path.join(base_dir, "splits/train.jsonl")

with open(train_path, encoding="utf-8") as f:
    restored_train = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

# Compare with current session
prompt_matches = saved_config["system_prompt"] == system_prompt
keys_match = set(saved_config["expected_keys"]) == set(expected_keys)
length_matches = len(restored_train) == len(train_dataset)

# Verify the first 400 records correspond to the same inputs
raw_path = os.path.join(base_dir, "train_teacher_labels.jsonl")

with open(raw_path, encoding="utf-8") as f:
    raw_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

indexed_records = {
    record["index"]: record
    for record in raw_records
}

inputs_match = all(
    indexed_records[i]["input"] == restored_train[i]["input"]
    for i in range(400)
)

print("=== RESUME READINESS ===")
print("Prompt matches:", prompt_matches)
print("Extraction keys match:", keys_match)
print("Training split length matches:", length_matches)
print("First 400 training inputs match:", inputs_match)
print("Restored training records:", len(restored_train))
print("Next index:", 400)

if all([
    prompt_matches,
    keys_match,
    length_matches,
    inputs_match
]):
    print("\n✅ Dataset and configuration restoration verified")
else:
    print("\n❌ Recovery mismatch — do not disconnect yet")

=== RESUME READINESS ===
Prompt matches: True
Extraction keys match: True
Training split length matches: True
First 400 training inputs match: True
Restored training records: 1137
Next index: 400

✅ Dataset and configuration restoration verified


## Test generation-function restoration without running the model

In [106]:
import ast
import os
import inspect

base_dir = "/content/drive/MyDrive/fashion-distillation"

source_path = os.path.join(
    base_dir, "config/generate_teacher_batch.py"
)

with open(source_path, encoding="utf-8") as f:
    source = f.read()

# Restore function without generating labels
namespace = {
    "os": os,
    "json": json,
    "torch": torch,
    "tqdm": tqdm,
    "teacher_model": teacher_model,
    "teacher_tokenizer": teacher_tokenizer,
    "train_dataset": restored_train,
    "system_prompt": saved_config["system_prompt"],
    "expected_keys": set(saved_config["expected_keys"]),
    "validate_attributes": restored_validator
}

exec(compile(source, source_path, "exec"), namespace)

restored_generate = namespace["generate_teacher_batch"]

# Check required generation and checkpointing logic
required_checks = {
    "Callable": callable(restored_generate),
    "Two range arguments": len(inspect.signature(restored_generate).parameters) == 2,
    "Reads existing labels": "train_teacher_labels.jsonl" in source,
    "Uses saved validator": "validate_attributes" in source,
    "Uses teacher model": "teacher_model.generate" in source,
}

print("=== GENERATION RESTORATION ===")

for name, passed in required_checks.items():
    print(f"{'✅' if passed else '❌'} {name}: {passed}")

assert all(required_checks.values()), "Recovery check failed"

print("\n✅ Generation function restored without inference")
print("✅ Ready to resume from index 400 after reconnecting")

=== GENERATION RESTORATION ===
✅ Callable: True
✅ Two range arguments: True
✅ Reads existing labels: True
✅ Uses saved validator: True
✅ Uses teacher model: True

✅ Generation function restored without inference
✅ Ready to resume from index 400 after reconnecting


## temp : resume script

In [107]:
import os
import json

base_dir = "/content/drive/MyDrive/fashion-distillation"
resume_path = os.path.join(base_dir, "config/resume_session.py")

resume_source = '''
import os
import json
import cloudpickle
import torch
from tqdm.auto import tqdm

base_dir = "/content/drive/MyDrive/fashion-distillation"

# Restore teacher configuration
with open(
    os.path.join(base_dir, "config/teacher_config.json"),
    encoding="utf-8"
) as f:
    config = json.load(f)

system_prompt = config["system_prompt"]
expected_keys = set(config["expected_keys"])

# Restore training dataset
with open(
    os.path.join(base_dir, "splits/train.jsonl"),
    encoding="utf-8"
) as f:
    train_dataset = [
        json.loads(line)
        for line in f if line.strip()
    ]

# Restore validation function
with open(
    os.path.join(base_dir, "config/validate_attributes.pkl"),
    "rb"
) as f:
    validate_attributes = cloudpickle.load(f)

# Restore generation function
source_path = os.path.join(
    base_dir, "config/generate_teacher_batch.py"
)

with open(source_path, encoding="utf-8") as f:
    source = f.read()

exec(compile(source, source_path, "exec"))

# Verify checkpoints
raw_path = os.path.join(
    base_dir, "train_teacher_labels.jsonl"
)

with open(raw_path, encoding="utf-8") as f:
    completed = {
        json.loads(line)["index"]
        for line in f if line.strip()
    }

assert all(i in completed for i in range(400))
assert len(train_dataset) == 1137

print("✅ Fashion distillation session restored")
print("Training records:", len(train_dataset))
print("Previously processed:", len(completed))
print("Next batch: 400–449")
print("Teacher model must be loaded before generation.")
'''

with open(resume_path, "w", encoding="utf-8") as f:
    f.write(resume_source)

compile(resume_source, resume_path, "exec")

print("✅ Resume script saved")
print("Path:", resume_path)
print("✅ Python syntax verified")

✅ Resume script saved
Path: /content/drive/MyDrive/fashion-distillation/config/resume_session.py
✅ Python syntax verified


## Final resume-script verification

In [108]:
import os
import json

base_dir = "/content/drive/MyDrive/fashion-distillation"

resume_path = os.path.join(
    base_dir, "config/resume_session.py"
)

with open(resume_path, encoding="utf-8") as f:
    source = f.read()

# Simulate a fresh session with a separate namespace
namespace = {
    "__name__": "__main__"
}

exec(compile(source, resume_path, "exec"), namespace)

# Verify restored components
assert len(namespace["train_dataset"]) == 1137
assert callable(namespace["validate_attributes"])
assert callable(namespace["generate_teacher_batch"])
assert len(namespace["completed"]) == 400

# Ensure the master checkpoint remains intact
master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v7.jsonl"
)

with open(master_path, encoding="utf-8") as f:
    master_records = [
        json.loads(line)
        for line in f if line.strip()
    ]

assert len(master_records) == 389

print("\n✅ FINAL CHECKPOINT VERIFIED")
print("Raw teacher records:", len(namespace["completed"]))
print("Accepted master records:", len(master_records))
print("Training examples:", len(namespace["train_dataset"]))
print("Next batch: 400–449")
print("✅ Ready for safe Colab disconnect")

✅ Fashion distillation session restored
Training records: 1137
Previously processed: 400
Next batch: 400–449
Teacher model must be loaded before generation.

✅ FINAL CHECKPOINT VERIFIED
Raw teacher records: 400
Accepted master records: 389
Training examples: 1137
Next batch: 400–449
✅ Ready for safe Colab disconnect


##resume: Restore Python dependencies

In [1]:
from google.colab import drive
import os
import torch

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/fashion-distillation"

assert os.path.isdir(base_dir), "Fashion distillation folder missing"

print("✅ Google Drive mounted")
print("✅ Project folder found")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

Mounted at /content/drive
✅ Google Drive mounted
✅ Project folder found
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
import json
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError

base_dir = "/content/drive/MyDrive/fashion-distillation"

with open(
    os.path.join(base_dir, "config/environment.json"),
    encoding="utf-8"
) as f:
    environment = json.load(f)

packages = [
    "transformers",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "peft",
    "trl",
    "cloudpickle",
    "huggingface_hub"
]

to_install = []

for package in packages:
    expected = environment["packages"][package]

    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None

    if installed != expected:
        to_install.append(f"{package}=={expected}")

    print(f"{package}: installed={installed}, expected={expected}")

if to_install:
    print("\nInstalling missing or mismatched packages...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", *to_install
    ])
else:
    print("\n✅ All required package versions already match")

print("\n✅ Dependency setup completed")
print("Restart runtime if Colab requests it.")

transformers: installed=5.20.2, expected=5.20.2
accelerate: installed=1.15.0, expected=1.15.0
bitsandbytes: installed=None, expected=0.50.2
datasets: installed=4.8.5, expected=5.0.1
peft: installed=0.21.2, expected=0.21.2
trl: installed=None, expected=1.16.0
cloudpickle: installed=3.1.2, expected=3.1.2
huggingface_hub: installed=1.33.0, expected=1.33.0

Installing missing or mismatched packages...

✅ Dependency setup completed
Restart runtime if Colab requests it.


## Restore the dataset, prompt, and validation functions

In [3]:
import os
import json

base_dir = "/content/drive/MyDrive/fashion-distillation"

resume_path = os.path.join(
    base_dir,
    "config/resume_session.py"
)

# Restore variables and functions into the notebook
with open(resume_path, encoding="utf-8") as f:
    resume_source = f.read()

exec(compile(resume_source, resume_path, "exec"), globals())

# Verify restored state
assert len(train_dataset) == 1137
assert len(completed) == 400
assert callable(validate_attributes)
assert callable(generate_teacher_batch)

print("\n=== RECOVERY VERIFICATION ===")
print("Training records:", len(train_dataset))
print("Teacher records completed:", len(completed))
print("Next batch:", "400–449")
print("✅ Dataset, prompt, and functions restored")

✅ Fashion distillation session restored
Training records: 1137
Previously processed: 400
Next batch: 400–449
Teacher model must be loaded before generation.

=== RECOVERY VERIFICATION ===
Training records: 1137
Teacher records completed: 400
Next batch: 400–449
✅ Dataset, prompt, and functions restored


## Reload Qwen2.5-7B Teacher on the T4 GPU

In [4]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

teacher_model_id = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

teacher_tokenizer = AutoTokenizer.from_pretrained(
    teacher_model_id
)

teacher_model = AutoModelForCausalLM.from_pretrained(
    teacher_model_id,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

teacher_model.eval()

print("✅ Teacher model loaded")
print("Model:", teacher_model_id)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print("✅ Ready to continue teacher labeling")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Teacher model loaded
Model: Qwen/Qwen2.5-7B-Instruct
GPU: Tesla T4
GPU memory allocated: 5.18 GB
✅ Ready to continue teacher labeling


## Verify teacher inference after recovery

In [5]:
import json
import torch

record = train_dataset[0]

messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user",
        "content": (
            f"CATALOG_CATEGORY: {record['category']}\n"
            f"DESCRIPTION: {record['input']}"
        )
    }
]

inputs = teacher_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(teacher_model.device)

with torch.inference_mode():
    outputs = teacher_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=teacher_tokenizer.eos_token_id
    )

response = teacher_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

print("INDEX: 0")
print("CATALOG:", record["category"])
print("DESCRIPTION:", record["input"])
print("\nTEACHER RESPONSE:")
print(response)

INDEX: 0
CATALOG: Garment Set
DESCRIPTION: TVP Alvin set. Set with a short-sleeved shirt and pair of shorts in woven fabric. Shirt with a collar and buttons down the front. Shorts with an adjustable elasticated waist, fake fly with a button, fake pockets, and detachable, adjustable elastic braces with imitation leather details.

TEACHER RESPONSE:
{
  "category": "garment set",
  "color": null,
  "material": "woven",
  "fit": null,
  "pattern": null,
  "sleeve_length": "short",
  "neckline": null
}


In [6]:
generate_teacher_batch(400, 450)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 47
Invalid: 3
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


In [7]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 400 <= r["index"] < 450 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 3

INDEX: 405
CATALOG: Outdoor Waistcoat
DESCRIPTION: NORA vest. Padded, gently fitted gilet in woven fabric with a stand-up collar and detachable, lined hood with a faux fur trim. Zip down the front with a chin guard, side pockets with a press-stud, and smocking at the sides. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "woven fabric",
  "fit": "gently fitted",
  "pattern": null,
  "sleeve_length": null,
  "neckline": "stand-up collar"
}
ERROR: None

INDEX: 408
CATALOG: Outdoor Waistcoat
DESCRIPTION: ROX Vest. Lightweight outdoor gilet with padded sections front and back and reflective details. Stand-up collar, zip down the front with a chin guard and a small zipped pocket at the back. Partly lined in fleece.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "fleece",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "stand-up collar"
}
ERROR: None

INDEX: 429
CATALOG:

In [8]:
import json

recover_indices = {405, 408}
exclude_indices = {429}

cleaned_batch_9 = []

for record in records:
    idx = record["index"]

    if not (400 <= idx < 450):
        continue

    if idx in exclude_indices:
        continue

    updated = record.copy()

    if idx in recover_indices:
        attributes = json.loads(
            record["raw_teacher_response"]
        )

        attributes["category"] = (
            record["catalog_category"].lower()
        )

        if idx == 408:
            attributes["material"] = None

        updated["attributes"] = attributes
        updated["valid"] = True
        updated["recovered_from_review"] = True

    if not updated["valid"] or updated["attributes"] is None:
        continue

    updated["attributes"] = validate_attributes(
        updated["attributes"],
        updated["input"]
    )

    cleaned_batch_9.append(updated)

cleaned_batch_9.sort(key=lambda r: r["index"])

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_400_449.jsonl"
)

assert len(cleaned_batch_9) == 49
assert len({r["index"] for r in cleaned_batch_9}) == 49

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_9:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Ninth batch saved")
print("Records retained:", len(cleaned_batch_9))
print("Recovered:", len(recover_indices))
print("Excluded indices:", sorted(exclude_indices))
print("Unique indices:", len(set(r["index"] for r in cleaned_batch_9)))
print("Output:", output_path)

✅ Ninth batch saved
Records retained: 49
Recovered: 2
Excluded indices: [429]
Unique indices: 49
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_400_449.jsonl


In [9]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v7.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_400_449.jsonl"
)

new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v8.jsonl"
)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

master_records = load_jsonl(master_path)
batch_records = load_jsonl(batch_path)

master_indices = {r["index"] for r in master_records}
batch_indices = {r["index"] for r in batch_records}

# Prevent accidental duplication or overwriting
assert len(master_records) == 389
assert len(batch_records) == 49
assert not (master_indices & batch_indices), "Overlapping indices detected!"

merged_records = sorted(
    master_records + batch_records,
    key=lambda r: r["index"]
)

assert len(merged_records) == 438
assert all(r["valid"] for r in merged_records)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

excluded = sorted(
    set(range(450)) -
    {r["index"] for r in merged_records}
)

print("✅ Master dataset updated")
print("Total records:", len(merged_records))
print("Unique indices:", len({r["index"] for r in merged_records}))
print("Excluded indices:", excluded)
print("Saved:", new_master_path)

✅ Master dataset updated
Total records: 438
Unique indices: 438
Excluded indices: [6, 12, 15, 42, 161, 198, 200, 228, 263, 341, 389, 429]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v8.jsonl


## Generate the final teacher-labeling batch

In [10]:
generate_teacher_batch(450, 500)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Batch complete
New records: 50
Valid: 41
Invalid: 9
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels.jsonl


In [11]:
import json

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels.jsonl"
)

with open(path, encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

invalid = [
    r for r in records
    if 450 <= r["index"] < 500 and not r["valid"]
]

print("Invalid records:", len(invalid))

for record in invalid:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("CATALOG:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])
    print("\nRAW TEACHER RESPONSE:")
    print(record["raw_teacher_response"])
    print("ERROR:", record["error"])

Invalid records: 9

INDEX: 456
CATALOG: Outdoor Waistcoat
DESCRIPTION: Lush Vest. Padded gilet with a detachable faux-fur trimmed hood lined with soft pile. Stand-up collar, side pockets and a zip down the front with a chin guard. Slightly longer and rounded at the back.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": null,
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "stand-up collar"
}
ERROR: None

INDEX: 460
CATALOG: Outdoor Waistcoat
DESCRIPTION: Freja Fur Vest. V-neck gilet in soft faux fur with a concealed hook-and-eye fastening at the top. Lined.

RAW TEACHER RESPONSE:
{
  "category": "waistcoat",
  "color": null,
  "material": "faux fur",
  "fit": null,
  "pattern": null,
  "sleeve_length": null,
  "neckline": "v-neck"
}
ERROR: None

INDEX: 465
CATALOG: Tailored Waistcoat
DESCRIPTION: Rockford S/S shirt set. Short-sleeved shirt with a bow tie or tie and a waistcoat in a cotton weave. Shirt with a collar and buttons 

In [12]:
import json
import copy

recover_indices = {456, 460, 466, 473, 480, 483}
exclude_indices = {465, 487, 488}

batch_records = {
    r["index"]: r
    for r in records
    if 450 <= r["index"] < 500
}

assert len(batch_records) == 50
assert all(not batch_records[i]["valid"] for i in recover_indices | exclude_indices)

cleaned_batch_10 = []

for idx in sorted(batch_records):
    if idx in exclude_indices:
        continue

    updated = copy.deepcopy(batch_records[idx])

    if idx in recover_indices:
        attributes = json.loads(
            updated["raw_teacher_response"]
        )

        attributes["category"] = (
            updated["catalog_category"].lower()
        )

        if idx == 466:
            attributes["color"] = None

        if idx == 480:
            attributes["sleeve_length"] = None

        updated["attributes"] = attributes
        updated["valid"] = True
        updated["recovered_from_review"] = True

    if not updated["valid"] or updated["attributes"] is None:
        continue

    updated["attributes"] = validate_attributes(
        updated["attributes"],
        updated["input"]
    )

    cleaned_batch_10.append(updated)

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_cleaned_450_499.jsonl"
)

assert len(cleaned_batch_10) == 47
assert len({r["index"] for r in cleaned_batch_10}) == 47

with open(output_path, "w", encoding="utf-8") as f:
    for record in cleaned_batch_10:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Tenth batch saved")
print("Records retained:", len(cleaned_batch_10))
print("Recovered:", len(recover_indices))
print("Excluded indices:", sorted(exclude_indices))
print("Unique indices:", len({r["index"] for r in cleaned_batch_10}))
print("Output:", output_path)

✅ Tenth batch saved
Records retained: 47
Recovered: 6
Excluded indices: [465, 487, 488]
Unique indices: 47
Output: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_cleaned_450_499.jsonl


## Merge the final batch into the master dataset

In [13]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v8.jsonl"
)

batch_path = os.path.join(
    base_dir, "train_teacher_labels_cleaned_450_499.jsonl"
)

new_master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v9.jsonl"
)

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

master_records = load_jsonl(master_path)
batch_records = load_jsonl(batch_path)

master_indices = [r["index"] for r in master_records]
batch_indices = [r["index"] for r in batch_records]

assert len(master_records) == 438
assert len(batch_records) == 47
assert len(set(master_indices)) == len(master_indices)
assert len(set(batch_indices)) == len(batch_indices)
assert not (set(master_indices) & set(batch_indices))

merged_records = sorted(
    master_records + batch_records,
    key=lambda r: r["index"]
)

assert len(merged_records) == 485
assert all(r["valid"] for r in merged_records)
assert all(
    isinstance(r["attributes"], dict)
    and set(r["attributes"]) == expected_keys
    for r in merged_records
)

with open(new_master_path, "w", encoding="utf-8") as f:
    for record in merged_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

excluded = sorted(
    set(range(500)) -
    {r["index"] for r in merged_records}
)

print("✅ Final master dataset saved")
print("Total accepted:", len(merged_records))
print("Unique indices:", len(set(r["index"] for r in merged_records)))
print("Total excluded:", len(excluded))
print("Excluded indices:", excluded)
print("Saved:", new_master_path)

✅ Final master dataset saved
Total accepted: 485
Unique indices: 485
Total excluded: 15
Excluded indices: [6, 12, 15, 42, 161, 198, 200, 228, 263, 341, 389, 429, 465, 487, 488]
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v9.jsonl


## Audit training-label quality

In [14]:
import json
from collections import Counter

master_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_master_v9.jsonl"
)

with open(master_path, encoding="utf-8") as f:
    training_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

attribute_names = [
    "category",
    "color",
    "material",
    "fit",
    "pattern",
    "sleeve_length",
    "neckline"
]

print("=== DATASET QUALITY AUDIT ===")
print("Total records:", len(training_records))

print("\nATTRIBUTE COVERAGE")
print(f"{'Attribute':<18} {'Non-null':>9} {'Null':>7} {'Coverage':>10}")

for attribute in attribute_names:
    values = [
        r["attributes"].get(attribute)
        for r in training_records
    ]

    non_null = sum(
        v is not None and str(v).strip() != ""
        for v in values
    )

    null_count = len(values) - non_null
    coverage = non_null / len(values) * 100

    print(
        f"{attribute:<18} "
        f"{non_null:>9} "
        f"{null_count:>7} "
        f"{coverage:>9.1f}%"
    )

print("\nTOP GARMENT CATEGORIES")
categories = Counter(
    r["attributes"]["category"]
    for r in training_records
)

for category, count in categories.most_common(25):
    print(f"{category:<25} {count}")

print("\nTOP MATERIAL VALUES")
materials = Counter(
    r["attributes"]["material"]
    for r in training_records
    if r["attributes"]["material"] is not None
)

for material, count in materials.most_common(15):
    print(f"{material:<25} {count}")

=== DATASET QUALITY AUDIT ===
Total records: 485

ATTRIBUTE COVERAGE
Attribute           Non-null    Null   Coverage
category                 485       0     100.0%
color                     22     463       4.5%
material                 471      14      97.1%
fit                      115     370      23.7%
pattern                   62     423      12.8%
sleeve_length            232     253      47.8%
neckline                 201     284      41.4%

TOP GARMENT CATEGORIES
dress                     32
trousers                  31
coat                      30
polo shirt                30
skirt                     27
top                       27
vest top                  25
blazer                    24
jacket                    24
t-shirt                   24
shorts                    23
blouse                    23
cardigan                  23
garment set               21
hoodie                    21
shirt                     20
leggings/tights           20
sweater                   19
o

## Detect suspicious material labels

In [15]:
from collections import Counter

construction_terms = {
    "woven",
    "woven fabric",
    "knit",
    "jersey",
    "sweatshirt fabric",
    "functional fabric",
    "twill",
    "ribbed",
    "piqué",
    "crepe weave",
    "crêpe weave"
}

suspicious = []

for record in training_records:
    material = record["attributes"].get("material")

    if isinstance(material, str):
        if material.lower().strip() in construction_terms:
            suspicious.append(record)

print("=== MATERIAL QUALITY INSPECTION ===")
print("Flagged records:", len(suspicious))

counts = Counter(
    r["attributes"]["material"]
    for r in suspicious
)

print("\nFlagged material values:")
for material, count in counts.most_common():
    print(f"{material:<25} {count}")

print("\nFIRST 12 EXAMPLES")

for record in suspicious[:12]:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("MATERIAL:", record["attributes"]["material"])
    print("DESCRIPTION:", record["input"])

=== MATERIAL QUALITY INSPECTION ===
Flagged records: 99

Flagged material values:
jersey                    37
woven                     22
sweatshirt fabric         13
knit                      9
functional fabric         8
woven fabric              7
twill                     3

FIRST 12 EXAMPLES

INDEX: 0
MATERIAL: woven
DESCRIPTION: TVP Alvin set. Set with a short-sleeved shirt and pair of shorts in woven fabric. Shirt with a collar and buttons down the front. Shorts with an adjustable elasticated waist, fake fly with a button, fake pockets, and detachable, adjustable elastic braces with imitation leather details.

INDEX: 1
MATERIAL: woven
DESCRIPTION: Daniels checked emb shirt. Shirt in woven fabric with a collar, buttons down the front and embroidery at the top and back. Dropped shoulders and long sleeves with buttoned cuffs. Short slits in the sides. Slightly longer at the back.

INDEX: 5
MATERIAL: functional fabric
DESCRIPTION: SULIMA jacket. Outdoor jacket with lightly padded 

In [16]:
import re
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())

unsupported_materials = []

for record in training_records:
    material = record["attributes"].get("material")

    if not isinstance(material, str) or not material.strip():
        continue

    normalized_material = normalize_text(material)
    normalized_description = normalize_text(record["input"])

    if normalized_material not in normalized_description:
        unsupported_materials.append(record)

print("=== MATERIAL GROUNDING AUDIT ===")
print("Total records:", len(training_records))
print("Potentially unsupported:", len(unsupported_materials))

print("\nFIRST 20 FLAGGED EXAMPLES")

for record in unsupported_materials[:20]:
    print("\n" + "=" * 55)
    print("INDEX:", record["index"])
    print("MATERIAL:", record["attributes"]["material"])
    print("DESCRIPTION:", record["input"])

=== MATERIAL GROUNDING AUDIT ===
Total records: 485
Potentially unsupported: 19

FIRST 20 FLAGGED EXAMPLES

INDEX: 37
MATERIAL: glittery jersey, organza, satin, taffeta
DESCRIPTION: MERMAID dance dress. Fancy dress costume in glittery jersey with a shimmering metallic print. Narrow shoulder straps with organza flounces, a glittery appliqué at the top, a flounced, V-shaped seam at the waist and a flared skirt in shimmering organza. Detachable, padded tail in patterned satin with a hook and loop fastening. The skirt is lined with taffeta.

INDEX: 128
MATERIAL: twill, recycled polyester
DESCRIPTION: pro carcoat price item new. Car coat in twill with a collar, concealed buttons down the front, diagonal welt front pockets and inner pockets. Long raglan sleeves with an adjustable tab at the cuffs, and a single back vent. Lined in satin made from recycled polyester.

INDEX: 134
MATERIAL: organic cotton, jersey, sweatshirt fabric
DESCRIPTION: TOBBE set. Set with a long-sleeved bodysuit and pai

## Apply five targeted material corrections

In [17]:
import json
import copy
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

corrections = {
    128: "twill",
    173: None,
    230: "lace",
    355: "functional fabric",
    472: "airy weave"
}

corrected_records = copy.deepcopy(training_records)

for record in corrected_records:
    idx = record["index"]

    if idx in corrections:
        previous = record["attributes"]["material"]
        updated = corrections[idx]

        record["attributes"]["material"] = updated

        print(f"INDEX {idx}: {previous!r} → {updated!r}")

assert len(corrected_records) == 485

corrected_indices = {
    r["index"]
    for r in corrected_records
    if r["index"] in corrections
}

assert corrected_indices == set(corrections)

output_path = os.path.join(
    base_dir,
    "train_teacher_labels_master_v10.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in corrected_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\n✅ Material corrections saved")
print("Records:", len(corrected_records))
print("Corrections:", len(corrections))
print("Saved:", output_path)

INDEX 128: 'twill, recycled polyester' → 'twill'
INDEX 173: 'cotton' → None
INDEX 230: 'lace, jersey' → 'lace'
INDEX 355: 'fleece, wind- and waterproof functional fabric' → 'functional fabric'
INDEX 472: 'air weave' → 'airy weave'

✅ Material corrections saved
Records: 485
Corrections: 5
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v10.jsonl


## Final attribute-quality audit

In [18]:
import json
import re
import unicodedata
from collections import Counter

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "train_teacher_labels_master_v10.jsonl"
)

with open(path, encoding="utf-8") as f:
    audited_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

def normalize_for_audit(value):
    value = unicodedata.normalize("NFKC", str(value)).lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())

attributes_to_check = [
    "color",
    "fit",
    "pattern",
    "sleeve_length",
    "neckline"
]

flagged = []

for record in audited_records:
    description = normalize_for_audit(record["input"])

    for attribute in attributes_to_check:
        value = record["attributes"].get(attribute)

        if not isinstance(value, str) or not value.strip():
            continue

        if normalize_for_audit(value) not in description:
            flagged.append({
                "index": record["index"],
                "attribute": attribute,
                "value": value,
                "description": record["input"]
            })

print("=== FINAL ATTRIBUTE QUALITY AUDIT ===")
print("Total records:", len(audited_records))
print("Total flagged attribute values:", len(flagged))

print("\nFLAGS BY ATTRIBUTE")
for attribute, count in Counter(
    item["attribute"] for item in flagged
).most_common():
    print(f"{attribute:<18} {count}")

print("\nFIRST 20 FLAGGED VALUES")

for item in flagged[:20]:
    print("\n" + "=" * 55)
    print("INDEX:", item["index"])
    print("ATTRIBUTE:", item["attribute"])
    print("VALUE:", item["value"])
    print("DESCRIPTION:", item["description"])

=== FINAL ATTRIBUTE QUALITY AUDIT ===
Total records: 485
Total flagged attribute values: 22

FLAGS BY ATTRIBUTE
sleeve_length      13
pattern            8
fit                1

FIRST 20 FLAGGED VALUES

INDEX: 37
ATTRIBUTE: pattern
VALUE: shimmering metallic print, flounced, V-shaped seam, patterned
DESCRIPTION: MERMAID dance dress. Fancy dress costume in glittery jersey with a shimmering metallic print. Narrow shoulder straps with organza flounces, a glittery appliqué at the top, a flounced, V-shaped seam at the waist and a flared skirt in shimmering organza. Detachable, padded tail in patterned satin with a hook and loop fastening. The skirt is lined with taffeta.

INDEX: 73
ATTRIBUTE: sleeve_length
VALUE: long
DESCRIPTION: BABY knitted cardigan. Baby Exclusive. Cardigan in soft, rib-knit organic cotton with buttons down the front and sewn-in turn-ups at the cuffs.

INDEX: 117
ATTRIBUTE: sleeve_length
VALUE: short
DESCRIPTION: ARK DRESS. Calf-length kaftan dress in soft cotton jersey 

## Inspect the two remaining flagged values

In [19]:
print("=== REMAINING FLAGGED VALUES ===")

for item in flagged[20:]:
    print("\n" + "=" * 55)
    print("INDEX:", item["index"])
    print("ATTRIBUTE:", item["attribute"])
    print("VALUE:", item["value"])
    print("DESCRIPTION:", item["description"])

=== REMAINING FLAGGED VALUES ===

INDEX: 412
ATTRIBUTE: pattern
VALUE: printed
DESCRIPTION: Audrey knot. Short-sleeved top in slub jersey with a print motif, cap sleeves and a tie at the hem.

INDEX: 454
ATTRIBUTE: pattern
VALUE: striped
DESCRIPTION: Valetta Blouse. Blouse in a patterned weave with a small stand-up collar, sewn-in ties at the top, buttons down the front and a yoke with a pleat at the back. Long sleeves with buttoned cuffs, and a rounded hem.


In [20]:
import json
import copy
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

# Explicit corrections from our audit
corrections = {
    37:  {"pattern": "metallic print"},
    73:  {"sleeve_length": None},
    190: {"sleeve_length": None},
    211: {"sleeve_length": None},
    213: {"sleeve_length": None},
    241: {"sleeve_length": None},
    255: {"sleeve_length": None},
    260: {"fit": None},
    292: {"sleeve_length": None},
    297: {"pattern": "patterned"},
    298: {"sleeve_length": None},
    302: {"sleeve_length": None},
    303: {"sleeve_length": None},
    305: {"sleeve_length": None},
    364: {"sleeve_length": None},
    378: {"pattern": "patterned"},
    454: {"pattern": "patterned"}
}

corrected_records = copy.deepcopy(audited_records)

applied = set()

for record in corrected_records:
    idx = record["index"]

    if idx not in corrections:
        continue

    for attribute, new_value in corrections[idx].items():
        old_value = record["attributes"][attribute]
        record["attributes"][attribute] = new_value

        print(
            f"INDEX {idx} | {attribute}: "
            f"{old_value!r} → {new_value!r}"
        )

    applied.add(idx)

assert applied == set(corrections)
assert len(corrected_records) == 485
assert len({r["index"] for r in corrected_records}) == 485

output_path = os.path.join(
    base_dir,
    "train_teacher_labels_master_v11.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in corrected_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("\n✅ Final attribute corrections saved")
print("Records retained:", len(corrected_records))
print("Records corrected:", len(applied))
print("Saved:", output_path)

INDEX 37 | pattern: 'shimmering metallic print, flounced, V-shaped seam, patterned' → 'metallic print'
INDEX 73 | sleeve_length: 'long' → None
INDEX 190 | sleeve_length: 'short' → None
INDEX 211 | sleeve_length: 'long' → None
INDEX 213 | sleeve_length: 'short' → None
INDEX 241 | sleeve_length: 'long' → None
INDEX 255 | sleeve_length: 'long' → None
INDEX 260 | fit: 'slim' → None
INDEX 292 | sleeve_length: 'long' → None
INDEX 297 | pattern: 'striped' → 'patterned'
INDEX 298 | sleeve_length: 'long' → None
INDEX 302 | sleeve_length: 'long' → None
INDEX 303 | sleeve_length: 'long' → None
INDEX 305 | sleeve_length: 'short' → None
INDEX 364 | sleeve_length: 'long' → None
INDEX 378 | pattern: 'striped' → 'patterned'
INDEX 454 | pattern: 'striped' → 'patterned'

✅ Final attribute corrections saved
Records retained: 485
Records corrected: 17
Saved: /content/drive/MyDrive/fashion-distillation/train_teacher_labels_master_v11.jsonl


## Prepare the student-training dataset

In [21]:
import json
import os
import random

base_dir = "/content/drive/MyDrive/fashion-distillation"

master_path = os.path.join(
    base_dir, "train_teacher_labels_master_v11.jsonl"
)

with open(master_path, encoding="utf-8") as f:
    clean_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(clean_records) == 485
assert len({r["index"] for r in clean_records}) == 485

examples = []

for record in clean_records:
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": (
                f"CATALOG_CATEGORY: {record['catalog_category']}\n"
                f"DESCRIPTION: {record['input']}"
            )
        },
        {
            "role": "assistant",
            "content": json.dumps(
                record["attributes"],
                ensure_ascii=False
            )
        }
    ]

    examples.append({
        "index": record["index"],
        "messages": messages
    })

random.Random(42).shuffle(examples)

# 90% student training, 10% internal validation
split_point = int(len(examples) * 0.9)

student_train = examples[:split_point]
student_val = examples[split_point:]

output_dir = os.path.join(base_dir, "student")
os.makedirs(output_dir, exist_ok=True)

for filename, data in [
    ("train.jsonl", student_train),
    ("validation.jsonl", student_val)
]:
    path = os.path.join(output_dir, filename)

    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("✅ Student dataset prepared")
print("Training examples:", len(student_train))
print("Internal validation examples:", len(student_val))
print("Output directory:", output_dir)

print("\nSAMPLE ASSISTANT TARGET:")
print(student_train[0]["messages"][-1]["content"])

✅ Student dataset prepared
Training examples: 436
Internal validation examples: 49
Output directory: /content/drive/MyDrive/fashion-distillation/student

SAMPLE ASSISTANT TARGET:
{"category": "shorts", "color": null, "material": "sweatshirt fabric", "fit": null, "pattern": null, "sleeve_length": null, "neckline": null}


In [22]:
import json
import os
from collections import Counter

base_dir = "/content/drive/MyDrive/fashion-distillation"
student_dir = os.path.join(base_dir, "student")

def read_jsonl(filename):
    with open(
        os.path.join(student_dir, filename),
        encoding="utf-8"
    ) as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

train_examples = read_jsonl("train.jsonl")
val_examples = read_jsonl("validation.jsonl")

required_keys = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

def check_example(example):
    messages = example["messages"]

    assert len(messages) == 3
    assert [m["role"] for m in messages] == [
        "system", "user", "assistant"
    ]

    assert messages[0]["content"] == system_prompt

    target = json.loads(messages[2]["content"])

    assert set(target.keys()) == required_keys
    assert all(
        value is None or isinstance(value, str)
        for value in target.values()
    )

    return target

for example in train_examples + val_examples:
    check_example(example)

train_indices = {x["index"] for x in train_examples}
val_indices = {x["index"] for x in val_examples}

assert len(train_indices) == len(train_examples)
assert len(val_indices) == len(val_examples)
assert train_indices.isdisjoint(val_indices)
assert len(train_examples) == 436
assert len(val_examples) == 49

print("✅ Student dataset validation passed")
print("Training records:", len(train_examples))
print("Internal validation records:", len(val_examples))
print("Overlapping indices:", len(train_indices & val_indices))
print("JSON schema:", sorted(required_keys))
print("✅ Ready for student baseline evaluation")

✅ Student dataset validation passed
Training records: 436
Internal validation records: 49
Overlapping indices: 0
JSON schema: ['category', 'color', 'fit', 'material', 'neckline', 'pattern', 'sleeve_length']
✅ Ready for student baseline evaluation


## Load the original Qwen2.5-0.5B student model

In [23]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Release teacher GPU memory
if "teacher_model" in globals():
    del teacher_model

if "teacher_tokenizer" in globals():
    del teacher_tokenizer

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory after teacher cleanup:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

# Load original 0.5B student model
student_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

student_tokenizer = AutoTokenizer.from_pretrained(
    student_model_id
)

student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

student_model.eval()

print("\n✅ Base student model loaded")
print("Model:", student_model_id)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU memory after teacher cleanup: 0.01 GB


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ Base student model loaded
Model: Qwen/Qwen2.5-0.5B-Instruct
GPU: Tesla T4
GPU memory allocated: 0.93 GB


## Test baseline fashion attribute extraction

In [24]:
import torch
import json
import time

example = val_examples[0]

messages = example["messages"][:2]

inputs = student_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(student_model.device)

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    outputs = student_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=student_tokenizer.eos_token_id
    )

torch.cuda.synchronize()
latency = time.perf_counter() - start

response = student_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

print("=== BASE STUDENT PREDICTION ===")
print("INDEX:", example["index"])
print("PRODUCT:", messages[1]["content"])

print("\nBASE STUDENT OUTPUT:")
print(response)

print("\nREFERENCE LABEL:")
print(example["messages"][2]["content"])

try:
    parsed = json.loads(response)
    print("\n✅ Valid JSON:", True)
    print("Keys:", list(parsed.keys()))
except json.JSONDecodeError:
    print("\n❌ Valid JSON:", False)

print(f"\nInference latency: {latency:.2f} seconds")

=== BASE STUDENT PREDICTION ===
INDEX: 401
PRODUCT: CATALOG_CATEGORY: Polo shirt
DESCRIPTION: Fav regular polo. Polo shirt in cotton piqué with a ribbed collar, button placket, short sleeves with ribbed trims, and slits in the sides.

BASE STUDENT OUTPUT:
```json
{
  "category": "Garment",
  "color": ["blue", "green"],
  "material": ["cotton", "piké"],
  "fit": "regular",
  "pattern": "ribbed",
  "sleeve_length": "short",
  "neckline": "ribbed"
}
```

REFERENCE LABEL:
{"category": "polo shirt", "color": null, "material": "cotton piqué", "fit": null, "pattern": null, "sleeve_length": "short", "neckline": "ribbed collar"}

❌ Valid JSON: False

Inference latency: 3.43 seconds


## Run the baseline evaluation on all 49 validation examples

In [25]:
import json
import time
import os
import torch
from tqdm.auto import tqdm

base_dir = "/content/drive/MyDrive/fashion-distillation"
output_path = os.path.join(
    base_dir, "student/baseline_predictions.jsonl"
)

expected = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

# Resume safely if the evaluation is interrupted
existing = {}

if os.path.exists(output_path):
    with open(output_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                existing[record["index"]] = record

student_model.eval()

with open(output_path, "a", encoding="utf-8") as f:
    for example in tqdm(val_examples):
        idx = example["index"]

        if idx in existing:
            continue

        inputs = student_tokenizer.apply_chat_template(
            example["messages"][:2],
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(student_model.device)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = student_model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=student_tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        latency = time.perf_counter() - start

        response = student_tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()

        strict_json = True
        try:
            prediction = json.loads(response)
        except (json.JSONDecodeError, TypeError):
            strict_json = False
            cleaned = response.removeprefix(
                "```json"
            ).removeprefix(
                "```"
            ).removesuffix("```").strip()

            try:
                prediction = json.loads(cleaned)
            except (json.JSONDecodeError, TypeError):
                prediction = None

        valid_schema = (
            isinstance(prediction, dict)
            and set(prediction) == expected
            and all(
                v is None or isinstance(v, str)
                for v in prediction.values()
            )
        )

        reference = json.loads(
            example["messages"][2]["content"]
        )

        result = {
            "index": idx,
            "response": response,
            "prediction": prediction,
            "reference": reference,
            "strict_json": strict_json,
            "valid_schema": valid_schema,
            "latency_seconds": latency
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()

        existing[idx] = result

results = list(existing.values())
results = [
    r for r in results
    if r["index"] in {e["index"] for e in val_examples}
]

schema_valid = [r for r in results if r["valid_schema"]]

category_matches = sum(
    r["prediction"]["category"].lower()
    == r["reference"]["category"].lower()
    for r in schema_valid
)

attribute_matches = sum(
    r["prediction"][key] == r["reference"][key]
    for r in schema_valid
    for key in expected
)

print("\n=== BASELINE EVALUATION ===")
print("Examples evaluated:", len(results))
print(
    "Strict JSON validity:",
    f"{sum(r['strict_json'] for r in results)}/{len(results)}"
)
print(
    "Schema validity:",
    f"{len(schema_valid)}/{len(results)}"
)
print(
    "Category agreement (schema-valid only):",
    f"{category_matches}/{len(schema_valid)}"
)
print(
    "Exact attribute agreement (schema-valid only):",
    f"{attribute_matches}/{len(schema_valid) * 7}"
)
print(
    "Mean inference latency:",
    round(
        sum(r["latency_seconds"] for r in results) / len(results),
        2
    ),
    "seconds"
)
print("Saved:", output_path)

  0%|          | 0/49 [00:00<?, ?it/s]


=== BASELINE EVALUATION ===
Examples evaluated: 49
Strict JSON validity: 1/49
Schema validity: 20/49
Category agreement (schema-valid only): 11/20
Exact attribute agreement (schema-valid only): 37/140
Mean inference latency: 2.33 seconds
Saved: /content/drive/MyDrive/fashion-distillation/student/baseline_predictions.jsonl


## Configure LoRA for student fine-tuning

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

student_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

student_tokenizer = AutoTokenizer.from_pretrained(
    student_model_id
)

student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

student_model.eval()

print("✅ Student model restored")
print("Model:", student_model_id)
print("GPU available:", torch.cuda.is_available())
print("Model device:", student_model.device)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Student model restored
Model: Qwen/Qwen2.5-0.5B-Instruct
GPU available: True
Model device: cuda:0
GPU memory allocated: 0.92 GB


In [1]:
import torch

print("=== GPU VERIFICATION ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
    print("✅ Ready for LoRA fine-tuning")
else:
    print("❌ GPU unavailable — do not begin training")

=== GPU VERIFICATION ===
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
✅ Ready for LoRA fine-tuning


In [2]:
from google.colab import drive
import os
import json

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/fashion-distillation"

required_files = {
    "Cleaned teacher labels":
        "train_teacher_labels_master_v11.jsonl",
    "Student training data":
        "student/train.jsonl",
    "Student validation data":
        "student/validation.jsonl",
    "Baseline predictions":
        "student/baseline_predictions.jsonl",
    "Environment configuration":
        "config/environment.json"
}

print("\n=== TRAINING CHECKPOINT VERIFICATION ===")

for label, filename in required_files.items():
    path = os.path.join(base_dir, filename)

    assert os.path.isfile(path), f"Missing: {filename}"

    with open(path, encoding="utf-8") as f:
        if filename.endswith(".jsonl"):
            count = sum(1 for line in f if line.strip())
            print(f"✅ {label}: {count} records")
        else:
            print(f"✅ {label}: found")

print("\n✅ All training checkpoints verified")

Mounted at /content/drive

=== TRAINING CHECKPOINT VERIFICATION ===
✅ Cleaned teacher labels: 485 records
✅ Student training data: 436 records
✅ Student validation data: 49 records
✅ Baseline predictions: 49 records
✅ Environment configuration: found

✅ All training checkpoints verified


## Restore the training dependencies

In [3]:
import os
import sys
import json
import subprocess
from importlib.metadata import version, PackageNotFoundError

base_dir = "/content/drive/MyDrive/fashion-distillation"

with open(
    os.path.join(base_dir, "config/environment.json"),
    encoding="utf-8"
) as f:
    environment = json.load(f)

packages = [
    "transformers",
    "accelerate",
    "peft",
    "datasets",
    "trl",
    "cloudpickle",
    "huggingface_hub"
]

to_install = []

print("=== DEPENDENCY CHECK ===")

for package in packages:
    expected = environment["packages"][package]

    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None

    print(
        f"{package}: "
        f"installed={installed}, expected={expected}"
    )

    if installed != expected:
        to_install.append(f"{package}=={expected}")

if to_install:
    print("\nInstalling required versions...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *to_install
    ])
else:
    print("\n✅ All dependency versions already match")

print("\n✅ Training dependencies ready")

=== DEPENDENCY CHECK ===
transformers: installed=5.20.2, expected=5.20.2
accelerate: installed=1.15.0, expected=1.15.0
peft: installed=0.21.2, expected=0.21.2
datasets: installed=4.8.5, expected=5.0.1
trl: installed=None, expected=1.16.0
cloudpickle: installed=3.1.2, expected=3.1.2
huggingface_hub: installed=1.33.0, expected=1.33.0

Installing required versions...

✅ Training dependencies ready


## oad Qwen2.5-0.5B-Instruct on GPU

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "GPU unavailable. Do not continue."

student_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

student_tokenizer = AutoTokenizer.from_pretrained(
    student_model_id
)

student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

student_model.eval()

print("✅ Student model loaded")
print("Model:", student_model_id)
print("Device:", student_model.device)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

assert student_model.device.type == "cuda", (
    "Student model is not on GPU"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Student model loaded
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0
GPU: Tesla T4
GPU memory allocated: 0.92 GB


In [6]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "torchao"
])

print("✅ Incompatible torchao package removed")
print("⚠️ Restart the Colab runtime before retrying LoRA")

✅ Incompatible torchao package removed
⚠️ Restart the Colab runtime before retrying LoRA


## Attach the LoRA adapter

In [7]:
import torch
from peft import LoraConfig, get_peft_model, TaskType

assert student_model.device.type == "cuda", "GPU required"

# Reduce memory usage during training
student_model.config.use_cache = False
student_model.gradient_checkpointing_enable()
student_model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

student_model = get_peft_model(
    student_model,
    lora_config
)

student_model.print_trainable_parameters()

print("\n✅ LoRA adapter attached")
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

✅ LoRA adapter attached
GPU: Tesla T4
GPU memory allocated: 0.95 GB


In [2]:
from google.colab import drive
import os

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/fashion-distillation"

files = [
    "train_teacher_labels_master_v11.jsonl",
    "student/train.jsonl",
    "student/validation.jsonl",
    "student/baseline_predictions.jsonl"
]

print("\n=== DATASET FILE VERIFICATION ===")

for filename in files:
    path = os.path.join(base_dir, filename)

    if os.path.isfile(path):
        print(f"✅ Found: {filename}")
    else:
        print(f"❌ Missing: {filename}")

print("\nStudent model available:", "student_model" in globals())
print("Tokenizer available:", "student_tokenizer" in globals())

Mounted at /content/drive

=== DATASET FILE VERIFICATION ===
✅ Found: train_teacher_labels_master_v11.jsonl
✅ Found: student/train.jsonl
✅ Found: student/validation.jsonl
✅ Found: student/baseline_predictions.jsonl

Student model available: False
Tokenizer available: False


## Load and format the training dataset

In [5]:
import os
import json
from datasets import Dataset

base_dir = "/content/drive/MyDrive/fashion-distillation"

def load_student_split(filename):
    path = os.path.join(base_dir, "student", filename)

    with open(path, encoding="utf-8") as f:
        records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    return Dataset.from_list([
        {"messages": r["messages"]}
        for r in records
    ])

train_sft_dataset = load_student_split("train.jsonl")
eval_sft_dataset = load_student_split("validation.jsonl")

assert len(train_sft_dataset) == 436
assert len(eval_sft_dataset) == 49

# Check that the tokenizer supports assistant-token masking
template = student_tokenizer.chat_template

print("=== SFT DATASET ===")
print("Training examples:", len(train_sft_dataset))
print("Validation examples:", len(eval_sft_dataset))
print("Columns:", train_sft_dataset.column_names)

print("\nTokenizer assistant-mask support:")
print("Has generation markers:", "{% generation %}" in str(template))

print("\nSample conversation roles:")
print([m["role"] for m in train_sft_dataset[0]["messages"]])

print("\n✅ SFT datasets loaded")

=== SFT DATASET ===
Training examples: 436
Validation examples: 49
Columns: ['messages']

Tokenizer assistant-mask support:
Has generation markers: False

Sample conversation roles:
['system', 'user', 'assistant']

✅ SFT datasets loaded


In [8]:
example = train_sft_dataset[0]["messages"]

prompt_output = student_tokenizer.apply_chat_template(
    example[:2],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True
)

full_output = student_tokenizer.apply_chat_template(
    example,
    tokenize=True,
    add_generation_prompt=False,
    return_dict=True
)

prompt_ids = prompt_output["input_ids"]
full_ids = full_output["input_ids"]

# Handle tensors if returned
if hasattr(prompt_ids, "tolist"):
    prompt_ids = prompt_ids.tolist()

if hasattr(full_ids, "tolist"):
    full_ids = full_ids.tolist()

# Find the longest matching token prefix
common = 0

for a, b in zip(prompt_ids, full_ids):
    if a != b:
        break
    common += 1

print("=== CORRECTED TOKEN BOUNDARY DIAGNOSIS ===")
print("Prompt tokens:", len(prompt_ids))
print("Full conversation tokens:", len(full_ids))
print("Matching prefix tokens:", common)
print("Prompt is exact prefix:", full_ids[:len(prompt_ids)] == prompt_ids)

print("\nPROMPT END:")
print(repr(student_tokenizer.decode(prompt_ids[-25:])))

print("\nFULL CONVERSATION AT BOUNDARY:")
print(repr(student_tokenizer.decode(
    full_ids[max(0, common - 10):common + 25]
)))

print("\n✅ Token IDs extracted correctly")

=== CORRECTED TOKEN BOUNDARY DIAGNOSIS ===
Prompt tokens: 995
Full conversation tokens: 1043
Matching prefix tokens: 995
Prompt is exact prefix: True

PROMPT END:
'DESCRIPTION: Venice shorts. Shorts in lightweight sweatshirt fabric with an elasticated waist and side pockets.<|im_end|>\n<|im_start|>assistant\n'

FULL CONVERSATION AT BOUNDARY:
' waist and side pockets.<|im_end|>\n<|im_start|>assistant\n{"category": "shorts", "color": null, "material": "sweatshirt fabric", "fit":'

✅ Token IDs extracted correctly


## Prepare masked training and validation datasets

In [9]:
from datasets import Dataset

MAX_LENGTH = 1536

def tokenize_for_sft(example):
    messages = example["messages"]

    prompt_ids = student_tokenizer.apply_chat_template(
        messages[:2],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True
    )["input_ids"]

    full_ids = student_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True
    )["input_ids"]

    if hasattr(prompt_ids, "tolist"):
        prompt_ids = prompt_ids.tolist()
    if hasattr(full_ids, "tolist"):
        full_ids = full_ids.tolist()

    assert full_ids[:len(prompt_ids)] == prompt_ids, (
        "Prompt alignment mismatch"
    )

    # Avoid silently truncating JSON targets.
    if len(full_ids) > MAX_LENGTH:
        raise ValueError(
            f"Sequence length {len(full_ids)} exceeds {MAX_LENGTH}"
        )

    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    assert any(label != -100 for label in labels)

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }

tokenized_train = train_sft_dataset.map(
    tokenize_for_sft,
    remove_columns=train_sft_dataset.column_names
)

tokenized_val = eval_sft_dataset.map(
    tokenize_for_sft,
    remove_columns=eval_sft_dataset.column_names
)

print("✅ Assistant-only datasets prepared")
print("Training records:", len(tokenized_train))
print("Validation records:", len(tokenized_val))

sample = tokenized_train[0]

print("\nSAMPLE TOKEN STATISTICS")
print("Total tokens:", len(sample["input_ids"]))
print("Masked tokens:", sample["labels"].count(-100))
print(
    "Trainable tokens:",
    sum(label != -100 for label in sample["labels"])
)

print("\nTRAINABLE RESPONSE:")
print(student_tokenizer.decode([
    label for label in sample["labels"]
    if label != -100
]))

Map:   0%|          | 0/436 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

✅ Assistant-only datasets prepared
Training records: 436
Validation records: 49

SAMPLE TOKEN STATISTICS
Total tokens: 1043
Masked tokens: 995
Trainable tokens: 48

TRAINABLE RESPONSE:
{"category": "shorts", "color": null, "material": "sweatshirt fabric", "fit": null, "pattern": null, "sleeve_length": null, "neckline": null}<|im_end|>



## Configure the training data collator

In [10]:
import torch
from torch.nn.utils.rnn import pad_sequence

# Use EOS for padding, because Qwen tokenizers may not
# provide a dedicated padding token.
student_tokenizer.pad_token = student_tokenizer.eos_token
student_tokenizer.padding_side = "right"

def sft_data_collator(features):
    input_ids = [
        torch.tensor(f["input_ids"], dtype=torch.long)
        for f in features
    ]

    attention_masks = [
        torch.tensor(f["attention_mask"], dtype=torch.long)
        for f in features
    ]

    labels = [
        torch.tensor(f["labels"], dtype=torch.long)
        for f in features
    ]

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=student_tokenizer.pad_token_id
        ),
        "attention_mask": pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0
        ),
        "labels": pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100
        )
    }

# Test with two training examples
batch = sft_data_collator([
    tokenized_train[0],
    tokenized_train[1]
])

assert batch["input_ids"].shape == batch["labels"].shape
assert batch["attention_mask"].shape == batch["labels"].shape
assert torch.all(batch["labels"][batch["attention_mask"] == 0] == -100)

print("✅ Training data collator verified")
print("Batch shape:", tuple(batch["input_ids"].shape))
print("Masked tokens:", int((batch["labels"] == -100).sum()))
print("Supervised tokens:", int((batch["labels"] != -100).sum()))
print("Padding tokens:", int((batch["attention_mask"] == 0).sum()))
print("GPU:", torch.cuda.get_device_name(0))

✅ Training data collator verified
Batch shape: (2, 1131)
Masked tokens: 2162
Supervised tokens: 100
Padding tokens: 88
GPU: Tesla T4


## Configure LoRA fine-tuning

In [12]:
import os
import torch
from transformers import Trainer, TrainingArguments

base_dir = "/content/drive/MyDrive/fashion-distillation"
checkpoint_dir = os.path.join(
    base_dir, "student", "lora_checkpoints_v1"
)
os.makedirs(checkpoint_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42
)

trainer = Trainer(
    model=student_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=sft_data_collator
)

print("✅ LoRA trainer configured")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:", 2 * 4)
print("Warmup:", training_args.warmup_steps)
print("Checkpoint directory:", checkpoint_dir)
print("GPU:", torch.cuda.get_device_name(0))

✅ LoRA trainer configured
Training examples: 436
Validation examples: 49
Epochs: 3
Effective batch size: 8
Warmup: 0.05
Checkpoint directory: /content/drive/MyDrive/fashion-distillation/student/lora_checkpoints_v1
GPU: Tesla T4


## Diagnose the student model state

In [14]:
import torch

print("=== STUDENT MODEL DIAGNOSTICS ===")

print("Model class:", type(student_model).__name__)
print("Model device:", student_model.device)
print("GPU available:", torch.cuda.is_available())

print(
    "LoRA configuration present:",
    hasattr(student_model, "peft_config")
)

lora_modules = [
    name
    for name, module in student_model.named_modules()
    if hasattr(module, "lora_A")
]

print("LoRA modules found:", len(lora_modules))

total_params = sum(
    p.numel() for p in student_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in student_model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print(
    "Trainer uses current model:",
    trainer.model is student_model
)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

=== STUDENT MODEL DIAGNOSTICS ===
Model class: Qwen2ForCausalLM
Model device: cuda:0
GPU available: True
LoRA configuration present: False
LoRA modules found: 0
Total parameters: 494,032,768
Trainable parameters: 494,032,768
Trainer uses current model: True
GPU memory allocated: 1.84 GB


## Remove incompatible TorchAO

In [16]:
import sys
import subprocess
from importlib.metadata import (
    version,
    PackageNotFoundError
)

try:
    print("Installed torchao:", version("torchao"))
except PackageNotFoundError:
    print("torchao is already uninstalled")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "uninstall",
    "-y",
    "torchao"
])

try:
    print("Remaining torchao version:", version("torchao"))
    print("❌ Package still installed")
except PackageNotFoundError:
    print("✅ Incompatible torchao removed successfully")

print("\n⚠️ Restart the Colab session before retrying LoRA.")

Installed torchao: 0.10.0
✅ Incompatible torchao removed successfully

⚠️ Restart the Colab session before retrying LoRA.


## verify the environment

In [17]:
import torch
import importlib.util
from importlib.metadata import version, PackageNotFoundError

print("=== ENVIRONMENT VERIFICATION ===")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for package in ["torch", "transformers", "peft", "torchao"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: Not installed")

# TorchAO must not be installed
try:
    version("torchao")
    raise RuntimeError("Incompatible torchao is still installed")
except PackageNotFoundError:
    print("✅ TorchAO conflict resolved")

assert torch.cuda.is_available(), "T4 GPU unavailable"

print("✅ Environment ready for LoRA restoration")

=== ENVIRONMENT VERIFICATION ===
CUDA available: True
GPU: Tesla T4
torch: 2.11.0+cu130
transformers: 5.20.2
peft: 0.21.2
torchao: Not installed
✅ TorchAO conflict resolved
✅ Environment ready for LoRA restoration


## Restore the student model and attach LoRA

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

assert torch.cuda.is_available(), "GPU unavailable"

student_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# Restore tokenizer
student_tokenizer = AutoTokenizer.from_pretrained(
    student_model_id
)

student_tokenizer.pad_token = student_tokenizer.eos_token
student_tokenizer.padding_side = "right"

# Restore original student model
student_model = AutoModelForCausalLM.from_pretrained(
    student_model_id,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# Configure for fine-tuning
student_model.config.use_cache = False
student_model.gradient_checkpointing_enable()
student_model.enable_input_require_grads()

# Attach LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

student_model = get_peft_model(
    student_model,
    lora_config
)

# Keep trainable adapter weights in FP32
for param in student_model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

# Verify configuration
assert hasattr(student_model, "peft_config")

student_model.print_trainable_parameters()

print("\n✅ Student model restored")
print("✅ LoRA adapter attached")
print("Model class:", type(student_model).__name__)
print("Device:", student_model.device)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

✅ Student model restored
✅ LoRA adapter attached
Model class: PeftModelForCausalLM
Device: cuda:0
GPU: Tesla T4
GPU memory allocated: 2.8 GB


## Step 136F: Restore student datasets and assistant-only masking

In [19]:
import os
import json
from google.colab import drive
from datasets import Dataset

if not os.path.isfile(
    "/content/drive/MyDrive/fashion-distillation/student/train.jsonl"
):
    drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/fashion-distillation"

def load_split(filename):
    path = os.path.join(base_dir, "student", filename)

    with open(path, encoding="utf-8") as f:
        records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    return Dataset.from_list([
        {"messages": record["messages"]}
        for record in records
    ])

train_sft_dataset = load_split("train.jsonl")
eval_sft_dataset = load_split("validation.jsonl")

MAX_LENGTH = 1536

def tokenize_for_sft(example):
    messages = example["messages"]

    prompt_ids = student_tokenizer.apply_chat_template(
        messages[:2],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True
    )["input_ids"]

    full_ids = student_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True
    )["input_ids"]

    if hasattr(prompt_ids, "tolist"):
        prompt_ids = prompt_ids.tolist()

    if hasattr(full_ids, "tolist"):
        full_ids = full_ids.tolist()

    assert full_ids[:len(prompt_ids)] == prompt_ids
    assert len(full_ids) <= MAX_LENGTH

    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    assert any(label != -100 for label in labels)

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }

tokenized_train = train_sft_dataset.map(
    tokenize_for_sft,
    remove_columns=["messages"]
)

tokenized_val = eval_sft_dataset.map(
    tokenize_for_sft,
    remove_columns=["messages"]
)

assert len(tokenized_train) == 436
assert len(tokenized_val) == 49

sample = tokenized_train[0]

print("\n✅ Training datasets restored")
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))
print("Sample total tokens:", len(sample["input_ids"]))
print("Sample masked tokens:", sample["labels"].count(-100))
print(
    "Sample supervised tokens:",
    sum(x != -100 for x in sample["labels"])
)
print("✅ Assistant-only masking restored")

Map:   0%|          | 0/436 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]


✅ Training datasets restored
Training examples: 436
Validation examples: 49
Sample total tokens: 1043
Sample masked tokens: 995
Sample supervised tokens: 48
✅ Assistant-only masking restored


## Restore the data collator and trainer


In [20]:
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import Trainer, TrainingArguments

base_dir = "/content/drive/MyDrive/fashion-distillation"

checkpoint_dir = os.path.join(
    base_dir, "student", "lora_checkpoints_v1"
)

os.makedirs(checkpoint_dir, exist_ok=True)

def sft_data_collator(features):
    return {
        "input_ids": pad_sequence(
            [
                torch.tensor(f["input_ids"], dtype=torch.long)
                for f in features
            ],
            batch_first=True,
            padding_value=student_tokenizer.pad_token_id
        ),
        "attention_mask": pad_sequence(
            [
                torch.tensor(f["attention_mask"], dtype=torch.long)
                for f in features
            ],
            batch_first=True,
            padding_value=0
        ),
        "labels": pad_sequence(
            [
                torch.tensor(f["labels"], dtype=torch.long)
                for f in features
            ],
            batch_first=True,
            padding_value=-100
        )
    }

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_steps=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    fp16=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42
)

assert hasattr(student_model, "peft_config"), (
    "LoRA adapter is missing"
)

trainer = Trainer(
    model=student_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=sft_data_collator
)

# Verify correct model and masking
batch = sft_data_collator([
    tokenized_train[0],
    tokenized_train[1]
])

assert trainer.model is student_model
assert batch["input_ids"].shape == batch["labels"].shape
assert (batch["labels"] != -100).any()
assert (
    batch["labels"][batch["attention_mask"] == 0] == -100
).all()

print("✅ LoRA trainer restored")
print("Model:", type(trainer.model).__name__)
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:", 2 * 4)
print("Checkpoint directory:", checkpoint_dir)
print("✅ Assistant-only masking verified")
print("✅ Trainer references LoRA model")

✅ LoRA trainer restored
Model: PeftModelForCausalLM
Training examples: 436
Validation examples: 49
Epochs: 3
Effective batch size: 8
Checkpoint directory: /content/drive/MyDrive/fashion-distillation/student/lora_checkpoints_v1
✅ Assistant-only masking verified
✅ Trainer references LoRA model


In [21]:
import torch
import os

assert torch.cuda.is_available()
assert hasattr(trainer.model, "peft_config")
assert trainer.model is student_model
assert trainer.state.global_step == 0, (
    "Training has already started. Check checkpoints before restarting."
)

print("Starting fashion attribute distillation training")
print("Model:", type(trainer.model).__name__)
print("GPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Checkpoint directory:", training_args.output_dir)

train_result = trainer.train()

# Save final LoRA adapter independently of trainer checkpoints
adapter_dir = os.path.join(
    base_dir,
    "student",
    "fashion_lora_adapter_v1"
)

trainer.model.save_pretrained(adapter_dir)
student_tokenizer.save_pretrained(adapter_dir)

print("\n✅ Fine-tuning completed")
print("Final training loss:", train_result.training_loss)
print("Total optimizer steps:", trainer.state.global_step)
print("Final adapter saved:", adapter_dir)
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)

🚀 Starting fashion attribute distillation training
Model: PeftModelForCausalLM
GPU: Tesla T4
Training examples: 436
Validation examples: 49
Checkpoint directory: /content/drive/MyDrive/fashion-distillation/student/lora_checkpoints_v1


Epoch,Training Loss,Validation Loss
1,0.068140,0.054886
2,0.031932,0.047904
3,0.016677,0.048798



✅ Fine-tuning completed
Final training loss: 0.05424397646477728
Total optimizer steps: 165
Final adapter saved: /content/drive/MyDrive/fashion-distillation/student/fashion_lora_adapter_v1
Peak GPU memory: 6.94 GB


## Evaluate the fine-tuned model on one example

In [22]:
import json
import time
import torch

# Switch to inference mode
student_model.eval()
student_model.config.use_cache = True

# Find the same example used in the baseline
example = next(
    x for x in eval_sft_dataset
    if "Fav regular polo" in x["messages"][1]["content"]
)

messages = example["messages"][:2]

inputs = student_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(student_model.device)

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    outputs = student_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=student_tokenizer.eos_token_id
    )

torch.cuda.synchronize()
latency = time.perf_counter() - start

response = student_tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

reference = json.loads(example["messages"][2]["content"])

print("=== FINE-TUNED STUDENT PREDICTION ===")
print("PRODUCT:", messages[1]["content"])

print("\nFINE-TUNED OUTPUT:")
print(response)

print("\nREFERENCE LABEL:")
print(json.dumps(reference, indent=2))

try:
    prediction = json.loads(response)

    valid_schema = (
        isinstance(prediction, dict)
        and set(prediction) == {
            "category", "color", "material", "fit",
            "pattern", "sleeve_length", "neckline"
        }
        and all(
            v is None or isinstance(v, str)
            for v in prediction.values()
        )
    )

    print("\n✅ Valid JSON")
    print("Schema valid:", valid_schema)

except json.JSONDecodeError:
    print("\n❌ Invalid JSON")

print(f"Inference latency: {latency:.2f} seconds")

=== FINE-TUNED STUDENT PREDICTION ===
PRODUCT: CATALOG_CATEGORY: Polo shirt
DESCRIPTION: Fav regular polo. Polo shirt in cotton piqué with a ribbed collar, button placket, short sleeves with ribbed trims, and slits in the sides.

FINE-TUNED OUTPUT:
{"category": "polo shirt", "color": null, "material": "cotton piqué", "fit": null, "pattern": null, "sleeve_length": "short", "neckline": "ribbed collar"}

REFERENCE LABEL:
{
  "category": "polo shirt",
  "color": null,
  "material": "cotton piqu\u00e9",
  "fit": null,
  "pattern": null,
  "sleeve_length": "short",
  "neckline": "ribbed collar"
}

✅ Valid JSON
Schema valid: True
Inference latency: 6.09 seconds


## Evaluate the fine-tuned model on all 49 validation examples

In [23]:
import json
import os
import time
import torch
from tqdm.auto import tqdm

output_path = os.path.join(
    base_dir,
    "student",
    "finetuned_predictions_v1.jsonl"
)

with open(
    os.path.join(base_dir, "student", "validation.jsonl"),
    encoding="utf-8"
) as f:
    validation_examples = [
        json.loads(line)
        for line in f if line.strip()
    ]

expected_keys = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

existing = {}
if os.path.exists(output_path):
    with open(output_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                result = json.loads(line)
                existing[result["index"]] = result

student_model.eval()

with open(output_path, "a", encoding="utf-8") as f:
    for example in tqdm(validation_examples):
        idx = example["index"]

        if idx in existing:
            continue

        inputs = student_tokenizer.apply_chat_template(
            example["messages"][:2],
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(student_model.device)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = student_model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=student_tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        latency = time.perf_counter() - start

        response = student_tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()

        strict_json = True
        try:
            prediction = json.loads(response)
        except (json.JSONDecodeError, TypeError):
            strict_json = False
            cleaned = (
                response.removeprefix("```json")
                .removeprefix("```")
                .removesuffix("```")
                .strip()
            )
            try:
                prediction = json.loads(cleaned)
            except (json.JSONDecodeError, TypeError):
                prediction = None

        valid_schema = (
            isinstance(prediction, dict)
            and set(prediction) == expected_keys
            and all(
                value is None or isinstance(value, str)
                for value in prediction.values()
            )
        )

        result = {
            "index": idx,
            "response": response,
            "prediction": prediction,
            "reference": json.loads(
                example["messages"][2]["content"]
            ),
            "strict_json": strict_json,
            "valid_schema": valid_schema,
            "latency_seconds": latency
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()
        existing[idx] = result

validation_ids = {x["index"] for x in validation_examples}
results = [
    existing[idx]
    for idx in sorted(validation_ids)
    if idx in existing
]

valid_results = [r for r in results if r["valid_schema"]]

category_matches = sum(
    r["prediction"]["category"].lower()
    == r["reference"]["category"].lower()
    for r in valid_results
)

field_matches = sum(
    r["prediction"][key] == r["reference"][key]
    for r in valid_results
    for key in expected_keys
)

print("\n=== FINE-TUNED EVALUATION ===")
print("Examples evaluated:", len(results))
print("Strict JSON:", sum(r["strict_json"] for r in results), "/", len(results))
print("Schema valid:", len(valid_results), "/", len(results))
print("Category agreement:", category_matches, "/", len(valid_results))
print("Exact field agreement:", field_matches, "/", len(valid_results) * 7)
print(
    "Mean latency:",
    round(sum(r["latency_seconds"] for r in results) / len(results), 2),
    "seconds"
)
print("Saved:", output_path)

  0%|          | 0/49 [00:00<?, ?it/s]


=== FINE-TUNED EVALUATION ===
Examples evaluated: 49
Strict JSON: 49 / 49
Schema valid: 49 / 49
Category agreement: 49 / 49
Exact field agreement: 318 / 343
Mean latency: 3.25 seconds
Saved: /content/drive/MyDrive/fashion-distillation/student/finetuned_predictions_v1.jsonl


## Calculate comparable metrics across all 49 examples

In [24]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation/student"

def load_predictions(filename):
    with open(os.path.join(base_dir, filename), encoding="utf-8") as f:
        return {
            r["index"]: r
            for line in f
            if (r := json.loads(line))
        }

baseline = load_predictions("baseline_predictions.jsonl")
finetuned = load_predictions("finetuned_predictions_v1.jsonl")

assert set(baseline) == set(finetuned)
assert len(baseline) == 49

keys = [
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
]

def calculate_metrics(records):
    correct_fields = 0
    correct_categories = 0
    fully_correct = 0

    for record in records.values():
        prediction = record["prediction"]
        reference = record["reference"]

        if not record["valid_schema"]:
            continue

        field_matches = sum(
            prediction[key] == reference[key]
            for key in keys
        )

        correct_fields += field_matches
        correct_categories += (
            prediction["category"].lower()
            == reference["category"].lower()
        )
        fully_correct += field_matches == len(keys)

    return {
        "field_accuracy": correct_fields / (len(records) * 7),
        "category_accuracy": correct_categories / len(records),
        "exact_record_accuracy": fully_correct / len(records)
    }

print("=== ALL-EXAMPLE COMPARISON ===")

base_metrics = calculate_metrics(baseline)
fine_metrics = calculate_metrics(finetuned)

for metric in base_metrics:
    print(f"\n{metric}")
    print(f"Base:       {base_metrics[metric]:.2%}")
    print(f"Fine-tuned: {fine_metrics[metric]:.2%}")
    print(
        "Change:",
        f"{(fine_metrics[metric] - base_metrics[metric]) * 100:+.1f} percentage points"
    )

=== ALL-EXAMPLE COMPARISON ===

field_accuracy
Base:       10.79%
Fine-tuned: 92.71%
Change: +81.9 percentage points

category_accuracy
Base:       22.45%
Fine-tuned: 100.00%
Change: +77.6 percentage points

exact_record_accuracy
Base:       0.00%
Fine-tuned: 57.14%
Change: +57.1 percentage points


## Inspect our reserved test dataset

In [25]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

test_records = read_jsonl(
    os.path.join(base_dir, "splits/test.jsonl")
)

student_train_records = read_jsonl(
    os.path.join(base_dir, "student/train.jsonl")
)

student_val_records = read_jsonl(
    os.path.join(base_dir, "student/validation.jsonl")
)

training_inputs = {
    record["messages"][1]["content"].split(
        "DESCRIPTION: ", 1
    )[-1]
    for record in student_train_records + student_val_records
}

test_inputs = {
    record["input"]
    for record in test_records
}

overlap = training_inputs & test_inputs

print("=== INDEPENDENT TEST SPLIT ===")
print("Test records:", len(test_records))
print("Unique test descriptions:", len(test_inputs))
print("Overlap with student dataset:", len(overlap))

print("\nSAMPLE TEST RECORD:")
print(json.dumps(test_records[0], indent=2, ensure_ascii=False))

assert len(test_records) == 143
assert len(overlap) == 0

print("\n✅ Test split is separate from student training and validation")

=== INDEPENDENT TEST SPLIT ===
Test records: 143
Unique test descriptions: 143
Overlap with student dataset: 0

SAMPLE TEST RECORD:
{
  "input": "Lena Romper Set.. Set with a top and pair of dungarees in soft jersey made from a cotton blend.",
  "category": "Garment Set"
}

✅ Test split is separate from student training and validation


## Create an independent test benchmark

In [26]:
import json
import random
import os
from collections import Counter

base_dir = "/content/drive/MyDrive/fashion-distillation"

# Test records were already loaded in Step 141
assert len(test_records) == 143

# Reproducible sample independent of model predictions
rng = random.Random(2026)
selected_indices = sorted(
    rng.sample(range(len(test_records)), 50)
)

benchmark = [
    {
        "test_index": i,
        "input": test_records[i]["input"],
        "catalog_category": test_records[i]["category"]
    }
    for i in selected_indices
]

benchmark_dir = os.path.join(
    base_dir, "evaluation"
)
os.makedirs(benchmark_dir, exist_ok=True)

benchmark_path = os.path.join(
    benchmark_dir,
    "test_benchmark_50_unlabeled.jsonl"
)

with open(benchmark_path, "w", encoding="utf-8") as f:
    for record in benchmark:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("✅ Independent benchmark selected")
print("Selected products:", len(benchmark))
print("Random seed:", 2026)
print("Saved:", benchmark_path)

print("\nCATEGORY DISTRIBUTION")
for category, count in Counter(
    r["catalog_category"] for r in benchmark
).most_common():
    print(f"{category:<25} {count}")

print("\nFIRST FIVE BENCHMARK PRODUCTS")
for record in benchmark[:5]:
    print("\nINDEX:", record["test_index"])
    print("CATEGORY:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])

✅ Independent benchmark selected
Selected products: 50
Random seed: 2026
Saved: /content/drive/MyDrive/fashion-distillation/evaluation/test_benchmark_50_unlabeled.jsonl

CATEGORY DISTRIBUTION
Hoodie                    5
Garment Set               4
Blouse                    4
Trousers                  4
Shorts                    3
Skirt                     3
Blazer                    3
Sweater                   3
Shirt                     3
Leggings/Tights           3
Vest top                  3
T-shirt                   2
Outdoor Waistcoat         2
Coat                      2
Jacket                    2
Top                       1
Polo shirt                1
Outdoor trousers          1
Tailored Waistcoat        1

FIRST FIVE BENCHMARK PRODUCTS

INDEX: 0
CATEGORY: Garment Set
DESCRIPTION: Lena Romper Set.. Set with a top and pair of dungarees in soft jersey made from a cotton blend.

INDEX: 1
CATEGORY: Top
DESCRIPTION: PE ELKE TOP 3. Sleeveless top woven in a viscose and linen blend wi

## the independent evaluation labels

In [27]:
import json
import os

base_dir = "/content/drive/MyDrive/fashion-distillation"

benchmark_path = os.path.join(
    base_dir,
    "evaluation/test_benchmark_50_unlabeled.jsonl"
)

annotation_path = os.path.join(
    base_dir,
    "evaluation/test_benchmark_50_annotations.jsonl"
)

with open(benchmark_path, encoding="utf-8") as f:
    benchmark = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(benchmark) == 50

def create_annotation(record):
    return {
        "test_index": record["test_index"],
        "input": record["input"],
        "catalog_category": record["catalog_category"],
        "attributes": {
            "category": record["catalog_category"].lower(),
            "color": None,
            "material": None,
            "fit": None,
            "pattern": None,
            "sleeve_length": None,
            "neckline": None
        },
        "reviewed": False
    }

annotations = [
    create_annotation(record)
    for record in benchmark
]

# Protect any previously reviewed annotations
if os.path.exists(annotation_path):
    raise FileExistsError(
        "Annotation file already exists. "
        "Refusing to overwrite prior work."
    )

with open(annotation_path, "w", encoding="utf-8") as f:
    for record in annotations:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Independent annotation template created")
print("Products:", len(annotations))
print("Reviewed:", sum(r["reviewed"] for r in annotations))
print("Saved:", annotation_path)

print("\nFIRST THREE PRODUCTS FOR REVIEW")

for record in annotations[:3]:
    print("\n" + "=" * 55)
    print("TEST INDEX:", record["test_index"])
    print("DESCRIPTION:", record["input"])
    print("CURRENT ATTRIBUTES:", record["attributes"])

✅ Independent annotation template created
Products: 50
Reviewed: 0
Saved: /content/drive/MyDrive/fashion-distillation/evaluation/test_benchmark_50_annotations.jsonl

FIRST THREE PRODUCTS FOR REVIEW

TEST INDEX: 0
DESCRIPTION: Lena Romper Set.. Set with a top and pair of dungarees in soft jersey made from a cotton blend.
CURRENT ATTRIBUTES: {'category': 'garment set', 'color': None, 'material': None, 'fit': None, 'pattern': None, 'sleeve_length': None, 'neckline': None}

TEST INDEX: 1
DESCRIPTION: PE ELKE TOP 3. Sleeveless top woven in a viscose and linen blend with a slight sheen. Elastication around the neckline and armholes, a concealed zip and hook-and-eye fastener at the back, and a straight-cut hem with short slits in the sides.
CURRENT ATTRIBUTES: {'category': 'top', 'color': None, 'material': None, 'fit': None, 'pattern': None, 'sleeve_length': None, 'neckline': None}

TEST INDEX: 3
DESCRIPTION: Daze PU shorts. Shorts in imitation leather with a regular waist, face pockets and a

## Display the first 20 products for independent annotation

In [28]:
import json
import os

annotation_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "evaluation/test_benchmark_50_annotations.jsonl"
)

with open(annotation_path, encoding="utf-8") as f:
    annotations = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(annotations) == 50

print("=== INDEPENDENT ANNOTATION: FIRST 20 ===")

for record in annotations[:20]:
    print("\n" + "=" * 60)
    print("TEST INDEX:", record["test_index"])
    print("CATALOG CATEGORY:", record["catalog_category"])
    print("DESCRIPTION:", record["input"])

print("\nProducts displayed:", len(annotations[:20]))
print("Previously reviewed:", sum(r["reviewed"] for r in annotations))

=== INDEPENDENT ANNOTATION: FIRST 20 ===

TEST INDEX: 0
CATALOG CATEGORY: Garment Set
DESCRIPTION: Lena Romper Set.. Set with a top and pair of dungarees in soft jersey made from a cotton blend.

TEST INDEX: 1
CATALOG CATEGORY: Top
DESCRIPTION: PE ELKE TOP 3. Sleeveless top woven in a viscose and linen blend with a slight sheen. Elastication around the neckline and armholes, a concealed zip and hook-and-eye fastener at the back, and a straight-cut hem with short slits in the sides.

TEST INDEX: 3
CATALOG CATEGORY: Shorts
DESCRIPTION: Daze PU shorts. Shorts in imitation leather with a regular waist, face pockets and a visible zip at the front. Lined.

TEST INDEX: 9
CATALOG CATEGORY: T-shirt
DESCRIPTION: Color L/S Tee. Long-sleeved top in soft cotton jersey with a print motif and slits in the sides. Slightly longer at the back.

TEST INDEX: 11
CATALOG CATEGORY: T-shirt
DESCRIPTION: R-NECK LS BASIC 3 PK. Jersey tops in soft cotton with long sleeves and ribbing around the neckline.

TEST I

## Save the 20 reviewed reference labels

In [29]:
import json
import os
import copy

annotations_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "evaluation/test_benchmark_50_annotations.jsonl"
)

output_path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "evaluation/test_benchmark_20_reviewed_v1.jsonl"
)

with open(annotations_path, encoding="utf-8") as f:
    annotations = [
        json.loads(line)
        for line in f if line.strip()
    ]

# category is copied from the catalog, as required by our task.
# Unspecified attributes remain null.
references = {
    0:  {"material": "cotton blend jersey"},
    1:  {"material": "viscose and linen blend", "sleeve_length": "sleeveless"},
    3:  {"material": "imitation leather"},
    9:  {"material": "cotton jersey", "pattern": "printed", "sleeve_length": "long"},
    11: {"material": "cotton jersey", "sleeve_length": "long", "neckline": "round neck"},
    18: {"material": "cotton weave", "sleeve_length": "short", "neckline": "collar"},
    20: {"material": "textured weave"},
    25: {"material": "cotton", "pattern": "embroidered", "sleeve_length": "long", "neckline": "frilled collar"},
    26: {"material": "Tencel lyocell and silk blend", "pattern": "patterned"},
    28: {"neckline": "stand-up collar"},
    30: {"material": "sweatshirt fabric", "sleeve_length": "long"},
    32: {"material": "airy weave", "pattern": "burnout pattern", "sleeve_length": "long", "neckline": "stand-up collar"},
    36: {"material": "jersey", "sleeve_length": "long", "neckline": "v-neck"},
    37: {"material": "organic cotton, jersey, corduroy"},
    39: {"material": "woven fabric", "fit": "straight-cut", "neckline": "notch lapels"},
    40: {"material": "textured weave"},
    43: {"material": "cotton"},
    44: {"material": "fine knit", "sleeve_length": "long"},
    45: {"material": "sweatshirt fabric", "pattern": "printed", "sleeve_length": "long"},
    48: {"material": "woven fabric", "fit": "straight-cut", "neckline": "shawl collar"}
}

reviewed = []

for record in annotations:
    idx = record["test_index"]

    if idx not in references:
        continue

    item = copy.deepcopy(record)
    item["attributes"].update(references[idx])
    item["reviewed"] = True
    reviewed.append(item)

assert len(reviewed) == 20
assert {r["test_index"] for r in reviewed} == set(references)

# Do not overwrite previous benchmark versions
assert not os.path.exists(output_path), "Reviewed benchmark already exists"

with open(output_path, "w", encoding="utf-8") as f:
    for record in reviewed:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ Independent reference benchmark saved")
print("Reviewed examples:", len(reviewed))
print("Unreviewed examples remaining:", len(annotations) - len(reviewed))
print("Saved:", output_path)

print("\nFIRST REFERENCE:")
print(json.dumps(reviewed[0], indent=2, ensure_ascii=False))

✅ Independent reference benchmark saved
Reviewed examples: 20
Unreviewed examples remaining: 30
Saved: /content/drive/MyDrive/fashion-distillation/evaluation/test_benchmark_20_reviewed_v1.jsonl

FIRST REFERENCE:
{
  "test_index": 0,
  "input": "Lena Romper Set.. Set with a top and pair of dungarees in soft jersey made from a cotton blend.",
  "catalog_category": "Garment Set",
  "attributes": {
    "category": "garment set",
    "color": null,
    "material": "cotton blend jersey",
    "fit": null,
    "pattern": null,
    "sleeve_length": null,
    "neckline": null
  },
  "reviewed": true
}


## Evaluate the fine-tuned model on 20 unseen products

In [30]:
import os
import json
import time
import torch
from tqdm.auto import tqdm

base_dir = "/content/drive/MyDrive/fashion-distillation"

benchmark_path = os.path.join(
    base_dir,
    "evaluation/test_benchmark_20_reviewed_v1.jsonl"
)

output_path = os.path.join(
    base_dir,
    "evaluation/test_finetuned_predictions_v1.jsonl"
)

with open(benchmark_path, encoding="utf-8") as f:
    benchmark = [
        json.loads(line)
        for line in f if line.strip()
    ]

with open(
    os.path.join(base_dir, "config/teacher_config.json"),
    encoding="utf-8"
) as f:
    config = json.load(f)

system_prompt = config["system_prompt"]
expected_keys = set(config["expected_keys"])

assert len(benchmark) == 20
assert hasattr(student_model, "peft_config")
assert student_model.device.type == "cuda"

student_model.eval()
student_model.config.use_cache = True

existing = {}

if os.path.isfile(output_path):
    with open(output_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                result = json.loads(line)
                existing[result["test_index"]] = result

with open(output_path, "a", encoding="utf-8") as f:
    for record in tqdm(benchmark):
        idx = record["test_index"]

        if idx in existing:
            continue

        messages = [
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": (
                    f"CATALOG_CATEGORY: {record['catalog_category']}\n"
                    f"DESCRIPTION: {record['input']}"
                )
            }
        ]

        inputs = student_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(student_model.device)

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = student_model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=student_tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        latency = time.perf_counter() - start

        response = student_tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()

        try:
            prediction = json.loads(response)
            strict_json = True
        except json.JSONDecodeError:
            strict_json = False
            cleaned = (
                response.removeprefix("```json")
                .removeprefix("```")
                .removesuffix("```")
                .strip()
            )
            try:
                prediction = json.loads(cleaned)
            except json.JSONDecodeError:
                prediction = None

        valid_schema = (
            isinstance(prediction, dict)
            and set(prediction) == expected_keys
            and all(
                v is None or isinstance(v, str)
                for v in prediction.values()
            )
        )

        result = {
            "test_index": idx,
            "response": response,
            "prediction": prediction,
            "reference": record["attributes"],
            "strict_json": strict_json,
            "valid_schema": valid_schema,
            "latency_seconds": latency
        }

        f.write(json.dumps(result, ensure_ascii=False) + "\n")
        f.flush()
        existing[idx] = result

results = [
    existing[r["test_index"]]
    for r in benchmark
]

valid = [r for r in results if r["valid_schema"]]

field_matches = sum(
    r["prediction"][key] == r["reference"][key]
    for r in valid
    for key in expected_keys
)

print("\n=== INDEPENDENT TEST: FINE-TUNED MODEL ===")
print("Products evaluated:", len(results))
print("Strict JSON:", sum(r["strict_json"] for r in results), "/20")
print("Schema valid:", len(valid), "/20")
print("Exact field matches:", field_matches, "/140")
print("Field agreement:", f"{field_matches / 140:.2%}")
print(
    "Mean latency:",
    round(sum(r["latency_seconds"] for r in results) / 20, 2),
    "seconds"
)
print("Saved:", output_path)

  0%|          | 0/20 [00:00<?, ?it/s]


=== INDEPENDENT TEST: FINE-TUNED MODEL ===
Products evaluated: 20
Strict JSON: 20 /20
Schema valid: 20 /20
Exact field matches: 119 /140
Field agreement: 85.00%
Mean latency: 3.49 seconds
Saved: /content/drive/MyDrive/fashion-distillation/evaluation/test_finetuned_predictions_v1.jsonl


## Evaluate the original 0.5B model on the same test set

In [31]:
import os
import json
import time
import torch
from tqdm.auto import tqdm

base_dir = "/content/drive/MyDrive/fashion-distillation"

output_path = os.path.join(
    base_dir,
    "evaluation/test_base_predictions_v1.jsonl"
)

# Use the same 20 benchmark records and system prompt
assert len(benchmark) == 20
assert hasattr(student_model, "peft_config")

existing = {}

if os.path.isfile(output_path):
    with open(output_path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                result = json.loads(line)
                existing[result["test_index"]] = result

student_model.eval()

# Temporarily disable LoRA without modifying saved weights
with student_model.disable_adapter():
    with open(output_path, "a", encoding="utf-8") as f:
        for record in tqdm(benchmark):
            idx = record["test_index"]

            if idx in existing:
                continue

            messages = [
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": (
                        f"CATALOG_CATEGORY: {record['catalog_category']}\n"
                        f"DESCRIPTION: {record['input']}"
                    )
                }
            ]

            inputs = student_tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt"
            ).to(student_model.device)

            torch.cuda.synchronize()
            start = time.perf_counter()

            with torch.inference_mode():
                outputs = student_model.generate(
                    **inputs,
                    max_new_tokens=200,
                    do_sample=False,
                    pad_token_id=student_tokenizer.eos_token_id
                )

            torch.cuda.synchronize()
            latency = time.perf_counter() - start

            response = student_tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[-1]:],
                skip_special_tokens=True
            ).strip()

            strict_json = True

            try:
                prediction = json.loads(response)
            except (json.JSONDecodeError, TypeError):
                strict_json = False
                cleaned = (
                    response.removeprefix("```json")
                    .removeprefix("```")
                    .removesuffix("```")
                    .strip()
                )
                try:
                    prediction = json.loads(cleaned)
                except (json.JSONDecodeError, TypeError):
                    prediction = None

            valid_schema = (
                isinstance(prediction, dict)
                and set(prediction) == expected_keys
                and all(
                    v is None or isinstance(v, str)
                    for v in prediction.values()
                )
            )

            result = {
                "test_index": idx,
                "response": response,
                "prediction": prediction,
                "reference": record["attributes"],
                "strict_json": strict_json,
                "valid_schema": valid_schema,
                "latency_seconds": latency
            }

            f.write(json.dumps(result, ensure_ascii=False) + "\n")
            f.flush()
            existing[idx] = result

results = [existing[r["test_index"]] for r in benchmark]
valid = [r for r in results if r["valid_schema"]]

field_matches = sum(
    r["prediction"][key] == r["reference"][key]
    for r in valid
    for key in expected_keys
)

print("\n=== INDEPENDENT TEST: BASE MODEL ===")
print("Products evaluated:", len(results))
print("Strict JSON:", sum(r["strict_json"] for r in results), "/20")
print("Schema valid:", len(valid), "/20")
print("Exact field matches:", field_matches, "/140")
print("Field agreement:", f"{field_matches / 140:.2%}")
print(
    "Mean latency:",
    round(sum(r["latency_seconds"] for r in results) / 20, 2),
    "seconds"
)
print("Saved:", output_path)
print("✅ LoRA adapter automatically re-enabled")

  0%|          | 0/20 [00:00<?, ?it/s]


=== INDEPENDENT TEST: BASE MODEL ===
Products evaluated: 20
Strict JSON: 0 /20
Schema valid: 7 /20
Exact field matches: 11 /140
Field agreement: 7.86%
Mean latency: 2.58 seconds
Saved: /content/drive/MyDrive/fashion-distillation/evaluation/test_base_predictions_v1.jsonl
✅ LoRA adapter automatically re-enabled


## Identify where the fine-tuned model still fails

In [32]:
import os
import json
from collections import Counter

path = (
    "/content/drive/MyDrive/fashion-distillation/"
    "evaluation/test_finetuned_predictions_v1.jsonl"
)

with open(path, encoding="utf-8") as f:
    results = [
        json.loads(line)
        for line in f if line.strip()
    ]

keys = [
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
]

mismatches = []

for record in results:
    prediction = record["prediction"]
    reference = record["reference"]

    for key in keys:
        if prediction[key] != reference[key]:
            mismatches.append({
                "index": record["test_index"],
                "attribute": key,
                "expected": reference[key],
                "predicted": prediction[key]
            })

print("=== FINE-TUNED MODEL ERROR ANALYSIS ===")
print("Total mismatched fields:", len(mismatches))

print("\nMISMATCHES BY ATTRIBUTE")
for attribute, count in Counter(
    item["attribute"] for item in mismatches
).most_common():
    print(f"{attribute:<18} {count}")

print("\nALL MISMATCHES")
for item in mismatches:
    print(
        f"Index {item['index']:>3} | "
        f"{item['attribute']:<15} | "
        f"Expected: {str(item['expected']):<30} | "
        f"Predicted: {item['predicted']}"
    )

=== FINE-TUNED MODEL ERROR ANALYSIS ===
Total mismatched fields: 21

MISMATCHES BY ATTRIBUTE
material           11
pattern            5
neckline           3
fit                2

ALL MISMATCHES
Index   0 | material        | Expected: cotton blend jersey            | Predicted: jersey
Index   1 | fit             | Expected: None                           | Predicted: straight-cut
Index   9 | pattern         | Expected: printed                        | Predicted: print
Index  11 | material        | Expected: cotton jersey                  | Predicted: cotton
Index  11 | neckline        | Expected: round neck                     | Predicted: ribbing
Index  18 | material        | Expected: cotton weave                   | Predicted: cotton
Index  18 | neckline        | Expected: collar                         | Predicted: None
Index  25 | pattern         | Expected: embroidered                    | Predicted: None
Index  25 | neckline        | Expected: frilled collar                 | Pre

## Save the final evaluation report

In [33]:
import os
import json
from collections import Counter

base_dir = "/content/drive/MyDrive/fashion-distillation"
eval_dir = os.path.join(base_dir, "evaluation")

def load_jsonl(filename):
    with open(os.path.join(eval_dir, filename), encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

base_results = load_jsonl("test_base_predictions_v1.jsonl")
fine_results = load_jsonl("test_finetuned_predictions_v1.jsonl")

keys = [
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
]

def summarize(results):
    total = len(results)
    strict = sum(r["strict_json"] for r in results)
    schema = sum(r["valid_schema"] for r in results)

    correct_fields = sum(
        r["prediction"][key] == r["reference"][key]
        for r in results if r["valid_schema"]
        for key in keys
    )

    return {
        "examples": total,
        "strict_json_rate": strict / total,
        "schema_validity_rate": schema / total,
        "exact_field_agreement": correct_fields / (total * 7),
        "mean_latency_seconds": sum(
            r["latency_seconds"] for r in results
        ) / total
    }

report = {
    "experiment": "Qwen2.5-0.5B Fashion Attribute Extraction",
    "teacher": "Qwen/Qwen2.5-7B-Instruct",
    "student": "Qwen/Qwen2.5-0.5B-Instruct",
    "training_method": "Supervised LoRA fine-tuning on teacher-generated labels",
    "training_examples": 436,
    "internal_validation_examples": 49,
    "independent_test_examples": 20,
    "epochs": 3,
    "lora_rank": 16,
    "base": summarize(base_results),
    "finetuned": summarize(fine_results),
    "limitations": [
        "Small exploratory test benchmark of 20 products",
        "Reference labels have not been independently adjudicated",
        "Category is supplied in the input",
        "Exact matching penalizes synonymous attribute values",
        "Material extraction sometimes omits composition",
        "Training data contains few explicit color labels",
        "Latency depends on hardware and generation conditions",
        "This is supervised fine-tuning on teacher outputs, not logit-based knowledge distillation"
    ]
}

report_path = os.path.join(
    eval_dir, "fashion_distillation_v1_report.json"
)

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("✅ Evaluation report saved")
print("Path:", report_path)

print("\nBASE:")
print(json.dumps(report["base"], indent=2))

print("\nFINE-TUNED:")
print(json.dumps(report["finetuned"], indent=2))

✅ Evaluation report saved
Path: /content/drive/MyDrive/fashion-distillation/evaluation/fashion_distillation_v1_report.json

BASE:
{
  "examples": 20,
  "strict_json_rate": 0.0,
  "schema_validity_rate": 0.35,
  "exact_field_agreement": 0.07857142857142857,
  "mean_latency_seconds": 2.5797151398499865
}

FINE-TUNED:
{
  "examples": 20,
  "strict_json_rate": 1.0,
  "schema_validity_rate": 1.0,
  "exact_field_agreement": 0.85,
  "mean_latency_seconds": 3.4867085684999894
}


## Hugging Face Setup

In [36]:
from pathlib import Path

adapter_dir = Path(
    "/content/drive/MyDrive/fashion-distillation/"
    "student/fashion_lora_adapter_v1"
)

assert (adapter_dir / "adapter_config.json").exists(), (
    "Saved LoRA adapter not found"
)

readme_path = adapter_dir / "README.md"

model_card = '''---
base_model: Qwen/Qwen2.5-0.5B-Instruct
library_name: peft
pipeline_tag: text-generation
tags:
  - fashion
  - attribute-extraction
  - structured-output
  - qwen2.5
  - lora
  - response-distillation
---

# Qwen2.5-0.5B Fashion Attribute Extractor

A task-specific LoRA adapter for extracting structured fashion
attributes from product descriptions.

## Model Overview

- Base model: Qwen2.5-0.5B-Instruct
- Teacher: Qwen2.5-7B-Instruct
- Method: Supervised fine-tuning using teacher-generated labels
- Trainable LoRA parameters: Approximately 8.8 million
- Training examples: 436
- Internal validation examples: 49
- Training epochs: 3

This is response-based distillation, not logit-based distillation.

## Extracted Attributes

The model produces a JSON object with seven attributes:

1. category
2. color
3. material
4. fit
5. pattern
6. sleeve_length
7. neckline

Each attribute is a string or null.

The catalog category is provided in the input.

## Exploratory Evaluation

Evaluation used 20 unseen H&M product descriptions with
provisional, description-grounded reference annotations.

| Metric | Base model | Fine-tuned model |
|---|---:|---:|
| Strict JSON validity | 0% | 100% |
| Schema validity | 35% | 100% |
| Exact field agreement | 7.86% | 85.00% |
| Mean latency | 2.58 s | 3.49 s |

Invalid-schema predictions received zero field-match credit.

Hardware: Google Colab Tesla T4.

## Methodology

The teacher generated structured fashion attributes from
H&M product descriptions.

After filtering and review, 436 examples were used to
fine-tune the 0.5B student using LoRA.

The experiment used supervised assistant-response loss.
It did not use teacher logits or KL-divergence optimization.

## Usage

Load the Qwen2.5-0.5B-Instruct base model using Transformers,
then load this adapter using PEFT.

Use the same system prompt and input format as training:

CATALOG_CATEGORY: <category>
DESCRIPTION: <product description>

The complete training system prompt should be supplied separately
for reproducible inference.

## Limitations

- Only 20 independent test examples were evaluated.
- Reference annotations have not been independently adjudicated.
- Category is supplied in the input.
- Exact matching penalizes synonymous attribute values.
- Material composition is sometimes incomplete.
- Some patterns and necklines are omitted.
- Cross-dataset and multilingual generalization are untested.
- Measured latency depends on hardware and inference settings.

## Intended Use

Research and experimentation in structured fashion catalog
attribute extraction.

Human verification is recommended for production applications.
'''

readme_path.write_text(model_card, encoding="utf-8")

print("Hugging Face model card created")
print("Saved:", readme_path)

print("\nAdapter directory contents:")
for file in sorted(adapter_dir.iterdir()):
    print("-", file.name)

Hugging Face model card created
Saved: /content/drive/MyDrive/fashion-distillation/student/fashion_lora_adapter_v1/README.md

Adapter directory contents:
- README.md
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- tokenizer.json
- tokenizer_config.json


In [39]:
from huggingface_hub import HfApi, login
from huggingface_hub.utils import HfHubHTTPError

api = HfApi()

try:
    user = api.whoami()
except Exception:
    print("Hugging Face authentication required.")
    login()
    user = api.whoami()

print("Authenticated as:", user["name"])

repo_id = "minhack/qwen2.5-0.5b-fashion-attribute-extractor"

try:
    repo = api.repo_info(repo_id=repo_id, repo_type="model")
    print("Repository exists:", repo.id)
    print("Visibility:", "Private" if repo.private else "Public")

    files = api.list_repo_files(repo_id=repo_id)
    print("Adapter config uploaded:", "adapter_config.json" in files)
    print("Files:", files)

except HfHubHTTPError as error:
    print("Repository is unavailable with current credentials.")
    print("HTTP status:", error.response.status_code)
    print("Check username, repository existence and access permissions.")

Hugging Face authentication required.


Authenticated as: minhack
Repository is unavailable with current credentials.
HTTP status: 404
Check username, repository existence and access permissions.


In [40]:
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()

repo_id = "minhack/qwen2.5-0.5b-fashion-attribute-extractor"

adapter_dir = Path(
    "/content/drive/MyDrive/fashion-distillation/"
    "student/fashion_lora_adapter_v1"
)

assert (adapter_dir / "adapter_config.json").exists()
assert (adapter_dir / "README.md").exists()
assert any(
    (adapter_dir / name).exists()
    for name in ["adapter_model.safetensors", "adapter_model.bin"]
)

# Create the public repository
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,
    exist_ok=True
)

# Upload the saved adapter and model card
api.upload_folder(
    folder_path=str(adapter_dir),
    repo_id=repo_id,
    repo_type="model",
    commit_message="Publish fashion attribute extraction LoRA adapter v1"
)

# Verify uploaded files
files = api.list_repo_files(repo_id)

print("Repository:", f"https://huggingface.co/{repo_id}")
print("\nUploaded files:")

for filename in files:
    print("-", filename)

assert "adapter_config.json" in files
assert "README.md" in files
assert any(
    name in files
    for name in ["adapter_model.safetensors", "adapter_model.bin"]
)

print("\n✅ Model adapter published successfully")

Repository: https://huggingface.co/minhack/qwen2.5-0.5b-fashion-attribute-extractor

Uploaded files:
- .gitattributes
- README.md
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- tokenizer.json
- tokenizer_config.json

✅ Model adapter published successfully


## perform testing from hugging face model

In [41]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_id = "Qwen/Qwen2.5-0.5B-Instruct"
repo_id = "minhack/qwen2.5-0.5b-fashion-attribute-extractor"

tokenizer = AutoTokenizer.from_pretrained(base_id)

base = AutoModelForCausalLM.from_pretrained(
    base_id,
    torch_dtype=torch.float16
).to("cuda")

model = PeftModel.from_pretrained(base, repo_id)
model.eval()

with open(
    "/content/drive/MyDrive/fashion-distillation/config/teacher_config.json",
    encoding="utf-8"
) as f:
    system_prompt = json.load(f)["system_prompt"]

messages = [
    {"role": "system", "content": system_prompt},
    {
        "role": "user",
        "content": (
            "CATALOG_CATEGORY: Hoodie\n"
            "DESCRIPTION: Relaxed-fit cotton hoodie with long sleeves, "
            "drawstring hood and a printed graphic on the front."
        )
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to("cuda")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

print("=== HUGGING FACE INFERENCE TEST ===")
print(response)

prediction = json.loads(response)

expected = {
    "category", "color", "material", "fit",
    "pattern", "sleeve_length", "neckline"
}

assert set(prediction) == expected
assert all(
    v is None or isinstance(v, str)
    for v in prediction.values()
)

print("\n✅ Published adapter loaded and generated valid JSON")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 35.2MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

=== HUGGING FACE INFERENCE TEST ===
{"category": "hoodie", "color": null, "material": "cotton", "fit": "relaxed-fit", "pattern": "printed", "sleeve_length": "long", "neckline": null}

✅ Published adapter loaded and generated valid JSON


In [42]:

from pathlib import Path
from huggingface_hub import HfApi

repo_id = "minhack/qwen2.5-0.5b-fashion-attribute-extractor"
readme_path = Path(
    "/content/drive/MyDrive/fashion-distillation/"
    "student/fashion_lora_adapter_v1/README.md"
)

readme = readme_path.read_text(encoding="utf-8")

start = readme.index("## Exploratory Evaluation")
end = readme.index("## Methodology", start)

results_section = """## Exploratory Evaluation

Both models were evaluated on the same 20 previously unseen
H&M product descriptions. Each product has seven target
attributes, giving 140 attribute comparisons.

| Metric | Base 0.5B | Fine-tuned 0.5B |
|---|---:|---:|
| Attribute matches | 11/140 (7.86%) | 119/140 (85.00%) |
| Strict JSON validity | 0/20 (0%) | 20/20 (100%) |
| Schema validity | 7/20 (35%) | 20/20 (100%) |
| Mean inference latency | 2.58 s | 3.49 s |

### Metric definitions

- **Attribute matches:** Exact string matches against the
  reference values across seven attributes per product.
  Invalid-schema responses receive zero attribute matches.
- **Strict JSON validity:** Responses parse directly as JSON,
  without markdown fences or additional text.
- **Schema validity:** Responses contain exactly the seven
  required keys, with string or null values.
- **Mean inference latency:** Average generation time per
  product on a Google Colab Tesla T4.

### Error analysis

The fine-tuned model had 21 mismatched attribute values:

| Attribute | Mismatches |
|---|---:|
| Material | 11 |
| Pattern | 5 |
| Neckline | 3 |
| Fit | 2 |

Examples include predicting `jersey` instead of
`cotton blend jersey`, missing an embroidered pattern,
and returning `print` instead of `printed`.

Some differences reflect incomplete extraction, while others
reflect wording variations or reference-label ambiguity.

### Evaluation limitations

The 20 products were held out from training. Reference labels
were constructed from product descriptions but have not been
independently adjudicated. Category was supplied in the input.

These are exploratory exact-agreement results, not definitive
production accuracy estimates. Latency was measured under the
tested generation settings and hardware configuration.

"""

updated_readme = (
    readme[:start]
    + results_section
    + readme[end:]
)

readme_path.write_text(updated_readme, encoding="utf-8")

api = HfApi()
api.upload_file(
    path_or_fileobj=str(readme_path),
    path_in_repo="README.md",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Clarify evaluation metrics and add error analysis"
)

print("README updated successfully.")
print(f"https://huggingface.co/{repo_id}")


README updated successfully.
https://huggingface.co/minhack/qwen2.5-0.5b-fashion-attribute-extractor
